`cell 0`

# Self-Auditing GraphRAG — Reproduction Notebook **v2**

*Reformulating Reliability in Knowledge-Sparse Agricultural Domains:
A Self-Auditing GraphRAG Case Study on Hydroponic Wasabi*

One notebook reproducing every experiment in the paper, in the order the paper presents
them. Each part names the section it supports.

| Part | Paper section | Contents |
|---|---|---|
| 0 | §4.1.3 | Environment, paths, input hashes, API keys |
| 1 | §4.1.1 | Corpus: 19 papers → 770 triplets; 60-item Gold Standard |
| 2 | §3.2.2 | Domain-configurable reference tables |
| 3 | §3.2.1, §3.2.3 | Conflict taxonomy and rule-based detector |
| 4 | §3.2.4, §4.2.1 | Context auditor and Confidence Level assignment |
| 5 | §4.2.2, §4.2.3 | Representative cases and failure analysis |
| 6 | §4.2.4 | Injection test |
| 7 | §4.2.5, §4.2.6 | Gold-Standard false-alarm rate and recall |
| 8 | §3.3, §3.4 | Knowledge graph and causal traversal |
| 9 | §4.3.1 | Multi-LLM judge evaluation — B1–B5, five conditions |
| 10 | §4.3.2 | Case study C1 — longest causal chain (4 hops, measured) |
| 11 | §4.4 | Generalization: sparse-subset stability on strawberry |
| 12 | — | Figures and tables |
| 13 | — | Conformance audit and manuscript revision checklist |
| 14 | — | Manifest |

## What is new in v2

v2 does not replace v1. It shares v1's API caches in `repro_out/` — which is why it is
nearly free to run — and writes every output under a `_v2` suffix, so the v1 deposit is
untouched and still reproducible from the v1 notebook.

| Part | Change | New API calls |
|---|---|---|
| 8 | **B4**, the prompting control: B3's retrieval, no Confidence Levels, an instruction instead. Conditions renumbered so the proposed system is **B5** and the control is **B4** | — |
| 9e | B4 generated, judged, and compared to B3/B5; **McNemar** on the judge-free disclosure count | 122 (cached after the first run) |
| 9f | **The standard RAG rubric** (RAGAS/ARES triad) with anchored scales, judged with the evidence in view | 300 |
| 0 | Outputs routed to `MANIFEST_v2.json`, `figures_v2/`, `tables_v2/` | — |

Everything else is identical to v1 and hits the existing caches.

## Provenance

The detector, the reference tables and the injection design are the author's, transcribed
from the original experiment notebooks. Three behaviours specified in the paper but absent
from the original inline detector are added in Part 3 and are individually flagged in the
code and quantified in Part 3's sensitivity table, so a reader can see exactly what each
one contributes.

### Which evaluation is reported

`REPORT` in Part 0 selects it, and it is set to **`"regenerated"`**. The regenerated run
is the primary result, for a reason that is a finding in its own right: the recorded
answers (`judge_inputs.json`) were produced *before* the §3.4 retrieval fix, when no query
reached a parameter node. No answer could carry a Confidence Level, so B5 was in substance
identical to B3 and the recorded Table 11 cannot support any claim about the auditing
layer. Part 9b regenerates all 100 answers (5 conditions x 20 queries) on the corrected
retriever and re-judges them
with the same rubric.

The recorded study is **not deleted and not overwritten**. `judge_inputs.json` and
`RQ2_judge_scores_canonical.json` stay in the deposit as inputs, Part 9b prints the two
runs side by side over the judges they share, and `MANIFEST.reporting_source` records
which one every figure and table came from. Setting `REPORT = "recorded"` in Part 0
reproduces the original numbers throughout.

## Determinism

| Stage | Nature | On re-run |
|---|---|---|
| Step 1 extraction | LLM call | Non-deterministic — the frozen corpus is loaded by default |
| Step 2 detection | Pure rules | Fully deterministic |
| Step 3 context audit | LLM call | Mildly non-deterministic even at `temperature=0`; cached |
| Knowledge graph, Gold FPR, recall, injection scoring | Pure rules | Deterministic given the audit cache |
| Answer generation, Part 9b | LLM call | Non-deterministic; cached and version-stamped |
| Multi-LLM judging | LLM call at `temperature=0` | Cached; deterministic given the cache |
| Statistics | Pure computation | Deterministic (bootstrap seeded) |

## Cost

Everything is cached under `repro_out/`. With the caches present a full top-to-bottom run
makes **no API calls**. From empty, with the full three-judge panel available:

| Stage | Calls | How it scales |
|---|---|---|
| Part 4 context audit | ~170 | one per rule-stage flag |
| Part 6 injection | ~170 | one per injected triplet, per protocol |
| Part 9b generation | 100 | 20 queries × 5 conditions, judge-count independent |
| Part 9b judging | up to 300 | **100 per judge**; 200 with two judges, 300 with three |
| Part 9d UR judging | up to 210 | **70 per judge** (14 disputed queries × 5 conditions) |
| Part 9f standard rubric | up to 300 | **100 per judge**, the RAG-triad rescoring |
| Part 10 illustration | 1 | one B5 answer, not scored |
| Part 11 | 0 | reads the strawberry extraction cache |

The three judging rows are upper bounds, and they total **810 of the ~1,290 calls**. J3 is
served locally from open weights, so if that server is not running the panel silently drops
to two judges and those rows fall to 200 / 140 / 200 — which is not only a cost difference:
the reliability coefficients are panel-size dependent, and Part 13 refuses to write §3.5's
model sentence when fewer than three judges scored the run. Check the judge count Part 9b
prints before reporting.

Every cache carries a stamp of the configuration that produced it — `DETECTOR_SIGNATURE`
for the audits, `PROMPT_VERSION` for the answers and the C1 illustration, the judge model
id and `answers_version` for the scores. A cached entry written under a different
configuration is discarded rather than reused, so no number in this notebook can come from
a run it does not describe.

## API keys

Keys are read from the environment or prompted for interactively. **No key is written
into this notebook.** `OPENAI_API_KEY` is needed only if a cache is missing.

`cell 1`

## Part 0 — Environment, paths, input hashes  *(§4.1.3)*

In [3]:
# ====== cell 2 ======
import os, sys, json, re, time, random, hashlib, platform, importlib.metadata, getpass
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone

# ------------------------------- configuration -------------------------------
RERUN_STEP1   = False           # True: re-extract from PDFs (non-deterministic)
MODEL_EXTRACT = "gpt-4o"        # pin a dated snapshot for archival runs
MODEL_AUDIT   = "gpt-4o"        # the notebook's own calls (oracle protocol, §4.3.2
                                # illustration, closed-book control). The Step 3 auditor
                                # hardcodes its own model inside GPT4oContextAuditor.
CHUNK_SIZE, OVERLAP = 8000, 400
FUZZY_TAU  = 0.50               # similarity threshold for Gold-Standard recall
N_STABILITY = 5                 # repeat count for the Step 3 stability check
SEED = 42
random.seed(SEED)

# Which evaluation Part 9 reports. The recorded study was produced before the §3.4
# retrieval defect was found and before the common answer budget was imposed, so its
# comparison between B3 and B5 is not interpretable; the regenerated run is what the
# manuscript reports. This is declared HERE, ahead of every cell that reads it, so that
# one top-to-bottom execution produces one internally consistent set of numbers. The
# earlier design switched the source midway through Part 9b, which left Part 9 reporting
# one run and Part 12 the other unless the reader knew to go back and re-execute.
# "recorded" reproduces the earlier evaluation instead; nothing else changes.
REPORT = "regenerated"

MANIFEST = {"started_utc": datetime.now(timezone.utc).isoformat(),
            "config": dict(RERUN_STEP1=RERUN_STEP1, MODEL_EXTRACT=MODEL_EXTRACT,
                           MODEL_AUDIT=MODEL_AUDIT, CHUNK_SIZE=CHUNK_SIZE, OVERLAP=OVERLAP,
                           FUZZY_TAU=FUZZY_TAU, N_STABILITY=N_STABILITY, SEED=SEED,
                           REPORT=REPORT)}

# ------------------------------- directories ---------------------------------
# Override with environment variables; the last entry of each list is the author's layout.
def pick(env, *cands, needs=()):
    for c in [os.environ.get(env, "")] + list(cands):
        if not c:
            continue
        p = Path(c)
        if p.exists() and all((p / n).exists() for n in needs):
            return p
    return None

DATA_DIR = pick("WASABI_DATA_DIR", ".", "..",
                r"C:\Users\USER\OneDrive\Desktop\와사비연구\와사비 관련 문헌 및 기술자료",
                needs=("gpt4o_extracted_merged.json",))
WORK_DIR = pick("WASABI_WORK_DIR", ".", r"C:\Users\USER\0_와사비실험",
                needs=("judge_inputs.json",))
BERRY_DIR = pick("STRAWBERRY_DIR", ".", r"C:\Users\USER\1_딸기실험",
                 needs=("extractions.json",))

assert DATA_DIR, ("Wasabi data folder not found. Set WASABI_DATA_DIR to the folder "
                  "containing gpt4o_extracted_merged.json.")
OUT_DIR = Path("repro_out"); OUT_DIR.mkdir(exist_ok=True)

# v2 writes NOTHING that v1 wrote. The API caches in OUT_DIR are shared and reused -
# that is the point, it makes this run nearly free - but every OUTPUT of this notebook
# is suffixed, so repro_out/MANIFEST.json, repro_out/figures and repro_out/tables from
# the v1 deposit survive untouched and stay reproducible from the v1 notebook.
OUT_TAG = "_v2"
print(f"notebook     : v2 (outputs suffixed {OUT_TAG!r}; v1 artefacts untouched)")

# ----------------------------- fresh run (optional) --------------------------
# Every API result in this notebook is cached, which makes re-execution free but lets a
# run be assembled from caches written in different sessions by different code. The
# stability check in Part 4 is what exposes that: repeats made by the current auditor did
# not bracket the cached distribution. FRESH_RUN archives every API cache so that one
# top-to-bottom execution produces every number from a single auditor pass.
#
# Caches are MOVED into repro_out/archive_<utc>/, never deleted. The recorded study
# (judge_inputs.json, RQ2_judge_scores_canonical.json) is an INPUT and is never touched.
# The PDF chunk index is deterministic and is kept, since rebuilding it costs minutes and
# changes nothing.
FRESH_RUN = False

API_CACHES = ["step3_audits.json", "step3_audits_multi.json", "stability_runs.json",
              "injection_audits_corpus.json", "injection_audits_oracle.json",
              "injection_audits.json", "injection_step3.json", "injection_step3_multi.json",
              "gold_audits.json", "closed_book.json", "case_c1_illustration.json",
              "answers_regenerated.json", "judge_scores_regenerated.json",
              "rejudge_scores.json", "audited_770_def1.json",
              # added with Parts 9d-9f; without them FRESH_RUN leaves paid scores behind
              # and the "clean slate" it promises is not one.
              "ur_scores.json", "judge_scores_std.json",
              "answers_b5.json", "scores_b5.json"]   # last two: pre-renumbering leftovers

if FRESH_RUN and globals().get("_FRESH_RUN_DONE"):
    print(f"FRESH_RUN already applied in this session ({_FRESH_RUN_DONE}).")
    print("Skipped, so that re-executing this cell cannot archive the results of the run")
    print("you just made. Set FRESH_RUN = False now; restart the kernel to force another.")
elif FRESH_RUN:
    import shutil
    arch = OUT_DIR / f"archive_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}"
    present = [n for n in API_CACHES if (OUT_DIR / n).exists()]
    moved = []
    if present:
        arch.mkdir(parents=True, exist_ok=True)
        for name in present:
            shutil.move(str(OUT_DIR / name), str(arch / name)); moved.append(name)
        print(f"FRESH_RUN: {len(moved)} cache file(s) archived -> {arch.name}")
    else:
        print("FRESH_RUN: no API caches left to archive - already a clean slate")
    for name in moved:
        print(f"  {name}")
    print("\nEvery API-backed cell will now call its model. Approximate call budget:")
    print("  Part 4  context audit          ~167   (one per detector flag)")
    print("  Part 4  stability              ~167 x N_STABILITY")
    print("  Part 6  injection, 2 protocols ~530")
    print("  Part 7  gold-standard flags       3")
    # Literal, not len(BASES): BASES is defined in Part 9, which has not run yet. The
    # arithmetic is 5 conditions (B1-B5) x 20 queries, and 3 judges where a judge is
    # involved. If a condition or a query is added, update this block with it.
    print("  Part 9b answer regeneration     100   (5 conditions x 20 queries)")
    print("  Part 9b judging                 300   (100 per judge, three judges)")
    print("  Part 9d uncertainty reporting   210   (70 per judge; 14 disputed x 5)")
    print("  Part 9f standard RAG rubric     300   (100 per judge, evidence shown)")
    print("  Part 10 C1 illustration           1")
    print("  Part 11 closed-book control       8")
    print("Run the notebook strictly top to bottom: the Confidence Levels from Part 4")
    print("feed the graph, the graph feeds retrieval, and retrieval feeds the answers.")
    _FRESH_RUN_DONE = arch.name if moved else "clean slate"
    MANIFEST["fresh_run"] = {"archive": arch.name, "archived": moved}
    print("\nWhen the run finishes, set FRESH_RUN back to False. Left True, the next")
    print("execution of this cell would archive the results you just produced.")
else:
    print("FRESH_RUN = False - cached API results are reused where present")


FILES = {
    "corpus":    DATA_DIR / "gpt4o_extracted_merged.json",
    "gold":      DATA_DIR / "triplets_gold_standard.json",
    "causal":    DATA_DIR / "causal_relations.json",
    "pipeline":  DATA_DIR / "pipeline.py",                      # reference implementation
    "answers":   (WORK_DIR / "judge_inputs.json") if WORK_DIR else None,
    "judge":     next((p for p in [(BERRY_DIR / "RQ2_judge_scores_canonical.json") if BERRY_DIR else None,
                                   (WORK_DIR / "RQ2_judge_scores_canonical.json") if WORK_DIR else None,
                                   DATA_DIR / "RQ2_judge_scores_canonical.json"]
                      if p and p.exists()), None),
    "berry":     (BERRY_DIR / "extractions.json") if BERRY_DIR else None,
}

def sha256(p, n=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(n), b""):
            h.update(b)
    return h.hexdigest()

print(f"wasabi data  : {DATA_DIR.resolve() if DATA_DIR  else '-'}")
print(f"work folder  : {WORK_DIR.resolve()  if WORK_DIR  else '- (Part 9 answers unavailable)'}")
print(f"strawberry   : {BERRY_DIR.resolve() if BERRY_DIR else '- (Part 11 unavailable)'}")
print(f"output       : {OUT_DIR.resolve()}\n")

print(f"{'input':<12}{'exists':>8}{'bytes':>12}  sha256[:16]")
MANIFEST["inputs"] = {}
for k, p in FILES.items():
    if p and Path(p).exists():
        h = sha256(p)
        MANIFEST["inputs"][k] = {"name": Path(p).name, "bytes": Path(p).stat().st_size, "sha256": h}
        print(f"{k:<12}{'yes':>8}{Path(p).stat().st_size:>12,}  {h[:16]}")
    else:
        MANIFEST["inputs"][k] = None
        print(f"{k:<12}{'NO':>8}{'-':>12}")

# ------------------------------- environment ---------------------------------
def ver(m):
    try:    return importlib.metadata.version(m)
    except Exception: return None

MANIFEST["environment"] = {
    "python": sys.version.split()[0], "platform": platform.platform(),
    "packages": {m: ver(m) for m in ("openai", "anthropic", "groq", "scikit-learn",
                                     "scipy", "numpy", "matplotlib", "networkx",
                                     "pdfplumber", "pypdf")}}
print("\npython", MANIFEST["environment"]["python"], "|", MANIFEST["environment"]["platform"])
print("packages:", {k: v for k, v in MANIFEST["environment"]["packages"].items() if v})
_missing = [k for k, v in MANIFEST["environment"]["packages"].items() if not v]
if _missing:
    print(f"not installed: {_missing}")
    for _m, _what in (("openai", "Parts 4, 6, 7, 9b, 10, 11 make no calls"),
                      ("anthropic", "judge J2 is skipped in Part 9b"),
                      ("groq", "judge J3 is skipped in Part 9b"),
                      ("networkx", "figures F6 and F7 are not drawn"),
                      ("pdfplumber", "B1 chunk index falls back to pypdf")):
        if _m in _missing:
            print(f"  {_m:<12} -> {_what}")

# --------------------- reference implementation (optional) -------------------
# pipeline.py is the deployed implementation. This notebook re-states the same rules
# self-containedly so it runs without it; Part 3 cross-checks the two when it is present.
# pipeline.py holds the detector, the reference tables and the graph classes. It is
# imported unmodified and deposited alongside this notebook; the notebook supplies only
# orchestration, the three additions documented in Part 3, and the reporting.
sys.path.insert(0, str(DATA_DIR))
for m in ("pipeline", "run_full_experiment"):
    sys.modules.pop(m, None)
import pipeline
from pipeline import (THRESHOLDS, KNOWN_UNIT_ALIASES, EQUIVALENT_UNITS, UNIT_CONVERSIONS,
                      UNIT_NOTATION_MAP, normalize_unit_notation,
                      ConflictDetector, KnowledgeGraph)
print(f"\npipeline.py loaded from {Path(pipeline.__file__).resolve()}")
try:
    from run_full_experiment import GPT4oContextAuditor
    HAS_REF_AUDITOR = True
except Exception:
    HAS_REF_AUDITOR = False

# ------------------------------- API keys ------------------------------------
def ensure_key(var, desc, required=False):
    if not os.environ.get(var):
        tag = "required" if required else "optional - press Enter to skip"
        try:
            v = getpass.getpass(f"  {var} ({desc}) [{tag}]: ").strip()
            if v:
                os.environ[var] = v
        except Exception:
            pass
    ok = bool(os.environ.get(var))
    print(f"  {var:<20} {'SET' if ok else 'not set':<8} {desc}")
    return ok

print()
HAS_OPENAI    = ensure_key("OPENAI_API_KEY",    "context audit, optional re-extraction")
HAS_ANTHROPIC = ensure_key("ANTHROPIC_API_KEY", "optional re-judging only")
HAS_GROQ      = ensure_key("GROQ_API_KEY",      "optional re-judging only")
if not HAS_OPENAI:
    print("\n[note] Without OPENAI_API_KEY the notebook runs from cache only.")

notebook     : v2 (outputs suffixed '_v2'; v1 artefacts untouched)
FRESH_RUN = False - cached API results are reused where present
wasabi data  : C:\Users\USER\OneDrive\Desktop\와사비연구\와사비 관련 문헌 및 기술자료
work folder  : C:\Users\USER\0_와사비실험
strawberry   : C:\Users\USER\1_딸기실험
output       : C:\Users\USER\0_와사비실험\repro_out

input         exists       bytes  sha256[:16]
corpus           yes     323,727  f1203bbb0d294dd8
gold             yes      12,709  d149b944ce3df8eb
causal           yes     178,553  46e02b74b6ee097d
pipeline         yes      48,927  e9895228930df51c
answers          yes      80,404  f5c821d210a0ded4
judge            yes      25,718  24cc3efb1d1e5758
berry            yes     404,668  05126b5f080d10fd

python 3.12.7 | Windows-11-10.0.26200-SP0
packages: {'openai': '2.26.0', 'anthropic': '0.116.0', 'groq': '1.1.2', 'scikit-learn': '1.5.1', 'scipy': '1.13.1', 'numpy': '1.26.4', 'matplotlib': '3.9.2', 'networkx': '3.3', 'pdfplumber': '0.11.9', 'pypdf': '6.8.0'}

pipeline.py l

  OPENAI_API_KEY (context audit, optional re-extraction) [optional - press Enter to skip]:  ········


  OPENAI_API_KEY       SET      context audit, optional re-extraction


  ANTHROPIC_API_KEY (optional re-judging only) [optional - press Enter to skip]:  ········


  ANTHROPIC_API_KEY    SET      optional re-judging only


  GROQ_API_KEY (optional re-judging only) [optional - press Enter to skip]:  ········


  GROQ_API_KEY         SET      optional re-judging only


`cell 3`

## Part 1 — Corpus and Gold Standard  *(§4.1.1)*

The frozen corpus `gpt4o_extracted_merged.json` (770 triplets from 19 documents) is the
input every downstream number is defined over; its SHA-256 is recorded above.

`RERUN_STEP1 = True` re-extracts from the source PDFs. Extraction walks each document in
overlapping windows of `CHUNK_SIZE` characters with `OVERLAP` characters of overlap, so no
document is truncated. Because extraction is an LLM call, a re-extracted corpus differs
from the frozen one and every downstream number moves with it.

The Gold Standard is 60 manually curated triplets used twice: as the false-alarm control
in Part 7, and as the seed material for the injection test in Part 6.

In [5]:
# ====== cell 4 ======
EXTRACTION_SYSTEM = """You are an expert agricultural knowledge extraction system.
Extract quantitative cultivation-parameter triplets from the given scientific text about wasabi hydroponics.
Return ONLY triplets that contain a measurable parameter (e.g., water temperature, DO, EC, pH, PPFD,
photoperiod, AITC, shading rate, rhizome weight, air temperature, etc.).
Output strict JSON with a single key "triplets" whose value is a list of objects:
  {"subject": <parameter or entity>, "predicate": <relation>, "object": <value with unit if any>,
   "unit": <unit or "">, "condition": <experimental condition or "">}
Rules: keep numeric values verbatim; include the measurement condition when stated; do NOT invent values."""

def chunk_text(text, size=CHUNK_SIZE, overlap=OVERLAP):
    if not text:
        return []
    out, i = [], 0
    while i < len(text):
        out.append(text[i:i + size]); i += size - overlap
    return out

def read_pdf(path):
    text = ""
    for loader in ("pdfplumber", "pypdf", "PyPDF2"):
        try:
            if loader == "pdfplumber":
                import pdfplumber
                with pdfplumber.open(path) as pdf:
                    for pg in pdf.pages:
                        text += (pg.extract_text() or "") + "\n"
            else:
                mod = __import__(loader)
                for pg in mod.PdfReader(str(path)).pages:
                    text += (pg.extract_text() or "") + "\n"
            if text.strip():
                break
        except Exception:
            text = ""
    return re.sub(r"[ \t]+", " ", text)

def rerun_step1(pdf_dir, cache_path=OUT_DIR / "step1_chunked.json"):
    from openai import OpenAI
    cli = OpenAI()
    cache = json.load(open(cache_path, encoding="utf-8")) if cache_path.exists() else {}
    pdfs = sorted(Path(pdf_dir).glob("S*.pdf"),
                  key=lambda p: int(re.match(r"^S(\d+)", p.name).group(1)))
    for p in pdfs:
        sid = p.name[:3]
        if sid in cache:
            print(f"  {sid}: {len(cache[sid])} (cached)"); continue
        seen, merged = set(), []
        for ch in chunk_text(read_pdf(p)):
            r = cli.chat.completions.create(
                model=MODEL_EXTRACT, temperature=0.0,
                response_format={"type": "json_object"},
                messages=[{"role": "system", "content": EXTRACTION_SYSTEM},
                          {"role": "user", "content": f"SOURCE_ID={sid}\n\nTEXT:\n{ch}"}])
            try:    got = json.loads(r.choices[0].message.content).get("triplets", [])
            except Exception: got = []
            for t in got:
                d = {k: str(t.get(k, "")).strip()
                     for k in ("subject", "predicate", "object", "unit", "condition")}
                d["source_id"] = sid
                key = (d["subject"].lower(), d["predicate"].lower(), d["object"].lower())
                if key in seen:
                    continue
                seen.add(key); merged.append(d)
        cache[sid] = merged
        json.dump(cache, open(cache_path, "w", encoding="utf-8"), ensure_ascii=False)
        print(f"  {sid}: {len(merged)}")
    return [t for sid in sorted(cache) for t in cache[sid]]

if RERUN_STEP1:
    print("Step 1: re-extracting from source PDFs")
    TRIPLETS = rerun_step1(DATA_DIR)
    META = {"source": "rerun", "chunk_size": CHUNK_SIZE, "overlap": OVERLAP}
else:
    d = json.load(open(FILES["corpus"], encoding="utf-8"))
    TRIPLETS, META = ((d.get("triplets", d), d.get("metadata", {}))
                      if isinstance(d, dict) else (d, {}))
    print("frozen corpus loaded (RERUN_STEP1 = False)")

for i, t in enumerate(TRIPLETS):
    t.setdefault("id", f"EXT-{i:04d}")

g = json.load(open(FILES["gold"], encoding="utf-8"))
GOLD = g.get("triplets", g) if isinstance(g, dict) else g

by_src = Counter(t.get("source_id") for t in TRIPLETS)
print(f"\ncorpus triplets   : {len(TRIPLETS)} from {len(by_src)} document(s)")
print(f"gold standard     : {len(GOLD)}")
print(f"with a unit       : {sum(1 for t in TRIPLETS if t.get('unit')):>5}")
print(f"with a condition  : {sum(1 for t in TRIPLETS if t.get('condition')):>5}")
print(f"\nper document: {dict(sorted(by_src.items()))}")
MANIFEST["corpus"] = {"n_triplets": len(TRIPLETS), "n_documents": len(by_src),
                      "n_gold": len(GOLD), "rerun": RERUN_STEP1, "metadata": META,
                      "per_document": dict(sorted(by_src.items()))}

frozen corpus loaded (RERUN_STEP1 = False)

corpus triplets   : 770 from 19 document(s)
gold standard     : 60
with a unit       :   670
with a condition  :   663

per document: {'S01': 37, 'S02': 33, 'S03': 75, 'S04': 22, 'S05': 80, 'S06': 69, 'S07': 48, 'S08': 32, 'S09': 77, 'S10': 1, 'S11': 20, 'S12': 23, 'S13': 44, 'S14': 77, 'S15': 33, 'S16': 41, 'S17': 32, 'S18': 17, 'S19': 9}


`cell 5`

## Part 2 — Domain-configurable reference tables  *(§3.2.2)*

Five tables define everything the detector knows about the domain. They live in
`pipeline.py` and are printed below so the configuration is part of the record.

- `THRESHOLDS` — plausible range and canonical unit for each parameter.
- `KNOWN_UNIT_ALIASES` — units that legitimately appear for a parameter in the literature.
- `EQUIVALENT_UNITS` — pairs denoting the same quantity, never flagged (mg/L ≡ ppm).
- `UNIT_CONVERSIONS` — conversion factors, plus two sentinels: `None` for a physically
  non-linear pair that is still a genuine mismatch (lux → µmol·m⁻²·s⁻¹ depends on the
  spectrum), and `"INCOMPATIBLE"` for pairs that are not the same quantity at all
  (`Degree hour` vs `°C`, `% of saturation` vs `mg/L`) and must therefore never be
  treated as a unit mismatch.
- `UNIT_NOTATION_MAP` — spelling variants collapsed before any comparison, so `mg L−1`
  and `mg/L` are not mistaken for different units.

These tables are the only domain-specific component: porting the pipeline to another crop
means replacing them, not the code. The ranges are literature-derived plausibility bounds,
not physical limits — their role is to raise a candidate for contextual review, not to
decide a conflict on their own.

In [7]:
# ====== cell 6 ======
# The four tables are defined in pipeline.py and printed here so the domain configuration
# is visible in the record rather than only in a source file.
print(f"{'parameter':<20}{'min':>8}{'max':>8}   canonical unit      permitted aliases")
print("-" * 92)
for k, v in THRESHOLDS.items():
    print(f"{k:<20}{v['min']:>8}{v['max']:>8}   {v['unit'] or '(dimensionless)':<18}"
          f"  {', '.join(KNOWN_UNIT_ALIASES.get(k, []))}")

extra = [k for k in KNOWN_UNIT_ALIASES if k not in THRESHOLDS]
print(f"\nparameters with aliases but no numeric range: {extra}")
print(f"  (a unit can be checked without a plausible range; T1 simply cannot fire there)")

# sorted() twice: within each pair and across the pairs. EQUIVALENT_UNITS is a set, so
# printing it directly gives a different order in every process - the only other place in
# this notebook where a set reached the output.
print(f"\nequivalent unit pairs (never flagged): "
      f"{sorted(sorted(s) for s in EQUIVALENT_UNITS)}")

incompatible = [k for k, v in UNIT_CONVERSIONS.items() if v == "INCOMPATIBLE"]
nonlinear    = [k for k, v in UNIT_CONVERSIONS.items() if v is None]
print(f"\nunit conversions          : {len(UNIT_CONVERSIONS)}")
print(f"  non-linear (flagged)    : {nonlinear}")
print(f"  incompatible (not a unit mismatch - a different quantity):")
for k in incompatible:
    print(f"      {k[0]:<18} vs {k[1]}")
print(f"\nnotation variants normalised before comparison: {len(UNIT_NOTATION_MAP)}")
for a, b in list(UNIT_NOTATION_MAP.items())[:6]:
    print(f"      {a:<16} -> {b}")
print(f"      ... and {len(UNIT_NOTATION_MAP)-6} more")

MANIFEST["reference_tables"] = {
    "thresholds": {k: {"min": v["min"], "max": v["max"], "unit": v["unit"]}
                   for k, v in THRESHOLDS.items()},
    "n_unit_aliases": len(KNOWN_UNIT_ALIASES),
    "n_conversions": len(UNIT_CONVERSIONS),
    "n_notation_variants": len(UNIT_NOTATION_MAP),
    "incompatible_pairs": [list(k) for k in incompatible],
    "nonlinear_pairs": [list(k) for k in nonlinear]}


parameter                min     max   canonical unit      permitted aliases
--------------------------------------------------------------------------------------------
water_temp                 5      20   °C                  °C, °F
growth_medium_temp         5      23   °C                  °C, °F
air_temp                   5      30   °C                  °C, °F
pH                       5.5     7.5   (dimensionless)     
DO                         6      14   mg/L                mg/L, ppm
EC                      0.05     3.7   dS/m                dS/m, mS/cm, μS/cm
PPF                       50     300   μmol/m²/s           μmol/m²/s, lux
PPFD                      50     900   μmol/m²/s           μmol/m²/s, lux
photoperiod                8      16   hours               hours, minutes
VPD                      0.3     2.5   kPa                 kPa, Pa

parameters with aliases but no numeric range: ['AITC', 'glucosinolates', 'altitude']
  (a unit can be checked without a plausible range

`cell 7`

## Part 3 — Conflict taxonomy and rule-based detector  *(§3.2.1, §3.2.3)*

**No API calls. Fully deterministic.**

| Type | Definition (§3.2.1) | Criterion |
|---|---|---|
| **T1** quantitative inconsistency | a numeric value that cannot be reconciled with the rest of the evidence | (i) a non-overlapping range reported for the same unconditioned parameter **by a different source**, or (ii) a value outside the parameter's plausible range |
| **T2** unit mismatch | a unit that is neither canonical nor a recognised equivalent | deviation from the canonical unit, or from the unit another triplet uses for the same parameter, where a conversion relationship between them is known |
| **T3** conditional-contradiction candidate | an unconditioned value where the literature reports condition-specific values | no `condition`, while a condition-bearing peer for the same parameter exists in the corpus |

The detector is `pipeline.ConflictDetector`, **imported unmodified**. `detect_all` wraps it
and supplies three behaviours the paper specifies that its public `detect()` does not,
each marked `[+]` in the code:

1. **[+] all applicable types, not the first match.** §3.2.3 states that the detector
   evaluates all three criteria in a single call. `detect()` returns at the first hit in
   the order T2 → T1 → T3, so a triplet with a non-canonical unit never reaches the T1 or
   T3 checks. Each flag is audited independently in Part 4.
2. **[+] T1 criterion (i) restricted to distinct sources.** §3.2.2 defines criterion (i)
   as cross-source disagreement. `_check_range_overlap_conflict` compares against every
   unconditioned, unit-compatible triplet regardless of provenance, so the treatment levels
   of one factorial experiment register as a literature conflict. The wrapper adds the
   source filter and reuses the same comparison logic.
3. **[+] T3 peer requirement.** A triplet qualifies only when a condition-bearing peer for
   the same parameter exists elsewhere. Without a peer there is nothing for the missing
   condition to contradict and the flag would be vacuous.

Part 4 ends with a sensitivity table quantifying each addition, so its effect on the
reported counts is visible rather than assumed.

In [9]:
# ====== cell 8 ======
# The detector is pipeline.ConflictDetector, imported unmodified. Its public entry point
# detect() returns the FIRST type that applies, in the order T2 -> T1 -> T3.
detector = ConflictDetector()

print("pipeline.ConflictDetector - checks available")
for name in ("_check_unit_mismatch", "_check_numeric_range", "_check_range_overlap_conflict",
             "_normalize_to_threshold_unit", "units_compatible", "_extract_numbers"):
    print(f"   {name:<32} {'yes' if hasattr(detector, name) else 'MISSING'}")

# number parsing handles the notation actually found in the corpus
for s in ("13.5~28.1", "5-20 °C", "11.4±2.4", "-3", "8 mg/L"):
    print(f"   _extract_numbers({s!r:<14}) -> {detector._extract_numbers(s)}")


pipeline.ConflictDetector - checks available
   _check_unit_mismatch             yes
   _check_numeric_range             yes
   _check_range_overlap_conflict    yes
   _normalize_to_threshold_unit     yes
   units_compatible                 yes
   _extract_numbers                 yes
   _extract_numbers('13.5~28.1'   ) -> [13.5, 28.1]
   _extract_numbers('5-20 °C'     ) -> [5.0, 20.0]
   _extract_numbers('11.4±2.4'    ) -> [11.4]
   _extract_numbers('-3'          ) -> [-3.0]
   _extract_numbers('8 mg/L'      ) -> [8.0]


In [10]:
# ====== cell 9 ======
# ---------------- conformance switches (see the table printed below) ----------------
# Each switch corresponds to a sentence in the paper. Where the paper states one rule
# unambiguously the default follows it. Where the paper contradicts itself the default is
# marked and both readings are quantified, so the choice is made in the manuscript rather
# than silently in the code.

MULTI_LABEL       = True            # §3.2.3 note: "detects all three conflict types in a
                                    # single call". Algorithm 1 lines 5-11 instead make the
                                    # T3 branch an `else if`, mutually exclusive with T1/T2.
                                    # The note describes the implementation, so it wins.

T1_CRITERION_I    = "non_overlapping"   # §3.2.2 text: "two or more sources report
                                    # NON-IDENTICAL values" -> "non_identical".
                                    # Table 3 example: "13-15°C vs 17-20°C" (disjoint) and
                                    # §3.4 warning text -> "non_overlapping".
                                    # PAPER IS INCONSISTENT. Both are computed below.

T1_CROSS_SOURCE   = True            # §3.2.2 names criterion (i) "cross-source disagreement".
                                    # Its worked example, however, compares S01 with S01
                                    # (vermiculite vs rockwool). PAPER IS INCONSISTENT; the
                                    # example contradicts the criterion's own name.

T3_REQUIRE_PEER   = False           # Algorithm 1 line 7 triggers T3 on
                                    # `condition = NULL AND subject IN THRESHOLDS` alone.
                                    # Requiring a condition-bearing peer is an addition that
                                    # the paper does not state; kept available, off by default.

print("conformance switches")
for k, v in [("MULTI_LABEL", MULTI_LABEL), ("T1_CRITERION_I", T1_CRITERION_I),
             ("T1_CROSS_SOURCE", T1_CROSS_SOURCE), ("T3_REQUIRE_PEER", T3_REQUIRE_PEER)]:
    print(f"  {k:<18} = {v}")


conformance switches
  MULTI_LABEL        = True
  T1_CRITERION_I     = non_overlapping
  T1_CROSS_SOURCE    = True
  T3_REQUIRE_PEER    = False


In [11]:
# ====== cell 10 ======
# ---------------- T1 criterion (i): cross-source disagreement (§3.2.2) ----------------
def check_criterion_i(triplet, all_triplets, det=detector,
                      cross_source=None, mode=None):
    """Compare a triplet against other reported values for the same parameter.

    mode="non_overlapping": flag only when the two ranges are disjoint (Table 3 example,
        and what pipeline._check_range_overlap_conflict implements).
    mode="non_identical":   flag whenever the reported values differ at all (the literal
        wording of §3.2.2 and of the §3.4 warning message).
    cross_source=True restricts the comparison to a different source_id, which is what the
        criterion is named after; False compares against every source.
    """
    cross_source = T1_CROSS_SOURCE if cross_source is None else cross_source
    mode = T1_CRITERION_I if mode is None else mode

    subject   = triplet.get("subject", "")
    condition = triplet.get("condition")
    unit      = normalize_unit_notation(triplet.get("unit", ""))
    if condition or not all_triplets:
        return None                       # Algorithm 1 compares unconditioned values only
    nums = det._extract_numbers(str(triplet.get("object", "")))
    if not nums:
        return None
    lo, hi = min(nums), max(nums)
    src = triplet.get("source_id") or triplet.get("source_gs_id")

    for other in all_triplets:
        if other.get("id") == triplet.get("id"):   continue
        if other.get("subject") != subject:        continue
        if other.get("condition"):                 continue
        osrc = other.get("source_id") or other.get("source_gs_id")
        if cross_source and src is not None and osrc == src:
            continue
        ounit = normalize_unit_notation(other.get("unit", ""))
        if not det.units_compatible(unit, ounit):  continue
        onums = det._extract_numbers(str(other.get("object", "")))
        if not onums:                              continue
        if ounit and unit and ounit != unit:
            onums = det._normalize_to_threshold_unit(onums, ounit, unit, subject)
            if onums is None:                      continue
        olo, ohi = min(onums), max(onums)

        if mode == "non_identical":
            hit = (abs(lo - olo) > 1e-9) or (abs(hi - ohi) > 1e-9)
            why = f"Reported value [{lo}, {hi}] is not identical to {other.get('id')}'s [{olo}, {ohi}]"
        else:
            hit = (hi < olo) or (lo > ohi)
            why = f"Range [{lo}, {hi}] does not overlap {other.get('id')}'s [{olo}, {ohi}]"
        if hit:
            return {"reason": why + f" for the same unconditioned parameter"
                            + (f", reported by a different source ({osrc})" if cross_source else ""),
                    "other_id": other.get("id"), "other_source": osrc,
                    "criterion": "cross_source_disagreement" if cross_source else "any_source_disagreement",
                    "mode": mode}
    return None


def has_conditioned_peer(triplet, all_triplets, det=detector):
    subject = triplet.get("subject", "")
    unit    = normalize_unit_notation(triplet.get("unit", ""))
    return any(t.get("subject") == subject and t.get("id") != triplet.get("id")
               and t.get("condition")
               and det.units_compatible(unit, t.get("unit", ""))
               for t in (all_triplets or []))


def detect_all(triplet, all_triplets, det=detector, multi_label=None, cross_source=None,
               mode=None, require_peer=None, algorithm1_exclusive=False):
    """Rule-based detection. Returns [(type, details), ...]; empty means PASS.

    algorithm1_exclusive=True reproduces Algorithm 1 lines 5-11 literally: the T3 branch is
    evaluated only when neither T1 nor T2 fired.
    """
    multi_label  = MULTI_LABEL if multi_label is None else multi_label
    require_peer = T3_REQUIRE_PEER if require_peer is None else require_peer

    if not multi_label and not algorithm1_exclusive:
        ct, d_ = det.detect(triplet, all_triplets)
        return [(ct, d_ or {})] if ct else []

    subject   = triplet.get("subject", "")
    obj_raw   = triplet.get("object", "")
    unit      = normalize_unit_notation(triplet.get("unit", ""))
    # "" and "   " mean the same thing as None here, and the rest of the notebook already
    # treats them that way (`if not t.get("condition")`). The T3 trigger below tests
    # `is None`, so without this an empty string would silently skip T3. The deposited
    # corpus stores all 107 missing conditions as None, so this changes nothing today; it
    # keeps the rule correct if the corpus is ever re-extracted.
    _c = triplet.get("condition")
    condition = None if (_c is None or (isinstance(_c, str) and not _c.strip())) else _c
    flags = []

    t2 = det._check_unit_mismatch(triplet, all_triplets)          # T2 (Table 3)
    if t2:
        flags.append(("T2", t2))

    t1 = det._check_numeric_range(subject, obj_raw, unit)          # criterion (ii)
    if t1:
        t1 = dict(t1, criterion="range_violation")
    else:
        t1 = check_criterion_i(triplet, all_triplets, det, cross_source, mode)  # criterion (i)
    if t1:
        flags.append(("T1", t1))

    # Algorithm 1 line 7: condition is NULL and the subject has a tabulated range.
    if not (algorithm1_exclusive and flags):
        if subject in det.thresholds and condition is None and det._extract_numbers(obj_raw):
            if (not require_peer) or has_conditioned_peer(triplet, all_triplets, det):
                flags.append(("T3_candidate", {
                    "reason": "Numeric value for a parameter in THRESHOLDS with no stated "
                              "condition (Algorithm 1, line 7)",
                    "subject": subject, "value": obj_raw,
                    "criterion": "unconditioned_value"}))
    return flags

print("detect_all ready — parameterised by the conformance switches above")


detect_all ready — parameterised by the conformance switches above


In [12]:
# ====== cell 11 ======
# ------------- what each reading of the paper costs, at the rule stage -------------
# Deterministic and free: it only re-runs the detector under different settings.
READINGS = {
    "as configured above":
        dict(),
    "§3.2.2 literal: non-identical values":
        dict(mode="non_identical"),
    "criterion (i) over any source (§3.2.2 example is S01 vs S01)":
        dict(cross_source=False),
    "Algorithm 1 literal: T3 only when T1/T2 did not fire":
        dict(algorithm1_exclusive=True),
    "first match only (pipeline.detect)":
        dict(multi_label=False),
    "T3 requires a condition-bearing peer (an addition)":
        dict(require_peer=True),
}

print(f"{'reading':<58}{'flagged':>9}{'flags':>7}{'T1':>6}{'T2':>6}{'T3':>6}")
print("-" * 92)
READING_COUNTS = {}
for name, kw in READINGS.items():
    fm = {t.get("id"): detect_all(t, TRIPLETS, **kw) for t in TRIPLETS}
    bt = Counter(ft for f in fm.values() for ft, _ in f)
    READING_COUNTS[name] = {"flagged": sum(1 for f in fm.values() if f),
                            "flags": sum(len(f) for f in fm.values()),
                            **{k: bt.get(k, 0) for k in ("T1", "T2", "T3_candidate")}}
    r = READING_COUNTS[name]
    print(f"{name:<58}{r['flagged']:>9}{r['flags']:>7}{r['T1']:>6}{r['T2']:>6}"
          f"{r['T3_candidate']:>6}")
print("\nThe row selected by the switches is the one used from here on. The others are")
print("printed so the manuscript can state which reading of §3.2.2 and Algorithm 1 it means.")
MANIFEST["rule_stage_readings"] = READING_COUNTS


reading                                                     flagged  flags    T1    T2    T3
--------------------------------------------------------------------------------------------
as configured above                                             133    167    68    65    34
§3.2.2 literal: non-identical values                            133    170    71    65    34
criterion (i) over any source (§3.2.2 example is S01 vs S01)      175    215   116    65    34
Algorithm 1 literal: T3 only when T1/T2 did not fire            133    143    68    65    10
first match only (pipeline.detect)                              175    175   106    65     4
T3 requires a condition-bearing peer (an addition)              133    167    68    65    34

The row selected by the switches is the one used from here on. The others are
printed so the manuscript can state which reading of §3.2.2 and Algorithm 1 it means.


In [13]:
# ====== cell 12 ======
# ---------------------- apply to the corpus (§4.2.1, Step 2) ----------------------
STEP2 = {t.get("id"): detect_all(t, TRIPLETS) for t in TRIPLETS}

n_flagged = sum(1 for f in STEP2.values() if f)
n_flags   = sum(len(f) for f in STEP2.values())
n_multi   = sum(1 for f in STEP2.values() if len(f) > 1)
by_type   = Counter(ft for f in STEP2.values() for ft, _ in f)
t1_crit   = Counter(d.get("criterion") for f in STEP2.values() for ft, d in f if ft == "T1")

print("Step 2 - rule-based detection (deterministic, no API calls)")
print("=" * 60)
print(f"  triplets examined     : {len(TRIPLETS):>5}")
print(f"  flagged               : {n_flagged:>5}   ({n_flagged/len(TRIPLETS):>6.1%})")
print(f"  not flagged           : {len(TRIPLETS)-n_flagged:>5}   ({1-n_flagged/len(TRIPLETS):>6.1%})")
print(f"  flags raised in total : {n_flags:>5}")
print(f"  carrying >1 type      : {n_multi:>5}")
for k in ("T1", "T2", "T3_candidate"):
    print(f"  {k:<21} : {by_type.get(k,0):>5}")
print(f"\n  T1 by criterion       : {dict(t1_crit)}")
print("\n  type combinations:")
for combo, c in Counter(tuple(sorted(ft for ft, _ in f))
                        for f in STEP2.values() if f).most_common():
    print(f"    {str(combo):<36}: {c:>4}")

MANIFEST["step2"] = {"flagged": n_flagged, "total_flags": n_flags, "multi_type": n_multi,
                     "by_type": dict(by_type), "t1_by_criterion": dict(t1_crit)}
json.dump({k: v for k, v in STEP2.items()},
          open(OUT_DIR / "step2_results.json", "w", encoding="utf-8"),
          ensure_ascii=False, default=str)

Step 2 - rule-based detection (deterministic, no API calls)
  triplets examined     :   770
  flagged               :   133   ( 17.3%)
  not flagged           :   637   ( 82.7%)
  flags raised in total :   167
  carrying >1 type      :    33
  T1                    :    68
  T2                    :    65
  T3_candidate          :    34

  T1 by criterion       : {'range_violation': 50, 'cross_source_disagreement': 18}

  type combinations:
    ('T2',)                             :   53
    ('T1',)                             :   37
    ('T1', 'T3_candidate')              :   21
    ('T3_candidate',)                   :   10
    ('T1', 'T2')                        :    9
    ('T2', 'T3_candidate')              :    2
    ('T1', 'T2', 'T3_candidate')        :    1


`cell 13`

## Part 4 — Context auditor and Confidence Level  *(§3.2.4, §4.2.1)*

Every flag from Part 3 is submitted independently to `GPT4oContextAuditor`, imported
unmodified from `run_full_experiment.py`. It is the implementation that produced the
cached results, so a run from a cold cache and a run from the cache agree; the notebook
defines no auditor prompt of its own.

The auditor is not uniformly an LLM call. **T1 and T2** flags go to GPT-4o with the
triplet, the detector's reasoning and the domain thresholds for that parameter. **T3**
passes a deterministic gate first: if every condition-bearing peer for the parameter traces
back to a single source paper there is no independent corroboration, and the flag is
returned as `undefined` with no model call at all. Only when peers come from two or more
papers does the model judge whether the specific comparison is a fair one. That gate is why
some flags carry no TRUE/FALSE verdict; Definition 1 has no level for them, so they are
treated conservatively as cleared, which can only lower the confirmed-conflict count.

Results are cached in `repro_out/step3_audits.json` under `<triplet id>|<conflict type>`;
an entry already present is never re-requested.

**Definition 1** assigns the Confidence Level from the detector type and the auditor verdict
alone, `C(t) = f(D(t), A(t))`:

| D(t) | A(t) | C(t) |
|---|---|---|
| none | — | Verified |
| any | FALSE_ALARM | Verified |
| T1 or T2 | TRUE_CONFLICT | **Conflicted** |
| T3 only | TRUE_CONFLICT | **Disputed** |

The auditor also emits its own re-labelling of the conflict type. That field is recorded but
deliberately excluded from the assignment, so the Confidence Level stays a function of the
two stages rather than of the language model's own taxonomy.

In [15]:
# ====== cell 14 ======
# The auditor is run_full_experiment.GPT4oContextAuditor, imported unmodified. It is the
# implementation that produced the cached results, so a fresh run and the cache agree.
#
# What it does, per flag:
#   T1 / T2  -> one GPT-4o call with the triplet, the detector's reasoning and the domain
#               thresholds for that parameter.
#   T3       -> a DETERMINISTIC gate first: if every condition-bearing peer for the
#               parameter traces back to a single source paper there is no independent
#               corroboration, and the flag is returned as audit_action "undefined"
#               (final_conflict "T3_undefined") without any model call. Only when peers
#               come from two or more papers does GPT-4o judge whether the specific
#               comparison is a fair one.
#   None     -> passes without a call.
#
# Note: the model is hardcoded to "gpt-4o" inside that class, so MODEL_AUDIT below is
# recorded for the manifest but does not override it.
assert HAS_REF_AUDITOR, ("run_full_experiment.GPT4oContextAuditor could not be imported; "
                         "it is required - this notebook does not reimplement the auditor")
MANIFEST["auditor"] = {"class": "run_full_experiment.GPT4oContextAuditor",
                       "model": "gpt-4o (hardcoded in the class)",
                       "t3_gate": "deterministic: peers from a single paper -> undefined"}

_AUDITOR = None

def audit_flag(triplet, ctype, details, all_triplets):
    """One audit. Delegates to the deployed auditor; no prompt is defined here."""
    global _AUDITOR
    if _AUDITOR is None:
        from openai import OpenAI
        _AUDITOR = GPT4oContextAuditor(OpenAI())
    return _AUDITOR.audit(triplet, ctype, details or {}, all_triplets)


def normalize_verdict(audit):
    """Map an auditor response onto TRUE_CONFLICT / FALSE_ALARM / ERROR / None.

    Three states that must not be conflated:
      TRUE_CONFLICT / FALSE_ALARM - the auditor decided.
      ERROR  - the call failed. The item was never audited. Silently treating this as
               "not confirmed" would let an API outage manufacture Verified triplets.
      None   - the auditor answered but the answer is not one of the two verdicts.
    Only the exact known verdict strings are accepted: mapping anything-not-TRUE onto
    FALSE_ALARM turned "MAYBE", "UNCERTAIN" and a truncated reply into clearances.
    """
    if not isinstance(audit, dict):
        return None
    if audit.get("error"):
        return "ERROR"
    for k in ("verdict", "audit_verdict", "decision"):
        v = audit.get(k)
        if v:
            v = re.sub(r"[^A-Z]+", "_", str(v).upper()).strip("_")
            if v in ("TRUE_CONFLICT", "TRUE", "CONFLICT"):   return "TRUE_CONFLICT"
            if v in ("FALSE_ALARM", "FALSE", "ALARM", "PASS"): return "FALSE_ALARM"
            return None
    act = str(audit.get("audit_action", "")).lower()
    if act.startswith("flag"):   return "TRUE_CONFLICT"
    if act in ("clear", "pass"): return "FALSE_ALARM"
    return None


def report_audit_errors(store, label, fatal=True):
    """Refuse to report a run that contains items the auditor never decided.

    A failed call leaves an item unaudited, and an unaudited item silently becomes
    Verified - an API outage would manufacture clean triplets. Printing a warning is not
    enough for a notebook whose output is a published table, so this raises by default.
    Pass fatal=False only to inspect a partial run.
    """
    bad = [k for k, v in store.items()
           if not str(k).startswith("__") and normalize_verdict(v) == "ERROR"]
    if bad:
        msg = (f"{label}: {len(bad)} audit(s) FAILED and were never decided "
               f"(e.g. {bad[:3]}). These items are NOT audited and are NOT false alarms; "
               f"leaving them in would report them as Verified. Re-run the audit cell to "
               f"complete them.")
        if fatal:
            raise RuntimeError(msg)
        print(f"[!] {msg}")
    return bad

def assign_confidence(confirmed_types):
    """Definition 1: C(t) = f(D(t), A(t))."""
    if any(ft in ("T1", "T2") for ft in confirmed_types):
        return "Conflicted"
    if any(str(ft).startswith("T3") for ft in confirmed_types):
        return "Disputed"
    return "Verified"

assert assign_confidence([])                     == "Verified"
assert assign_confidence(["T1"])                 == "Conflicted"
assert assign_confidence(["T2"])                 == "Conflicted"
assert assign_confidence(["T3_candidate"])       == "Disputed"
assert assign_confidence(["T2", "T3_candidate"]) == "Conflicted"
print("Definition 1 mapping: self-test passed")

Definition 1 mapping: self-test passed


In [16]:
# ====== cell 15 ======
# The cache is keyed "<triplet id>|<conflict type>", and is stamped with the detector
# configuration that produced it.
#
# The stamp is not bookkeeping. An audit is a function of the PROMPT, and the prompt
# carries the detector's own reasoning for the flag - which peer triplet it compared
# against and on what criterion. Change a conformance switch and the same triplet-and-type
# key can name a different comparison, so an entry cached under one configuration is not a
# valid answer under another. Without the stamp such a cache loads silently, every flag
# reports as already audited, no call is made, and the run reports numbers that answer a
# question the current detector never asked. That is what produced the discrepancy the
# stability check in the next cell reports.
AUDIT_CACHE = OUT_DIR / "step3_audits.json"
RUN_AUDIT   = True          # False: use the cache as-is, make no calls

DETECTOR_SIGNATURE = {"MULTI_LABEL": MULTI_LABEL, "T1_CRITERION_I": T1_CRITERION_I,
                      "T1_CROSS_SOURCE": T1_CROSS_SOURCE, "T3_REQUIRE_PEER": T3_REQUIRE_PEER,
                      "auditor": "GPT4oContextAuditor"}
_SIG_KEY = "__detector_config__"

def load_audit_cache(path, expect=DETECTOR_SIGNATURE):
    """Return (usable audits, legacy-key count, mismatch description or None)."""
    if not path.exists():
        return {}, 0, None
    raw = json.load(open(path, encoding="utf-8"))
    sig = raw.get(_SIG_KEY)
    usable = {k: v for k, v in raw.items() if "|" in k}
    legacy = len(raw) - len(usable) - (1 if sig is not None else 0)
    if sig is None:
        return usable, legacy, "unstamped (written before the configuration was recorded)"
    if sig != expect:
        d = [f"{k}: {sig.get(k)!r} -> {expect[k]!r}" for k in expect if sig.get(k) != expect[k]]
        return usable, legacy, "; ".join(d)
    return usable, legacy, None

def save_audit_cache(path, audits):
    json.dump({_SIG_KEY: DETECTOR_SIGNATURE, **audits},
              open(path, "w", encoding="utf-8"), ensure_ascii=False, default=str)

AUDITS, n_unusable, mismatch = load_audit_cache(AUDIT_CACHE)
if n_unusable:
    print(f"[cache] {n_unusable} entry/entries in {AUDIT_CACHE.name} use the old "
          f"triplet-id-only key format and are ignored")
if mismatch and AUDITS:
    print(f"[cache] {AUDIT_CACHE.name} was produced under a different configuration "
          f"({mismatch})")
    print("        Those audits answer a different comparison and are DISCARDED. "
          "Re-auditing.")
    AUDITS = {}

# Caches written by earlier sessions are only carried over when their stamp matches. A
# file whose stamp differs, or which has none, is reported and left alone: silently
# importing it is how a run ends up mixing two detector configurations.
for legacy in ("step3_audits_multi.json",):
    lp = OUT_DIR / legacy
    if not lp.exists():
        continue
    old_c, skipped, mm = load_audit_cache(lp)
    if mm:
        print(f"[cache] {legacy} NOT carried over - {mm}")
        print(f"        ({len(old_c)} audit(s) in that file were made for a different "
              f"detector configuration)")
        continue
    added = {k: v for k, v in old_c.items() if k not in AUDITS}
    AUDITS.update(added)
    if added:
        print(f"[cache] {len(added)} audit(s) carried over from {legacy}"
              + (f" ({skipped} ignored)" if skipped else ""))
print(f"[cache] {len(AUDITS)} usable audit(s) available")

todo = [(tid, ft, d_) for tid, flags in STEP2.items()
        for ft, d_ in flags if f"{tid}|{ft}" not in AUDITS]
print(f"flags to audit: {len(todo)} of {n_flags}"
      + (f"   (~${len(todo)*0.004:.2f})" if todo else ""))

if RUN_AUDIT and todo:
    if not HAS_OPENAI:
        raise RuntimeError("OPENAI_API_KEY is not set - re-run Part 0 or set RUN_AUDIT=False")
    from openai import OpenAI
    cli = OpenAI()
    tmap = {t.get("id"): t for t in TRIPLETS}
    for i, (tid, ft, d_) in enumerate(todo, 1):
        try:
            AUDITS[f"{tid}|{ft}"] = audit_flag(tmap[tid], ft, d_, TRIPLETS)
        except Exception as e:
            print(f"  [warn] {tid}|{ft}: {str(e)[:120]}")
            AUDITS[f"{tid}|{ft}"] = {"error": str(e)}
        if i % 20 == 0 or i == len(todo):
            print(f"  {i}/{len(todo)}")
            save_audit_cache(AUDIT_CACHE, AUDITS)
    save_audit_cache(AUDIT_CACHE, AUDITS)
elif not todo:
    print("all flags already audited")

# The auditor declines to judge ("undefined") when the comparison peers all come from one
# source paper. Definition 1 has no such level, so those flags are treated conservatively
# as cleared, which can only lower the confirmed-conflict count. They are listed here so
# the effect is visible rather than silent.
undecided = [f"{tid}|{ft}" for tid, flags in STEP2.items() for ft, _ in flags
             if normalize_verdict(AUDITS.get(f"{tid}|{ft}", {})) is None]
verdicts = Counter(normalize_verdict(AUDITS.get(f"{tid}|{ft}", {}))
                   for tid, flags in STEP2.items() for ft, _ in flags)
print(f"\nA(t) over {n_flags} flags: {dict(verdicts)}")
if undecided:
    print(f"  {len(undecided)} flag(s) returned no verdict (auditor action 'undefined':")
    print(f"  the comparison peers all trace to one source paper); treated as cleared")
    print(f"  {undecided[:8]}")
MANIFEST["step3"] = {"n_usable_audits": len(AUDITS), "n_ignored_legacy_keys": n_unusable,
                     "verdicts": {str(k): v for k, v in verdicts.items()},
                     "undecided_flags": undecided}

[cache] 167 usable audit(s) available
flags to audit: 0 of 167
all flags already audited

A(t) over 167 flags: {'FALSE_ALARM': 116, 'TRUE_CONFLICT': 44, None: 7}
  7 flag(s) returned no verdict (auditor action 'undefined':
  the comparison peers all trace to one source paper); treated as cleared
  ['EXT-0388|T3_candidate', 'EXT-0389|T3_candidate', 'EXT-0496|T3_candidate', 'EXT-0654|T3_candidate', 'EXT-0655|T3_candidate', 'EXT-0659|T3_candidate', 'EXT-0663|T3_candidate']


In [17]:
# ====== cell 16 ======
# ------------------------ Confidence Level distribution (§4.2.1) ------------------------
ROWS = []
for t in TRIPLETS:
    tid   = t.get("id")
    flags = STEP2.get(tid, [])
    confirmed = [ft for ft, _ in flags
                 if normalize_verdict(AUDITS.get(f"{tid}|{ft}", {})) == "TRUE_CONFLICT"]
    ROWS.append(dict(t, step2_types=[ft for ft, _ in flags], confirmed_types=confirmed,
                     confidence_level=assign_confidence(confirmed),
                     auditor_final_type=[(AUDITS.get(f"{tid}|{ft}", {}).get("final_type")
                                          or AUDITS.get(f"{tid}|{ft}", {}).get("final_conflict"))
                                         for ft, _ in flags]))

D = Counter(r["confidence_level"] for r in ROWS)
n = len(ROWS)
print("Confidence Level distribution (Definition 1)")
print("=" * 48)
for k in ("Verified", "Conflicted", "Disputed"):
    print(f"  {k:<12}: {D.get(k,0):>5}   {D.get(k,0)/n:>6.1%}")
print(f"  {'total':<12}: {n:>5}")
print(f"\nconfirmed conflicts : {D.get('Conflicted',0) + D.get('Disputed',0)}")
print(f"confirmed by type   : {dict(Counter(ft for r in ROWS for ft in r['confirmed_types']))}")

print("\nconfirmation rate by detector type")
for ft in ("T1", "T2", "T3_candidate"):
    cand = by_type.get(ft, 0)
    conf = sum(1 for r in ROWS if ft in r["confirmed_types"])
    if cand:
        print(f"  {ft:<14}: {conf:>4} / {cand:<4} = {conf/cand:>6.1%}")

print("\nD(t) x A(t) contingency")
for (d_, a_), c in sorted(Counter((ft, normalize_verdict(AUDITS.get(f'{tid}|{ft}', {})))
                                  for tid, fl in STEP2.items() for ft, _ in fl).items(),
                          key=lambda x: str(x[0])):
    print(f"  {str(d_):<14} x {str(a_):<14}: {c:>4}")

t1_basis = Counter(dict(STEP2[r["id"]]).get("T1", {}).get("criterion")
                   for r in ROWS if "T1" in r["confirmed_types"])
print(f"\nconfirmed T1 by criterion: {dict(t1_basis)}")

# how often the auditor's own label differs from the detector's
relabelled = [(r["id"], ft, at) for r in ROWS
              for ft, at in zip(r["step2_types"], r["auditor_final_type"])
              if at and not str(at).startswith(str(ft)[:2])]
print(f"auditor proposed a different type on {len(relabelled)} flag(s); Definition 1 uses "
      f"the detector's type, so the proposal is recorded and ignored")
for tid_, ft_, at_ in relabelled[:5]:
    print(f"    {tid_}: detector {ft_} -> auditor {at_}")

MANIFEST["confidence"] = {
    "distribution": dict(D),
    "confirmed_by_type": dict(Counter(ft for r in ROWS for ft in r["confirmed_types"])),
    "confirmed_t1_by_criterion": dict(t1_basis),
    "auditor_type_overrides": len(relabelled),
    "auditor_relabel_examples": [{"id": t, "detector": f, "auditor": a}
                                 for t, f, a in relabelled]}
json.dump(ROWS, open(OUT_DIR / "audited_triplets.json", "w", encoding="utf-8"),
          ensure_ascii=False, default=str)
print(f"\nsaved -> {OUT_DIR / 'audited_triplets.json'}")

Confidence Level distribution (Definition 1)
  Verified    :   734    95.3%
  Conflicted  :    36     4.7%
  Disputed    :     0     0.0%
  total       :   770

confirmed conflicts : 36
confirmed by type   : {'T1': 31, 'T3_candidate': 1, 'T2': 12}

confirmation rate by detector type
  T1            :   31 / 68   =  45.6%
  T2            :   12 / 65   =  18.5%
  T3_candidate  :    1 / 34   =   2.9%

D(t) x A(t) contingency
  T1             x FALSE_ALARM   :   37
  T1             x TRUE_CONFLICT :   31
  T2             x FALSE_ALARM   :   53
  T2             x TRUE_CONFLICT :   12
  T3_candidate   x FALSE_ALARM   :   26
  T3_candidate   x TRUE_CONFLICT :    1
  T3_candidate   x None          :    7

confirmed T1 by criterion: {'range_violation': 24, 'cross_source_disagreement': 7}
auditor proposed a different type on 3 flag(s); Definition 1 uses the detector's type, so the proposal is recorded and ignored
    EXT-0449: detector T2 -> auditor T1
    EXT-0506: detector T1 -> auditor T2
   

In [18]:
# ====== cell 17 ======
# ------------- sensitivity of the reported counts to the three Part 3 additions -------------
# Deterministic; uses only cached audits, so a variant whose flags are not in the cache is
# reported with its coverage rather than silently under-counted.
def variant_distribution(flag_map):
    out = Counter()
    for t in TRIPLETS:
        tid = t.get("id")
        conf = [ft for ft, _ in flag_map.get(tid, [])
                if normalize_verdict(AUDITS.get(f"{tid}|{ft}", {})) == "TRUE_CONFLICT"]
        out[assign_confidence(conf)] += 1
    return out

VARIANTS = {
    "as specified (all three)":
        STEP2,
    "without multi-label (first match only)":
        {t.get("id"): detect_all(t, TRIPLETS, multi_label=False) for t in TRIPLETS},
    "without cross-source restriction on T1":
        {t.get("id"): detect_all(t, TRIPLETS, cross_source=False) for t in TRIPLETS},
    "without the T3 peer requirement":
        {t.get("id"): detect_all(t, TRIPLETS, require_peer=False) for t in TRIPLETS},
}

print(f"{'detector variant':<40}{'flags':>7}{'Verif':>8}{'Confl':>7}{'Disp':>6}{'audited':>9}")
print("-" * 77)
sens = {}
for name, fm in VARIANTS.items():
    d = variant_distribution(fm)
    nf = sum(len(v) for v in fm.values())
    cov = sum(1 for tid, fl in fm.items() for ft, _ in fl
              if f"{tid}|{ft}" in AUDITS) / max(1, nf)
    sens[name] = {"flags": nf, **{k: d.get(k, 0) for k in ("Verified", "Conflicted", "Disputed")},
                  "audit_coverage": round(cov, 3)}
    print(f"{name:<40}{nf:>7}{d.get('Verified',0):>8}{d.get('Conflicted',0):>7}"
          f"{d.get('Disputed',0):>6}{cov:>9.0%}")
print("\n'audited' is the share of that variant's flags present in the audit cache; below "
      "100% its confirmed counts are a lower bound.")
MANIFEST["detector_sensitivity"] = sens

detector variant                          flags   Verif  Confl  Disp  audited
-----------------------------------------------------------------------------
as specified (all three)                    167     734     36     0     100%
without multi-label (first match only)      175     734     36     0      73%
without cross-source restriction on T1      215     734     36     0      78%
without the T3 peer requirement             167     734     36     0     100%

'audited' is the share of that variant's flags present in the audit cache; below 100% its confirmed counts are a lower bound.


In [19]:
# ====== cell 18 ======
# ---- Step 3 stability: the auditor is not fully deterministic at temperature = 0 ----
# §4.2.1 reports a stability check across repeated audit runs. Set True once; the runs are
# cached, so every later execution is free and the deposited cache reproduces the figure.
RUN_STABILITY = True       # first run costs (number of flags) x N_STABILITY calls
STAB_CACHE = OUT_DIR / "stability_runs.json"
AUDITOR_VERSION = "GPT4oContextAuditor"      # bump to invalidate runs made by another auditor
# The runs re-audit whatever STEP2 flagged, so they depend on the detector configuration as
# well as on the auditor. Without this stamp, flipping a conformance switch would leave the
# stability figure describing a flag set the notebook no longer produces.
STAB_SIGNATURE = hashlib.sha256(
    json.dumps(DETECTOR_SIGNATURE, sort_keys=True, default=str).encode("utf-8")
).hexdigest()[:12]
_stab = json.load(open(STAB_CACHE, encoding="utf-8")) if STAB_CACHE.exists() else {}
runs = []
if isinstance(_stab, list) or _stab.get("auditor") != AUDITOR_VERSION:
    if _stab:
        print("[cache] stability runs were produced by a different auditor - discarded")
elif _stab.get("detector_signature") not in (None, STAB_SIGNATURE):
    print("[cache] stability runs were produced under a different detector configuration "
          "- discarded")
else:
    runs = _stab.get("runs", [])
    if _stab.get("detector_signature") is None and runs:
        print("[cache] stability runs carry no detector stamp (written before this check). "
              "Kept, but they are only valid if the conformance switches are unchanged.")

if runs:
    print(f"[cache] {len(runs)} stability run(s) loaded")
elif RUN_STABILITY and HAS_OPENAI:
    from openai import OpenAI
    cli = OpenAI()
    tmap = {t.get("id"): t for t in TRIPLETS}
    flat = [(tid, ft, d_) for tid, fl in STEP2.items() for ft, d_ in fl]
    for r in range(1, N_STABILITY + 1):
        conf_by_tid = defaultdict(list)
        for tid, ft, d_ in flat:
            try:    a = audit_flag(tmap[tid], ft, d_, TRIPLETS)
            except Exception: a = {}
            if normalize_verdict(a) == "TRUE_CONFLICT":
                conf_by_tid[tid].append(ft)
        dr = Counter(assign_confidence(conf_by_tid.get(t.get("id"), [])) for t in TRIPLETS)
        runs.append(dict(dr)); print(f"Run {r}/{N_STABILITY}: {dict(dr)}")
        json.dump({"auditor": AUDITOR_VERSION, "detector_signature": STAB_SIGNATURE,
                   "runs": runs},
                  open(STAB_CACHE, "w", encoding="utf-8"), ensure_ascii=False)

if runs:
    print("\nacross runs")
    for k in ("Verified", "Conflicted", "Disputed"):
        v = [x.get(k, 0) for x in runs]
        print(f"  {k:<11}: {min(v)}-{max(v)}  (mean {sum(v)/len(v):.1f})")
    main = {"Verified": D.get("Verified", 0), "Conflicted": D.get("Conflicted", 0),
            "Disputed": D.get("Disputed", 0)}
    inside = {k: (min(x.get(k, 0) for x in runs) <= main[k] <= max(x.get(k, 0) for x in runs))
              for k in main}
    print(f"\n  reported run: {main}")
    print(f"  inside the repeat range: {inside}")
    if not all(inside.values()):
        print("  [!] the reported run falls OUTSIDE the repeat range, and consistently")
        print("      to one side. Temperature-0 drift scatters around the cached value; a")
        print("      one-sided offset does not. The cause is the cached audits and these")
        print("      repeats answering different prompts. The cell above now refuses a")
        print("      cache whose detector configuration differs - re-run from Part 0 with")
        print("      FRESH_RUN = True so every number comes from one auditor pass.")
    print("\n  Quote this range in §4.2.1 in place of the current stability sentence.")
    MANIFEST["stability"] = {"n_runs": len(runs), "runs": runs,
                             "auditor": AUDITOR_VERSION,
                             "reported_run": main, "reported_run_inside_range": inside}
else:
    print("RUN_STABILITY = False and no cache - §4.2.1's stability sentence is unmeasured.")
    print("Set RUN_STABILITY = True once; the runs are cached afterwards.")

[cache] stability runs carry no detector stamp (written before this check). Kept, but they are only valid if the conformance switches are unchanged.
[cache] 5 stability run(s) loaded

across runs
  Verified   : 730-735  (mean 731.8)
  Conflicted : 35-39  (mean 37.6)
  Disputed   : 0-1  (mean 0.6)

  reported run: {'Verified': 734, 'Conflicted': 36, 'Disputed': 0}
  inside the repeat range: {'Verified': True, 'Conflicted': True, 'Disputed': True}

  Quote this range in §4.2.1 in place of the current stability sentence.


In [103]:
# ═══════════════════════════════════════════════════════════════════════════
# Part 4 진단 — T3 게이트의 peer-list 절단(max_items=8)이 기권 7건에 미친 영향
#
# 목적: audit()의 T3 게이트는 _find_other_conditioned_values(..., max_items=8)이
#       코퍼스 순서로 8개를 채우면 break하므로, distinct_papers가 "모든 조건부
#       동료"가 아니라 "앞 8개만 본 결과"다. 절단이 없었다면 모델 판단으로
#       넘어갔을 항목을 찾아, 그 항목들만 _judge_t3_relevance로 실제 판정한다.
#
# 안전: 보고된 산출물을 일절 변경하지 않는다. AUDITS / audited_triplets.json /
#       CONFIDENCE 등급 / 그래프를 건드리지 않는다. 별도 파일에만 기록한다.
# 비용: 절단 영향을 받은 항목 수만큼 GPT-4o 호출 (예상 5회).
# ═══════════════════════════════════════════════════════════════════════════
import json
from collections import Counter
from pipeline import ConflictDetector

_uc = ConflictDetector.units_compatible

# ── 1. 감사된 트리플을 파일에서 읽는다 (메모리 상태를 건드리지 않기 위해) ──
_ap = OUT_DIR / "audited_triplets.json"
assert _ap.exists(), f"{_ap} 없음 — Part 4를 먼저 실행할 것"
_ROWS = json.load(open(_ap, encoding="utf-8"))
print(f"[diag] audited_triplets.json: {len(_ROWS)} rows")


# ── 2. 게이트의 peer 수집 로직을 그대로 재현한다 ──
def _peer_papers(t, cap):
    """run_full_experiment._find_other_conditioned_values 와 동일한 필터·순서."""
    subj, unit = t.get("subject"), t.get("unit", "")
    out = []
    for o in _ROWS:
        if o.get("id") == t.get("id"):          continue
        if o.get("subject") != subj:            continue
        if not o.get("condition"):              continue
        if unit and not _uc(unit, o.get("unit", "")):
            continue
        out.append(o)
        if len(out) >= cap:
            break
    return out


_T3 = [r for r in _ROWS if "T3_candidate" in (r.get("step2_types") or [])]
_INF = 10 ** 9

_gated, _truncated = [], []
for _t in _T3:
    _p8 = _peer_papers(_t, 8)
    if len({x["source_id"] for x in _p8}) > 1:
        continue                                   # 게이트가 안 걸린 항목
    _pall = _peer_papers(_t, _INF)
    _n_all = len({x["source_id"] for x in _pall})
    (_truncated if _n_all > 1 else _gated).append((_t, _pall, _n_all))

print(f"[diag] T3 candidates          : {len(_T3)}")
print(f"[diag] gate fired (undefined) : {len(_gated) + len(_truncated)}")
print(f"[diag]   진짜 단일출처         : {len(_gated)}")
for _t, _, _n in _gated:
    print(f"           {_t['id']}  {_t['subject']} ({_t.get('unit')})  sources={_n}")
print(f"[diag]   cap=8 절단 artefact  : {len(_truncated)}")
for _t, _, _n in _truncated:
    print(f"           {_t['id']}  {_t['subject']} ({_t.get('unit')})  "
          f"uncapped sources={_n}  {_t['predicate']}={_t['object']}")

if not _truncated:
    print("\n[diag] 절단 영향 항목 없음 — 모델 호출 없이 종료")


# ── 3. 감사자 객체 확보 (모델 호출 없이) ──
def _get_auditor():
    a = globals().get("_AUDITOR")
    if a is not None:
        return a
    try:                                            # 노트북의 지연 생성 경로를 깨움
        audit_flag(_ROWS[0], None, {}, _ROWS)       # conflict_type=None → 호출 없음
        a = globals().get("_AUDITOR")
        if a is not None:
            return a
    except Exception as e:
        print(f"[diag] audit_flag 경유 실패: {e}")
    from run_full_experiment import GPT4oContextAuditor
    for _args in ([], [OPENAI_API_KEY] if "OPENAI_API_KEY" in globals() else None):
        if _args is None:
            continue
        try:
            return GPT4oContextAuditor(*_args)
        except Exception as e:
            print(f"[diag] GPT4oContextAuditor{tuple(_args)} 실패: {e}")
    raise RuntimeError("감사자 객체를 만들지 못했다 — 오류 메시지를 알려줄 것")


# ── 4. 절단 영향 항목만 실제 판정 ──
T3_CAP_DIAG = {"n_t3_candidates": len(_T3),
               "n_gate_fired": len(_gated) + len(_truncated),
               "n_genuine_single_source": len(_gated),
               "n_truncated_by_cap": len(_truncated),
               "genuine_ids": [t["id"] for t, _, _ in _gated],
               "verdicts": {}}

if _truncated:
    _AUD = _get_auditor()
    print(f"\n[diag] {len(_truncated)}건에 대해 _judge_t3_relevance 호출\n")
    for _t, _pall, _n in _truncated:
        try:
            _r = _AUD._judge_t3_relevance(_t, _pall)
        except Exception as e:
            _r = {"audit_action": f"ERROR: {e}", "final_conflict": None}
        _act = _r.get("audit_action")
        _fin = _r.get("final_conflict")
        T3_CAP_DIAG["verdicts"][_t["id"]] = {
            "subject": _t["subject"], "unit": _t.get("unit"),
            "predicate": _t["predicate"], "object": _t["object"],
            "source_id": _t["source_id"], "uncapped_sources": _n,
            "audit_action": _act, "final_conflict": _fin,
            "audit_log": str(_r.get("audit_log", ""))[:400]}
        print(f"  {_t['id']}  {_t['subject']:20s} → action={_act!r}  final={_fin!r}")

    _conf = [k for k, v in T3_CAP_DIAG["verdicts"].items()
             if v["final_conflict"] == "T3"]
    T3_CAP_DIAG["n_would_be_confirmed"] = len(_conf)
    T3_CAP_DIAG["would_be_confirmed_ids"] = _conf

    print(f"\n[diag] 절단이 없었다면 T3로 확정되었을 항목: "
          f"{len(_conf)}/{len(_truncated)}  {_conf}")
    if _conf:
        print("[diag] → 보고된 Disputed = 0 은 부분적으로 이 절단의 결과다. "
              "§5.6에 건수를 명시할 것.")
    else:
        print("[diag] → 절단이 없었어도 전부 기각된다. "
              "Disputed = 0 은 이 절단에 의존하지 않는다. §5.6에 그렇게 쓸 것.")

# ── 5. 별도 파일에만 기록 (보고된 산출물은 그대로 둔다) ──
_dp = OUT_DIR / f"t3_cap_diagnostic{OUT_TAG}.json"
json.dump(T3_CAP_DIAG, open(_dp, "w", encoding="utf-8"),
          ensure_ascii=False, indent=2, default=str)
print(f"\n[diag] saved → {_dp.name}")
if "MANIFEST" in globals():
    MANIFEST["t3_cap_diagnostic"] = T3_CAP_DIAG
    print("[diag] MANIFEST['t3_cap_diagnostic'] 기록됨 "
          "(Part 14를 다시 돌려야 파일에 반영된다)")


[diag] audited_triplets.json: 770 rows
[diag] T3 candidates          : 34
[diag] gate fired (undefined) : 7
[diag]   진짜 단일출처         : 2
           EXT-0388  air_temp (Degree hour)  sources=1
           EXT-0389  air_temp (Degree hour)  sources=1
[diag]   cap=8 절단 artefact  : 5
           EXT-0496  VPD (kPa)  uncapped sources=2  upper_limit=2.0
           EXT-0654  growth_medium_temp (°C)  uncapped sources=3  optimal_range=12~13
           EXT-0655  growth_medium_temp (°C)  uncapped sources=3  upper_limit=18
           EXT-0659  growth_medium_temp (°C)  uncapped sources=3  set_value=13
           EXT-0663  growth_medium_temp (°C)  uncapped sources=3  set_value=20

[diag] 5건에 대해 _judge_t3_relevance 호출

  EXT-0496  VPD                  → action='clear'  final=None
  EXT-0654  growth_medium_temp   → action='clear'  final=None
  EXT-0655  growth_medium_temp   → action='clear'  final=None
  EXT-0659  growth_medium_temp   → action='clear'  final=None
  EXT-0663  growth_medium_temp   → action

`cell 19`

## Part 5 — Representative cases and failure analysis  *(§4.2.2, §4.2.3)*

The cells below select the cases the paper discusses directly from the audited corpus, so
the examples quoted in the text can be traced to triplet identifiers rather than retyped.

In [21]:
# ====== cell 20 ======
tmap = {t.get("id"): t for t in TRIPLETS}

def show(tid, ft):
    t = tmap[tid]; a = AUDITS.get(f"{tid}|{ft}", {})
    d = dict(STEP2[tid]).get(ft, {})
    print(f"  {tid} | {t.get('subject')} = {t.get('object')} {t.get('unit') or ''} "
          f"| source {t.get('source_id')} | condition: {t.get('condition') or 'none'}")
    print(f"      rule    : {str(d.get('reason'))[:150]}")
    print(f"      verdict : {normalize_verdict(a)}  ({str(a.get('reasoning') or a.get('audit_log') or '-')[:130]})")

print("=" * 78)
print("CONFIRMED CROSS-SOURCE T1  (§4.2.2: two sources disagree, ranges do not overlap)")
print("=" * 78)
shown = 0
for r in ROWS:
    if "T1" not in r["confirmed_types"]:
        continue
    if dict(STEP2[r["id"]]).get("T1", {}).get("criterion") != "cross_source_disagreement":
        continue
    show(r["id"], "T1"); shown += 1
    if shown >= 4: break
if not shown:
    print("  none in this run")

print("\n" + "=" * 78)
print("CONFIRMED T2  (§4.2.2: a unit that is not a recognised equivalent)")
print("=" * 78)
for r in [x for x in ROWS if "T2" in x["confirmed_types"]][:3]:
    show(r["id"], "T2")

print("\n" + "=" * 78)
print("CONFIRMED T3 -> Disputed  (§4.2.2: unconditioned where conditions matter)")
print("=" * 78)
dis = [x for x in ROWS if x["confidence_level"] == "Disputed"]
for r in dis[:3]:
    show(r["id"], "T3_candidate")
if not dis:
    print("  none: no T3 candidate was confirmed in this run.")
    print("  The Disputed level is then defined and exercised (Part 6) but unpopulated in")
    print("  the case study -- a property of this corpus, to be stated in §5.4.")

CONFIRMED CROSS-SOURCE T1  (§4.2.2: two sources disagree, ranges do not overlap)
  EXT-0652 | air_temp = 12~15 °C | source S15 | condition: none
      rule    : Range [12.0, 15.0] does not overlap EXT-0683's [25.0, 25.0] for the same unconditioned parameter, reported by a different source (S16)
      verdict : TRUE_CONFLICT  (The conflict arises from a numeric range disagreement between two sources regarding the optimal air temperature range for wasabi h)
  EXT-0653 | air_temp = 20 °C | source S15 | condition: none
      rule    : Range [20.0, 20.0] does not overlap EXT-0682's [8.0, 18.0] for the same unconditioned parameter, reported by a different source (S16)
      verdict : TRUE_CONFLICT  (The conflict arises from a numeric range discrepancy between two sources regarding the upper limit of air temperature for wasabi h)
  EXT-0656 | DO = 9.5 ppm | source S15 | condition: none
      rule    : Range [9.5, 9.5] does not overlap EXT-0033's [5.0, 5.0] for the same unconditioned parameter

In [22]:
# ====== cell 21 ======
# ----------------------------- failure analysis (§4.2.3) -----------------------------
print("Rule-stage flags cleared by the context auditor, by type")
print("=" * 62)
cleared = defaultdict(list)
for tid, flags in STEP2.items():
    for ft, d_ in flags:
        if normalize_verdict(AUDITS.get(f"{tid}|{ft}", {})) == "FALSE_ALARM":
            cleared[ft].append((tid, d_))
for ft in ("T1", "T2", "T3_candidate"):
    tot = by_type.get(ft, 0)
    print(f"\n{ft}: {len(cleared[ft])}/{tot} cleared"
          + (f" = {len(cleared[ft])/tot:.1%}" if tot else ""))
    for tid, d_ in cleared[ft][:3]:
        t = tmap[tid]; a = AUDITS.get(f"{tid}|{ft}", {})
        print(f"   {tid} | {t.get('subject')} = {t.get('object')} {t.get('unit') or ''}")
        print(f"       auditor: {str(a.get('reasoning') or a.get('audit_log') or '-')[:150]}")

unparsed = [(tid, ft) for tid, fl in STEP2.items() for ft, _ in fl
            if normalize_verdict(AUDITS.get(f"{tid}|{ft}", {})) is None]
print(f"\nflags with no parsable verdict: {len(unparsed)}"
      + (f"  {unparsed[:6]}" if unparsed else ""))
print("These are treated conservatively as cleared, which can only lower the reported "
      "conflict count.")
MANIFEST["failure_analysis"] = {"cleared_by_type": {k: len(v) for k, v in cleared.items()},
                                "unparsed": len(unparsed)}

Rule-stage flags cleared by the context auditor, by type

T1: 37/68 cleared = 54.4%
   EXT-0017 | pH = 8.0 null
       auditor: The pH value of 8.0 is outside the typical range of 5.5 to 7.5 for wasabi hydroponic cultivation. However, the condition specifies that this measureme
   EXT-0032 | pH = 6–7.5 
       auditor: The optimal pH range for wasabi hydroponic cultivation is generally accepted to be between 5.5 and 7.5, as supported by domain thresholds from sources
   EXT-0034 | DO = 9 mg L−1
       auditor: The detected conflict is based on a numeric range discrepancy between two sources (S01 and S15). However, both values (9 mg/L from S01 and 9.5 mg/L fr

T2: 53/65 cleared = 81.5%
   EXT-0004 | EC = 0.6 dS m−1
       auditor: The conflict detected is a unit mismatch between dS/m and μS/cm. However, this is a false positive because the units are directly convertible: 1 dS/m 
   EXT-0005 | EC = 1.3 dS m−1
       auditor: The conflict detected is a unit mismatch between dS/m and μS/cm

`cell 22`

## Part 6 — Injection test  *(§4.2.4)*

The corpus results in Part 4 measure how many conflicts the pipeline *finds*, but not how
many it *misses*: the true number of conflicts in the literature is unknown. The injection
test supplies that missing denominator. Each of the 60 Gold-Standard triplets is corrupted
in a controlled way and fed back through the pipeline, so recall is measurable against a
known ground truth.

The corruption rules are the author's, transcribed unchanged:

| | Corruption | Yield |
|---|---|---|
| **T1** | move the value outside the parameter's plausible range by 25% of the range width, alternating above and below by triplet index | one per Gold triplet whose subject has a threshold |
| **T2** | replace the unit with a realistic but wrong one for that parameter | one per Gold triplet with a unit and a defined wrong unit |
| **T3** | delete the `condition` field, leaving the value unqualified | one per Gold triplet that has a condition |

### Two audit protocols

The Step 3 auditor is run under two protocols, reported side by side, on the **same**
detections:

- **Corpus-referenced** (primary). The auditor receives the corrupted triplet and the
  wasabi corpus, exactly as in deployment. It is never shown the uncorrupted original.
  This is the protocol used everywhere else in the paper.
- **Oracle-referenced** (upper bound). The auditor is additionally shown the original
  Gold-Standard entry the item was derived from and asked whether the item conflicts with
  it. No deployed system has this information, so the result is a ceiling on what the
  audit stage could achieve given a perfect reference, not a measure of the system.

Reporting both separates the detector's contribution from the auditor's, and makes the
gap between them explicit rather than leaving the primary number to be read as the ceiling.

Detection and classification are also reported separately: a corruption caught under a
different type than the one injected is a detection success and a classification error.

In [24]:
# ====== cell 23 ======
# ---------------- injection construction (author's original rules) ----------------
WRONG_UNIT = {
    "water_temp": "°F", "air_temp": "°F", "growth_medium_temp": "°F", "summer_air_temp": "°F",
    "EC": "μS/cm", "PPF": "lux", "PPFD": "lux", "Amax": "lux", "VPD": "Pa", "DO": "μg/L",
    "glucosinolates": "μmol/kg DW", "altitude": "ft", "root_nitrogen": "mg/g DW",
    "rhizome_weight": "kg", "shading_rate": "ratio", "soil_moisture": "m3/m3", "CO2": "%",
    "GBPs": "μmol/kg DW", "WUE": "mmol/mol", "photoperiod": "minutes",
}

def wrong_unit_for(subject, current_unit):
    if subject == "AITC":
        return "mg/kg" if "mg/g" in (current_unit or "") else "mg/g"
    return WRONG_UNIT.get(subject)

def build_injections(gold):
    inj = []
    for t in gold:                                        # T1: value out of range
        if t["subject"] not in THRESHOLDS:
            continue
        th = THRESHOLDS[t["subject"]]
        width = th["max"] - th["min"]
        idx = int(t["id"].split("-")[1])
        val = round(th["max"] + width * 0.25, 2) if idx % 2 == 0 else round(th["min"] - width * 0.25, 2)
        inj.append({"id": t["id"] + "-T1", "type": "T1", "subject": t["subject"],
                    "predicate": t["predicate"], "object": str(val), "unit": t.get("unit"),
                    "condition": t.get("condition"), "source_gs_id": t["id"]})
    for t in gold:                                        # T2: wrong unit
        wu = wrong_unit_for(t["subject"], t.get("unit")) if t.get("unit") else None
        if not wu:
            continue
        inj.append({"id": t["id"] + "-T2", "type": "T2", "subject": t["subject"],
                    "predicate": t["predicate"], "object": t["object"], "unit": wu,
                    "condition": t.get("condition"), "source_gs_id": t["id"]})
    for t in gold:                                        # T3: condition removed
        if not t.get("condition"):
            continue
        inj.append({"id": t["id"] + "-T3", "type": "T3", "subject": t["subject"],
                    "predicate": t["predicate"], "object": t["object"], "unit": t.get("unit"),
                    "condition": None, "source_gs_id": t["id"]})
    return inj

INJ = build_injections(GOLD)
gs_by_id = {t["id"]: t for t in GOLD}
print(f"injections built from {len(GOLD)} Gold triplets: "
      f"{dict(Counter(i['type'] for i in INJ))}  total {len(INJ)}")

# Detection context is the wasabi corpus ALONE, so a corrupted item is judged against the
# literature and never against its own sibling corruptions.
#
# This previously read TRIPLETS + INJ, which put all 155 corruptions into the peer set: a
# corrupted unit could be "confirmed" by another corrupted unit for the same parameter, and
# a condition-stripped item could find its condition-bearing peer among the injections
# rather than in the literature. That inflated rule-stage detection from 122/155 to
# 133/155 (T2 66.7% -> 77.2%, T3 76.7% -> 85.0%). The corrected figures are the lower ones.
#
# Re-auditing is NOT required: the 199 surviving flags are a strict subset of the 266 that
# were cached, and the auditor's context section (built by
# _find_other_conditioned_values) is byte-identical for all 199 under either context, which
# was checked before making this change. 67 cached audits simply go unused.
CONTEXT = TRIPLETS
inj_flags = {it["id"]: detect_all(it, CONTEXT) for it in INJ}
print(f"flagged: {sum(1 for f in inj_flags.values() if f)}/{len(INJ)} | "
      f"total flags {sum(len(f) for f in inj_flags.values())}")

injections built from 60 Gold triplets: {'T1': 38, 'T2': 57, 'T3': 60}  total 155
flagged: 122/155 | total flags 199


In [25]:
# ====== cell 24 ======
# --------------------------- Step 3 under both protocols ---------------------------
INJ_CACHE  = OUT_DIR / "injection_audits_corpus.json"
ORACLE_CACHE = OUT_DIR / "injection_audits_oracle.json"
RUN_INJ_CORPUS = True       # primary protocol
RUN_INJ_ORACLE = True      # upper-bound protocol; set True once to measure the ceiling
                            # (~170 calls, cached afterwards) so §4.2.4 can report both

icorpus = json.load(open(INJ_CACHE, encoding="utf-8")) if INJ_CACHE.exists() else {}
for legacy in ("injection_step3_multi.json", "injection_audits.json"):
    p = OUT_DIR / legacy
    if p.exists():
        old = json.load(open(p, encoding="utf-8"))
        add = {k: v for k, v in old.items() if k not in icorpus}
        icorpus.update(add)
        if add:
            print(f"[cache] {len(add)} audit(s) carried over from {legacy}")
ioracle = json.load(open(ORACLE_CACHE, encoding="utf-8")) if ORACLE_CACHE.exists() else {}

ORACLE_PROMPT = """You are a scientific knowledge auditor for a wasabi hydroponic cultivation knowledge graph.
A rule-based detector flagged the following triplet as a potential {conflict_type} conflict:
  Subject: {subject}
  Predicate: {predicate}
  Object: {object} {unit}
  Condition: {condition}
  Detector reasoning: {details}

Reference entry for the same underlying fact, before modification:
  {reference}

Decide whether the flagged triplet is a TRUE_CONFLICT (a genuine numeric, unit or contextual
error relative to the reference) or a FALSE_ALARM. Respond in JSON:
{{"verdict": "TRUE_CONFLICT" or "FALSE_ALARM", "confidence": 0.0-1.0, "reasoning": "..."}}"""

def audit_oracle(cli, it, ctype, details, gs_row, model=MODEL_AUDIT):
    ref = (f"{gs_row.get('subject','')} {gs_row.get('predicate','')} = {gs_row.get('object','')} "
           f"{gs_row.get('unit','') or ''} (condition: {gs_row.get('condition','none')}, "
           f"source: {gs_row.get('ref','')})")
    r = cli.chat.completions.create(
        model=model, temperature=0.0, response_format={"type": "json_object"}, max_tokens=400,
        messages=[{"role": "user", "content": ORACLE_PROMPT.format(
            conflict_type=ctype, subject=it["subject"], predicate=it["predicate"],
            object=it["object"], unit=it.get("unit") or "", condition=it.get("condition") or "none",
            details=json.dumps(details, ensure_ascii=False, default=str), reference=ref)}])
    return json.loads(r.choices[0].message.content)

imap = {it["id"]: it for it in INJ}
todo_c = [(i, ft, d_) for i, fl in inj_flags.items() for ft, d_ in fl if f"{i}|{ft}" not in icorpus]
todo_o = [(i, ft, d_) for i, fl in inj_flags.items() for ft, d_ in fl if f"{i}|{ft}" not in ioracle]
print(f"corpus-referenced to audit: {len(todo_c)} | oracle-referenced to audit: {len(todo_o)}")

if RUN_INJ_CORPUS and todo_c and HAS_OPENAI:
    from openai import OpenAI
    cli = OpenAI()
    for k, (i, ft, d_) in enumerate(todo_c, 1):
        try:    icorpus[f"{i}|{ft}"] = audit_flag(imap[i], ft, d_, CONTEXT)
        except Exception as e: icorpus[f"{i}|{ft}"] = {"error": str(e)}
        if k % 20 == 0 or k == len(todo_c):
            print(f"  corpus {k}/{len(todo_c)}")
            json.dump(icorpus, open(INJ_CACHE, "w", encoding="utf-8"), ensure_ascii=False, default=str)
json.dump(icorpus, open(INJ_CACHE, "w", encoding="utf-8"), ensure_ascii=False, default=str)

if RUN_INJ_ORACLE and todo_o and HAS_OPENAI:
    from openai import OpenAI
    cli = OpenAI()
    for k, (i, ft, d_) in enumerate(todo_o, 1):
        try:    ioracle[f"{i}|{ft}"] = audit_oracle(cli, imap[i], ft, d_, gs_by_id[imap[i]["source_gs_id"]])
        except Exception as e: ioracle[f"{i}|{ft}"] = {"error": str(e)}
        if k % 20 == 0 or k == len(todo_o):
            print(f"  oracle {k}/{len(todo_o)}")
            json.dump(ioracle, open(ORACLE_CACHE, "w", encoding="utf-8"), ensure_ascii=False, default=str)
json.dump(ioracle, open(ORACLE_CACHE, "w", encoding="utf-8"), ensure_ascii=False, default=str)
print(f"cached: corpus {len(icorpus)} | oracle {len(ioracle)}")

corpus-referenced to audit: 0 | oracle-referenced to audit: 0
cached: corpus 266 | oracle 266


In [65]:
# ====== cell 25 ======
# ----------------------------------- results (§4.2.4) -----------------------------------
def score(cache):
    per, det_n, conf_n, cov_n, cov_d = Counter(), 0, 0, 0, 0
    for it in INJ:
        fl = inj_flags[it["id"]]
        ty = it["type"]
        ok = any(normalize_verdict(cache.get(f"{it['id']}|{ft}", {})) == "TRUE_CONFLICT"
                 for ft, _ in fl)
        per[(ty, "n")]       += 1
        per[(ty, "det")]     += 1 if fl else 0
        per[(ty, "type_ok")] += 1 if any(str(ft).startswith(ty) for ft, _ in fl) else 0
        per[(ty, "conf")]    += 1 if ok else 0
        det_n  += 1 if fl else 0
        conf_n += 1 if ok else 0
        cov_d  += len(fl)
        cov_n  += sum(1 for ft, _ in fl if f"{it['id']}|{ft}" in cache)
    return per, det_n, conf_n, (cov_n / cov_d if cov_d else 0.0)

per, det_n, conf_n_c, cov_c = score(icorpus)
_,   _,     conf_n_o, cov_o = score(ioracle)

print("Step 2 - rule-based detection (deterministic)")
print(f"{'injected':<10}{'n':>5}{'detected':>18}{'type match':>18}")
for ty in ("T1", "T2", "T3"):
    n_ = per[(ty, "n")]
    if not n_: continue
    f = lambda k: f"{per[(ty,k)]:>4} ({per[(ty,k)]/n_:>6.1%})"
    print(f"{ty:<10}{n_:>5}{f('det'):>18}{f('type_ok'):>18}")
print(f"{'total':<10}{len(INJ):>5}{f'{det_n} ({det_n/len(INJ):.1%})':>18}")

print("\nEnd-to-end conflict detection rate, by audit protocol")
print(f"{'injected':<10}{'n':>5}{'corpus-referenced':>22}{'oracle-referenced':>22}")
for ty in ("T1", "T2", "T3"):
    n_ = per[(ty, "n")]
    if not n_: continue
    c = sum(1 for it in INJ if it["type"] == ty and any(
        normalize_verdict(icorpus.get(f"{it['id']}|{ft}", {})) == "TRUE_CONFLICT"
        for ft, _ in inj_flags[it["id"]]))
    o = sum(1 for it in INJ if it["type"] == ty and any(
        normalize_verdict(ioracle.get(f"{it['id']}|{ft}", {})) == "TRUE_CONFLICT"
        for ft, _ in inj_flags[it["id"]]))
    print(f"{ty:<10}{n_:>5}{f'{c} ({c/n_:>6.1%})':>22}{f'{o} ({o/n_:>6.1%})':>22}")
print(f"{'total':<10}{len(INJ):>5}"
      f"{f'{conf_n_c} ({conf_n_c/len(INJ):.1%})':>22}{f'{conf_n_o} ({conf_n_o/len(INJ):.1%})':>22}")
print(f"\nconditional on detection : corpus {conf_n_c}/{det_n} = {conf_n_c/det_n:.1%}"
      + (f" | oracle {conf_n_o}/{det_n} = {conf_n_o/det_n:.1%}" if conf_n_o else ""))
print(f"audit coverage           : corpus {cov_c:.0%} | oracle {cov_o:.0%}")
if cov_o < 1.0:
    print("  The oracle column is incomplete; set RUN_INJ_ORACLE = True to measure the ceiling.")
print("\nReport the corpus-referenced column as the system's detection rate. The oracle "
      "column is an upper bound obtained by giving the auditor the uncorrupted entry, "
      "which no deployed system has.")

MANIFEST["injection"] = {
    "n": len(INJ), "detected": det_n,
    "confirmed_corpus_referenced": conf_n_c, "confirmed_oracle_referenced": conf_n_o,
    "coverage": {"corpus": round(cov_c, 3), "oracle": round(cov_o, 3)},
    "by_type": {ty: {k: per[(ty, k)] for k in ("n", "det", "type_ok", "conf")}
                for ty in ("T1", "T2", "T3") if per[(ty, "n")]}}

Step 2 - rule-based detection (deterministic)
injected      n          detected        type match
T1           38       38 (100.0%)       38 (100.0%)
T2           57       38 ( 66.7%)       38 ( 66.7%)
T3           60       46 ( 76.7%)       38 ( 63.3%)
total       155       122 (78.7%)

End-to-end conflict detection rate, by audit protocol
injected      n     corpus-referenced     oracle-referenced
T1           38           35 ( 92.1%)           38 (100.0%)
T2           57           36 ( 63.2%)           38 ( 66.7%)
T3           60           25 ( 41.7%)           42 ( 70.0%)
total       155            96 (61.9%)           118 (76.1%)

conditional on detection : corpus 96/122 = 78.7% | oracle 118/122 = 96.7%
audit coverage           : corpus 100% | oracle 100%

Report the corpus-referenced column as the system's detection rate. The oracle column is an upper bound obtained by giving the auditor the uncorrupted entry, which no deployed system has.


`cell 26`

## Part 7 — Gold-Standard false-alarm rate and recall  *(§4.2.5, §4.2.6)*

The Gold Standard is curated and internally consistent, so every rule-stage flag raised on
it is by construction a false alarm. Two quantities follow: the rate at which the rule
stage fires on clean data, and the rate that survives the context audit.

Recall is the complementary question — how much of the curated material the extraction
pipeline recovered from the raw PDFs. Because the Gold Standard is written in normalised
form while extraction preserves the source wording, matching uses TF-IDF cosine similarity
rather than string equality; the exact-match figure is printed alongside to show why.

In [28]:
# ====== cell 27 ======
# -------------------------- false-alarm rate (§4.2.5) --------------------------
gs_flags = {g["id"]: detect_all(g, GOLD) for g in GOLD}
flagged = {k: v for k, v in gs_flags.items() if v}
print(f"rule stage: {len(flagged)}/{len(GOLD)} = {len(flagged)/len(GOLD):.1%} false-alarm rate")
for gid, fl in flagged.items():
    g = gs_by_id[gid]
    print(f"  {gid} | {g['subject']} = {g['object']} {g.get('unit') or ''} "
          f"| {[ft for ft, _ in fl]} | condition: {g.get('condition') or 'none'}")

GS_CACHE = OUT_DIR / "gold_audits.json"
gaudits = json.load(open(GS_CACHE, encoding="utf-8")) if GS_CACHE.exists() else {}
gtodo = [(gid, ft, d_) for gid, fl in flagged.items() for ft, d_ in fl
         if f"{gid}|{ft}" not in gaudits]

if gtodo and HAS_OPENAI:
    from openai import OpenAI
    cli = OpenAI()
    print(f"\nauditing {len(gtodo)} gold flag(s)")
    for gid, ft, d_ in gtodo:
        try:    gaudits[f"{gid}|{ft}"] = audit_flag(gs_by_id[gid], ft, d_, GOLD)
        except Exception as e: gaudits[f"{gid}|{ft}"] = {"error": str(e)}
    json.dump(gaudits, open(GS_CACHE, "w", encoding="utf-8"), ensure_ascii=False, default=str)
elif gtodo:
    print(f"\n[!] {len(gtodo)} gold flag(s) unaudited - the post-audit rate needs OPENAI_API_KEY")

fully = all(f"{gid}|{ft}" in gaudits for gid, fl in flagged.items() for ft, _ in fl)
gs_conf = sum(1 for gid, fl in flagged.items()
              if any(normalize_verdict(gaudits.get(f"{gid}|{ft}", {})) == "TRUE_CONFLICT"
                     for ft, _ in fl))
print(f"\nrule stage      : {len(flagged)}/{len(GOLD)} = {len(flagged)/len(GOLD):.1%}")
if fully:
    print(f"after the audit : {gs_conf}/{len(GOLD)} = {gs_conf/len(GOLD):.1%}"
          f"   ({len(flagged)-gs_conf} cleared)")
    for gid, fl in flagged.items():
        for ft, _ in fl:
            a = gaudits.get(f"{gid}|{ft}", {})
            print(f"  {gid}|{ft}: {normalize_verdict(a)} - {str(a.get('reasoning') or a.get('audit_log') or '-')[:120]}")
MANIFEST["gold_false_alarm"] = {"n_gold": len(GOLD), "rule_stage": len(flagged),
                                "post_audit": gs_conf if fully else None,
                                "fully_audited": fully,
                                "flagged_ids": sorted(flagged)}

rule stage: 3/60 = 5.0% false-alarm rate
  GS-005 | pH = 6.4~8.0  | ['T1'] | condition: end of experiment, substrate dependent
  GS-009 | growth_medium_temp = 13.5~28.1 °C | ['T1'] | condition: year-round cultivation, seasonal variation
  GS-057 | water_temp = 1.5~19.5 °C | ['T1'] | condition: Kochi Prefecture, seasonal

rule stage      : 3/60 = 5.0%
after the audit : 2/60 = 3.3%   (1 cleared)
  GS-005|T1: TRUE_CONFLICT - The detected conflict is a numeric range conflict (T1) because the upper limit of the pH range (8.0) is outside the esta
  GS-009|T1: TRUE_CONFLICT - The detected conflict is a numeric range conflict (T1) because the upper limit of the provided range (28.1°C) exceeds th
  GS-057|T1: FALSE_ALARM - The detected conflict is based on a general range for wasabi cultivation water temperature, which is typically between 5


In [29]:
# ====== cell 28 ======
# ------------------------------- recall (§4.2.6) -------------------------------
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def surface(t):
    return f"{t.get('subject','')} {t.get('predicate','')} {t.get('object','')}".lower()

corpus_str, gold_str = [surface(t) for t in TRIPLETS], [surface(g) for g in GOLD]
vec  = TfidfVectorizer().fit(corpus_str + gold_str)
best = cosine_similarity(vec.transform(gold_str), vec.transform(corpus_str)).max(axis=1)

print(f"{'tau':>6}{'recall':>10}{'matched':>10}")
rec = {}
for tau in (0.40, 0.50, 0.60, 0.70, 0.80, 0.90):
    hit = int((best >= tau).sum()); rec[tau] = hit / len(GOLD)
    print(f"{tau:>6.2f}{rec[tau]:>10.1%}{hit:>10}"
          + ("   <- reported threshold" if abs(tau - FUZZY_TAU) < 1e-9 else ""))

miss = [GOLD[i]["id"] for i, b in enumerate(best) if b < FUZZY_TAU]
print(f"\nnot recovered at tau={FUZZY_TAU}: {len(miss)} -> {miss}")
for i in np.argsort(best)[:5]:
    print(f"   {GOLD[i]['id']} sim={best[i]:.2f} | {GOLD[i]['subject']} = {GOLD[i]['object']}")

cset  = {(t.get("subject"), t.get("predicate"), str(t.get("object"))) for t in TRIPLETS}
exact = sum(1 for g in GOLD if (g.get("subject"), g.get("predicate"), str(g.get("object"))) in cset)
print(f"\nexact-string recall: {exact}/{len(GOLD)} = {exact/len(GOLD):.1%}")
print("The gap between exact and similarity-based recall is the normalisation applied when "
      "the Gold Standard was curated, not missing extraction.")
MANIFEST["recall"] = {"by_tau": {str(k): round(v, 4) for k, v in rec.items()},
                      "tau": FUZZY_TAU, "exact": round(exact / len(GOLD), 4),
                      "missed_ids": miss}

   tau    recall   matched
  0.40     96.7%        58
  0.50     90.0%        54   <- reported threshold
  0.60     85.0%        51
  0.70     63.3%        38
  0.80     46.7%        28
  0.90     38.3%        23

not recovered at tau=0.5: 6 -> ['GS-004', 'GS-005', 'GS-017', 'GS-018', 'GS-048', 'GS-049']
   GS-048 sim=0.36 | soil_pH = 5.9~7.0
   GS-049 sim=0.38 | altitude = 1400~2300
   GS-017 sim=0.43 | DO = 7.2~13.5
   GS-018 sim=0.45 | DO = 7.2
   GS-004 sim=0.47 | pH = 5.7

exact-string recall: 20/60 = 33.3%
The gap between exact and similarity-based recall is the normalisation applied when the Gold Standard was curated, not missing extraction.


`cell 29`

## Part 8 — Knowledge graph and causal traversal  *(§3.3, §3.4)*

The graph carries two kinds of edge. **Relational** edges link a parameter to a reported
value and are labelled with the Confidence Level assigned in Part 4. **Causal** edges link
one parameter to another and come from the curated `causal_relations.json`.

The distinction matters for what can be retrieved. Relational edges alone form a star
around each parameter: every path is one hop, and no chain connects dissolved oxygen to
rhizome yield. Multi-hop traversal — and therefore the mechanism-level answers evaluated
in Part 9 — exists only because of the causal edges.

In [31]:
# ====== cell 30 ======
# pipeline.KnowledgeGraph, imported unmodified. Confidence Levels come from Definition 1
# (Part 4); the graph derives its edge label from the final_conflict field, so the level
# assigned there is passed through rather than recomputed.
cr = json.load(open(FILES["causal"], encoding="utf-8")) if FILES["causal"].exists() else {}
RELATIONS = cr.get("relations", cr) if isinstance(cr, dict) else cr
print(f"curated causal relations: {len(RELATIONS)}")

def audit_stub(row):
    """Carry the Part 4 Confidence Level into the graph without re-deriving it."""
    conf = row["confirmed_types"]
    fc = None
    if any(f in ("T1", "T2") for f in conf):
        fc = next(f for f in conf if f in ("T1", "T2"))
    elif any(str(f).startswith("T3") for f in conf):
        fc = "T3"
    return {"final_conflict": fc, "audit_log": "",
            "confidence": 0.9 if row["confidence_level"] == "Verified" else 0.5}

kg = KnowledgeGraph()
for t, r in zip(TRIPLETS, ROWS):
    kg.add_triplet(t, audit_stub(r))
for rel in RELATIONS:
    try:
        kg.add_causal_edge(rel)
    except Exception as e:
        print("  [warn] causal edge skipped:", rel.get("id"), str(e)[:80])

st = kg.get_stats()
print("\nKnowledge graph")
print("=" * 48)
for k, v in st.items():
    print(f"  {k:<26}: {v}")

# Reachability. pipeline.traverse_causal_chain is a breadth-first walk with a global
# visited set, so its "chains" are not simple paths and its hop counts are inflated;
# simple paths are enumerated here instead.
_adj = defaultdict(list)
for e in kg.edges:
    if e.get("edge_type") == "causal":
        _adj[e["source"]].append(e["target"])

def _simple_depths(limit=200000):
    d = Counter(); n = 0
    for start in list(_adj):
        stack = [(start, 0, {start})]
        while stack and n < limit:
            node, dep, seen = stack.pop()
            for m in _adj.get(node, ()):
                if m in seen:
                    continue
                d[dep + 1] += 1; n += 1
                stack.append((m, dep + 1, seen | {m}))
    return d

depths = _simple_depths()
print(f"\n  simple causal paths by hop count: {dict(sorted(depths.items()))}")
print(f"  longest simple causal chain     : {max(depths) if depths else 0} hops")

# entity fragmentation: near-synonymous cause/effect names break chains
ents = {e["source"] for e in kg.edges if e.get("edge_type") == "causal"} | \
       {e["target"] for e in kg.edges if e.get("edge_type") == "causal"}
outdeg = Counter(e["source"] for e in kg.edges if e.get("edge_type") == "causal")
print(f"\n  causal entities                 : {len(ents)}")
print(f"  appearing once or not at all as a cause: {sum(1 for e in ents if outdeg[e] <= 1)}")
print("  Near-synonymous names (DO_low, low_DO, DO_decrease, dissolved_oxygen_decrease)")
print("  are separate nodes, so chains break at those boundaries. Entity normalisation")
print("  would lengthen them; it is NOT applied here, and the hop counts above are the")
print("  figures the manuscript should quote.")
print("  Relational edges alone form a star around each parameter, so without the causal "
      "edges no path exceeds one hop and B3/B5 collapse onto B2.")
MANIFEST["kg"] = {k: (v if isinstance(v, (int, float, str)) else str(v)) for k, v in st.items()}
MANIFEST["kg"]["simple_causal_paths_by_depth"] = dict(sorted(depths.items()))
MANIFEST["kg"]["causal_entities"] = len(ents)
# get_stats() names its keys however pipeline.py names them; Part 14 quotes "nodes" and
# "edges", so the two headline counts are recorded under fixed names here as well.
MANIFEST["kg"]["nodes"] = len(kg.nodes)
MANIFEST["kg"]["edges"] = len(kg.edges)
MANIFEST["kg"]["longest_simple_causal_chain_hops"] = int(max(depths)) if depths else 0


curated causal relations: 374

Knowledge graph
  total_nodes               : 1017
  total_edges               : 1144
  uncertain_edges           : 36
  conflict_breakdown        : {'T1': 24, 'T2': 12}
  confidence_level_breakdown: {'Verified': 1108, 'Conflicted': 36}
  edge_type_breakdown       : {'relational': 770, 'causal': 374}
  avg_confidence            : 0.8383741258741259

  simple causal paths by hop count: {1: 374, 2: 75, 3: 18, 4: 3}
  longest simple causal chain     : 4 hops

  causal entities                 : 510
  appearing once or not at all as a cause: 451
  Near-synonymous names (DO_low, low_DO, DO_decrease, dissolved_oxygen_decrease)
  are separate nodes, so chains break at those boundaries. Entity normalisation
  would lengthen them; it is NOT applied here, and the hop counts above are the
  figures the manuscript should quote.
  Relational edges alone form a star around each parameter, so without the causal edges no path exceeds one hop and B3/B5 collapse onto B2.


In [32]:
# ====== cell 31 ======
# ------------------- §3.4 retrieval: node matching and subgraph -------------------
# Two corrections over the previous version, both required for the Confidence Level to
# reach an answer at all:
#
#   1. Parameter nodes are named by their corpus subject ("DO", "EC", "pH"), which a
#      keyword matcher over 3+ letter tokens can never hit from a query that says
#      "dissolved oxygen". A synonym table maps query language onto those node names.
#   2. Multi-hop retrieval traversed causal edges only. Every audited Confidence Level
#      lives on a RELATIONAL edge (parameter -> reported value); causal edges carry a
#      flat "Verified" assigned at construction, not an audit result. So the auditing
#      layer's output could not appear in B3/B5 evidence under any query. The subgraph
#      now includes the relational edges of the parameters the causal chains pass
#      through, which is what §3.3 annotates them for.
RANK = {"Verified": 0, "Disputed": 1, "Conflicted": 2}

PARAM_SYNONYMS = {
    "DO":                 ["dissolved oxygen", "do ", " do", "oxygen"],
    "EC":                 ["ec ", " ec", "electrical conductivity", "nutrient solution concentration",
                           "nutrient concentration"],
    "pH":                 ["ph ", " ph", "acidity"],
    "PPFD":               ["ppfd", "photosynthetic photon flux", "light intensity", "light level"],
    "PPF":                ["ppf"],
    "water_temp":         ["water temperature", "root-zone temperature", "root zone temperature",
                           "solution temperature", "water temp"],
    "air_temp":           ["air temperature", "ambient temperature", "air temp"],
    "growth_medium_temp": ["growth medium", "substrate temperature", "medium temperature"],
    "photoperiod":        ["photoperiod", "day length", "daylength", "light period"],
    "VPD":                ["vpd", "vapour pressure deficit", "vapor pressure deficit"],
    "AITC":               ["aitc", "allyl isothiocyanate", "isothiocyanate"],
    "shading_rate":       ["shading", "shade"],
    "rhizome_weight":     ["rhizome weight", "rhizome size"],
    "root_nitrogen":      ["root nitrogen"],
    "glucosinolates":     ["glucosinolate"],
    "altitude":           ["altitude", "elevation"],
    "CO2":                ["co2", "carbon dioxide"],
    "soil_moisture":      ["soil moisture"],
    "nutrient_concentration": ["nutrient concentration", "nutrient solution concentration"],
}

CAUSAL = defaultdict(list)
RELATIONAL = defaultdict(list)
for e in kg.edges:
    (CAUSAL if e.get("edge_type") == "causal" else RELATIONAL)[e["source"]].append(e)

STOP = {"what", "the", "for", "and", "how", "does", "are", "is", "in", "of", "to", "a",
        "should", "be", "it", "affect", "effect", "wasabi", "cultivation", "hydroponic"}

def match_nodes(query, k=3, k_param=3):
    """Return start nodes: parameter nodes named by the query, plus entity nodes whose
    label overlaps it. Parameters are matched through PARAM_SYNONYMS so that an
    abbreviation-named node is reachable from ordinary query language."""
    q = " " + query.lower() + " "
    params = [p for p, syns in PARAM_SYNONYMS.items()
              if p in kg.nodes and any(s in q for s in syns)]

    kws = [w.lower() for w in re.findall(r"[A-Za-z]{3,}", query) if w.lower() not in STOP]
    scored = []
    for nid, nd in kg.nodes.items():
        if nd.get("type") == "value" or nid in params:
            continue
        blob = f"{nid} {nd.get('label','')}".lower().replace("_", " ")
        s = sum(1 for w in kws if w in blob)
        if s:
            scored.append((s, len(nid), nid))
    scored.sort(key=lambda x: (-x[0], x[1]))
    return params[:k_param], [n for _, _, n in scored[:k]]

def path_confidence(edges):
    """A path is only as trustworthy as its weakest edge (§3.4)."""
    worst = max((RANK.get(e.get("confidence_level", "Verified"), 0) for e in edges), default=0)
    return ["Verified", "Disputed", "Conflicted"][worst]

def causal_paths(start, max_depth=4, limit=3000):
    out, stack = [], [(start, [], {start})]
    while stack and len(out) < limit:
        node, path, seen = stack.pop()
        for e in CAUSAL.get(node, []):
            if e["target"] in seen:
                continue
            np_ = path + [e]
            out.append(np_)
            if len(np_) < max_depth:
                stack.append((e["target"], np_, seen | {e["target"]}))
    return out

def _fmt_relational(e, lab, with_conf):
    nd = kg.nodes.get(e["target"], {})
    s = f"- {lab(e['source'])} --{e['predicate']}--> {nd.get('label','')} {nd.get('unit','')}".rstrip()
    if nd.get("condition"):
        s += f" (condition: {nd['condition']})"
    s += f"  <{e.get('source_id')}>"
    return s + (f"  [{e.get('confidence_level','Verified')}]" if with_conf else "")

def retrieve(query, multihop, with_confidence, limit=25, rel_limit=14):
    params, entities = match_nodes(query)
    lab = lambda n: kg.nodes.get(n, {}).get("label", n)
    lines, seen, levels = [], set(), []

    if not multihop:                                   # B2: one hop, relational
        for s0 in params + entities:
            for tgt, e in kg.get_neighbors(s0):
                key = (e["source"], e["target"])
                if key in seen:
                    continue
                seen.add(key)
                lvl = e.get("confidence_level", "Verified")
                levels.append(lvl)
                txt = (_fmt_relational(e, lab, with_confidence)
                       if e.get("edge_type", "relational") == "relational"
                       else f"- {lab(e['source'])} --{e['predicate']}--> {lab(e['target'])}"
                            f"  <{e.get('source_id')}>"
                            + (f"  [{lvl}]" if with_confidence else ""))
                lines.append((RANK.get(lvl, 0), 0, txt))
    else:                                              # B3 / B5: chains + their parameters
        touched = set(params)
        for s0 in params + entities:
            for path in causal_paths(s0):
                key = tuple((e["source"], e["target"]) for e in path)
                if key in seen:
                    continue
                seen.add(key)
                lvl = path_confidence(path)
                levels.append(lvl)
                chain = lab(path[0]["source"])
                for e in path:
                    chain += f" --{e['predicate']}--> {lab(e['target'])}"
                    touched.add(e["source"]); touched.add(e["target"])
                srcs = ",".join(sorted({e.get("source_id") or "" for e in path if e.get("source_id")}))
                lines.append((RANK.get(lvl, 0), -len(path),
                              f"- {chain}  <{srcs}>" + (f"  [{lvl}]" if with_confidence else "")))
        # the reported values for every parameter the chains pass through
        # sorted(), not set order. Python randomises string hashing per process, so
        # iterating `touched` directly gives a different candidate order in every kernel -
        # and rel[:rel_limit] then keeps a DIFFERENT 14 items. Measured: three hash seeds
        # give three different orderings. That made the retrieved evidence, and therefore
        # every answer built on it, irreproducible across restarts, which is not a defect a
        # notebook whose subject is reliability can carry.
        rel = []
        for n in sorted(touched):
            for e in RELATIONAL.get(n, []):
                key = (e["source"], e["target"])
                if key in seen:
                    continue
                seen.add(key)
                lvl = e.get("confidence_level", "Verified")
                rel.append((RANK.get(lvl, 0), 0, _fmt_relational(e, lab, with_confidence), lvl))
        # Ties broken on the rendered text so the order is total, not merely stable over
        # whatever order the loop above happened to produce.
        rel.sort(key=lambda x: (-x[0], x[2]))         # surface the graded ones first
        for r_ in rel[:rel_limit]:
            lines.append(r_[:3]); levels.append(r_[3])

    # Verified items are listed first, but a graded item must never be the one the limit
    # drops: the note below would then tell the generator that the subgraph contains a
    # disagreement while showing none of it, which is the opposite of what §3.4 claims the
    # confidence layer does. The budget is spent on the graded items first and the
    # remainder filled with Verified ones, leaving the presentation order unchanged.
    lines.sort(key=lambda x: (x[0], x[1]))
    _graded = [l for l in lines if l[0] > 0][:limit]
    _plain  = [l for l in lines if l[0] == 0][:max(0, limit - len(_graded))]
    body = "\n".join(t for _, _, t in _plain + _graded)
    if with_confidence:
        n_c = sum(1 for l in levels if l == "Conflicted")
        n_d = sum(1 for l in levels if l == "Disputed")
        if n_c or n_d:
            body += (f"\n\nNote: this subgraph includes {n_c} Conflicted and {n_d} Disputed "
                     "item(s) - the literature does not agree on them. Say so explicitly and "
                     "do not present those values as settled.")
    return (body or "(no matching evidence in the knowledge graph)"), levels

# A common budget and a common grounding rule, applied identically to every condition.
# Two confounds are removed by this. First, length: in the recorded run answer length
# correlated with the judge score at r = 0.66, and B5's answers were 1384 characters
# against B1's 215, so part of B5's margin was volume. Second, ungrounded prior knowledge:
# on queries where the graph held nothing, B5 still scored highly by supplying generic
# hydroponic advice (7 of 20 answers, against 0 of 20 for B2). Neither is a property of
# the Self-Auditing layer. The cap is 150 words rather than something tighter because
# disclosing a disagreement costs words that a silent answer does not spend; too tight a
# cap would penalise B5 for doing the thing under test.
COMMON_RULE = (" Answer in at most 150 words. Use ONLY the retrieved evidence above: if it "
               "does not answer the question, say so plainly and stop. Do not supply "
               "general best-practice advice from prior knowledge, and do not pad.")

BASELINES = {
 "B1": dict(kind="chunk", conf=False,
            sys="Answer the agronomy question using ONLY the retrieved source passages."
                + COMMON_RULE),
 "B2": dict(kind="kg1", conf=False,
            sys="Answer using the retrieved knowledge-graph facts." + COMMON_RULE),
 "B3": dict(kind="kgN", conf=False,
            sys="Answer using the retrieved knowledge-graph subgraph. Trace the multi-step "
                "relations to explain the mechanism, not just the endpoint values."
                + COMMON_RULE),
 "B5": dict(kind="kgN", conf=True,
            sys="Answer using the retrieved knowledge-graph subgraph. Trace the multi-step "
                "relations to explain the mechanism. Each item carries a Confidence Level "
                "(Verified / Disputed / Conflicted) assigned by the Self-Auditing pipeline. "
                "Prefer Verified evidence, and state explicitly where the literature "
                "disagrees rather than presenting a disputed value as settled."
                + COMMON_RULE),
}

# B3 and B5 now differ by exactly one sentence - the Confidence Level instruction - and
# nothing else. That is what makes B5 - B3 an ablation rather than a comparison.
# The version covers everything that determines an answer, retrieval included: the
# evidence changed when graded items stopped being truncated, so answers written under v1
# are no longer answers to the same question.
# Bumped from v2: sorted(touched) changes which relational items survive the rel_limit
# truncation, so the evidence string differs from the one the v2 answers were written
# against. Reusing those answers would attribute them to evidence they never saw.
PROMPT_VERSION = "graded-evidence-kept-v3-deterministic"

# ---- does the auditing layer now reach the evidence? (deterministic, no API calls) ----
# The evaluation queries live with the recorded answers (Part 9); load them here so this
# check can run as soon as the graph exists.
if "QUERIES" not in globals():
    _ab = json.load(open(FILES["answers"], encoding="utf-8")) if FILES["answers"] else {}
    QUERIES = _ab.get("queries", [])
    print(f"[queries] {len(QUERIES)} evaluation queries loaded from {Path(FILES['answers']).name}"
          if QUERIES else "[queries] none available - coverage check skipped")

print("\nConfidence Levels reaching the retrieved evidence, over all evaluation queries")
print(f"{'':<4}{'queries with a non-Verified item':>34}{'Conflicted':>12}{'Disputed':>10}")
cover = {}
for b, cfg in (BASELINES.items() if QUERIES else []):
    if cfg["kind"] == "chunk":
        continue
    nq = c_tot = d_tot = 0
    for q in QUERIES:
        _, lv = retrieve(q, multihop=(cfg["kind"] == "kgN"), with_confidence=True)
        c = sum(1 for x in lv if x == "Conflicted"); d = sum(1 for x in lv if x == "Disputed")
        nq += 1 if (c or d) else 0; c_tot += c; d_tot += d
    cover[b] = {"queries_with_graded": nq, "conflicted_items": c_tot, "disputed_items": d_tot}
    print(f"{b:<4}{f'{nq}/{len(QUERIES)}':>34}{c_tot:>12}{d_tot:>10}")
if not QUERIES:
    print("  (skipped - no query set)")
MANIFEST["confidence_reach"] = cover

q0 = "How does low dissolved oxygen (DO) affect wasabi rhizome yield?"
print(f"\nretrieval check - {q0}")
p_, e_ = match_nodes(q0)
print(f"  matched parameter nodes: {p_}   entity nodes: {e_[:3]}")
for b, cfg in BASELINES.items():
    if cfg["kind"] == "chunk":
        continue
    body, lv = retrieve(q0, multihop=(cfg["kind"] == "kgN"), with_confidence=cfg["conf"])
    n_items = sum(1 for l in body.splitlines() if l.startswith("- "))
    print(f"  {b}: {n_items:>3} items | {dict(Counter(lv))} | {len(body):>5} chars")
print("\n--- B5 evidence (first lines) ---")
print(retrieve(q0, True, True)[0][:700])


[queries] 20 evaluation queries loaded from judge_inputs.json

Confidence Levels reaching the retrieved evidence, over all evaluation queries
      queries with a non-Verified item  Conflicted  Disputed
B2                               14/20          68         0
B3                               14/20          72         0
B5                               14/20          72         0

retrieval check - How does low dissolved oxygen (DO) affect wasabi rhizome yield?
  matched parameter nodes: ['DO']   entity nodes: ['DO_low', 'low_DO', 'DO_high']
  B2:  25 items | {'Verified': 35, 'Conflicted': 4} |  1619 chars
  B3:  25 items | {'Verified': 34, 'Conflicted': 4} |  1710 chars
  B5:  25 items | {'Verified': 34, 'Conflicted': 4} |  2187 chars

--- B5 evidence (first lines) ---
- Low dissolved oxygen concentration --causes--> Decrease in root water and nutrient uptake --depresses--> Growth of wasabi plantlets  <S01>  [Verified]
- Low dissolved oxygen --causes--> Decrease in root water and n

In [33]:
# ====== cell 32 ======
# ------------------- baseline definitions, as implemented (§4.1.2) -------------------
# This block is the single source of truth for what each condition does. Table 6 of the
# manuscript is to be rewritten from it (see the resolution recorded in Part 12).
#
# The keys B1..B5 are fixed: the recorded judgments in Part 9 are indexed by them. Only the
# display labels change. B3 is called "Multi-hop KG-RAG without auditing" rather than
# "GraphRAG w/o Audit" because no baseline implements GraphRAG's community-summary
# indexing; naming it GraphRAG would claim a component that is not present.
BASELINE_LABELS = {
    "B1": "Naive RAG",
    "B2": "Single-hop KG-RAG",
    "B3": "Multi-hop KG-RAG without auditing",
    "B5": "Multi-hop KG-RAG with Self-Auditing (proposed)",
}
BASELINE_SPEC = {
 "B1": {"retrieval": "TF-IDF over raw PDF text chunks, top 5",
        "graph": False, "multi_hop": False, "community_summaries": False,
        "confidence_shown": False},
 "B2": {"retrieval": "knowledge graph, one-hop neighbours of the matched nodes",
        "graph": True,  "multi_hop": False, "community_summaries": False,
        "confidence_shown": False},
 "B3": {"retrieval": "knowledge graph, multi-hop simple paths over causal edges",
        "graph": True,  "multi_hop": True,  "community_summaries": False,
        "confidence_shown": False},
 "B5": {"retrieval": "identical to B3",
        "graph": True,  "multi_hop": True,  "community_summaries": False,
        "confidence_shown": True},
}
print(f"{'':<4}{'label':<48}{'multi-hop':>10}{'confidence':>12}")
# Dict insertion order puts B5 before B4 (B4 is added in the next cell), so the
# display order is stated explicitly wherever a BASELINE_* dict is iterated.
for b, spec in sorted(BASELINE_SPEC.items()):
    print(f"{b:<4}{BASELINE_LABELS[b]:<48}{str(spec['multi_hop']):>10}"
          f"{str(spec['confidence_shown']):>12}")
    print(f"{'':<4}  retrieval: {spec['retrieval']}")
assert not any(s["community_summaries"] for s in BASELINE_SPEC.values()), \
    "community summarisation is not implemented anywhere; the spec must say so"
print("\nB2 -> B3 isolates retrieval depth (single-hop vs multi-hop).")
print("B3 -> B5 isolates the auditing layer: identical subgraph, Confidence Levels added.")
print("No condition implements GraphRAG's community-summary indexing, so none is named")
print("GraphRAG; Table 6 and §4.1.2 are to be rewritten from this table.")
MANIFEST["baseline_spec"] = BASELINE_SPEC
MANIFEST["baseline_labels"] = BASELINE_LABELS


    label                                            multi-hop  confidence
B1  Naive RAG                                            False       False
      retrieval: TF-IDF over raw PDF text chunks, top 5
B2  Single-hop KG-RAG                                    False       False
      retrieval: knowledge graph, one-hop neighbours of the matched nodes
B3  Multi-hop KG-RAG without auditing                     True       False
      retrieval: knowledge graph, multi-hop simple paths over causal edges
B5  Multi-hop KG-RAG with Self-Auditing (proposed)        True        True
      retrieval: identical to B3

B2 -> B3 isolates retrieval depth (single-hop vs multi-hop).
B3 -> B5 isolates the auditing layer: identical subgraph, Confidence Levels added.
No condition implements GraphRAG's community-summary indexing, so none is named
GraphRAG; Table 6 and §4.1.2 are to be rewritten from this table.


In [34]:
# ====== cell 33 ======
# ---- B4: the prompting control (§4.1.2) -------------------------------------------
# The question B1-B5 cannot answer. Every one of them is THIS system with a piece removed,
# so none of them is a control for the instruction itself. A reviewer will ask the
# obvious thing: if you simply
# TELL the generator to look for disagreement and report it, with no audited knowledge
# base at all, do you get the same behaviour?
#
# B4 retrieves EXACTLY what B3 and B5 retrieve and is shown NO Confidence Levels. It gets
# an instruction instead. That makes two clean contrasts at identical retrieval:
#     B5 - B3 : the effect of the auditing layer      (levels shown vs not shown)
#     B5 - B4 : the effect of AUDITING over PROMPTING (levels vs an instruction)
#
# The instruction below is deliberately as strong as B5's - it names the behaviour, asks
# for the divergent values, and forbids presenting them as settled. A weak B4 would make
# B5 look good for the wrong reason, and a referee would say so.
BASELINE_LABELS["B4"] = "Multi-hop KG-RAG + self-assessment prompt (control)"
BASELINE_SPEC["B4"] = {"retrieval": "identical to B3 and B5",
                       "graph": True, "multi_hop": True, "community_summaries": False,
                       "confidence_shown": False}
BASELINES["B4"] = dict(
    kind="kgN", conf=False,
    sys="Answer using the retrieved knowledge-graph subgraph. Trace the multi-step "
        "relations to explain the mechanism. Before answering, examine the evidence for "
        "disagreement: if two or more items report different values for the same "
        "quantity, treat that quantity as unsettled. State explicitly where the "
        "literature disagrees, give the divergent values, and do not present any of them "
        "as settled. Where the evidence agrees, answer normally."
        + COMMON_RULE)

# B4 is deliberately NOT added to BASES (Part 9). Table 11, Krippendorff's alpha, the
# agreement statistics and Figures F8/F9 are indexed by B1-B5 and must stay comparable
# with the recorded study, which has no B4. Part 9c's completeness check would also fail,
# since no B4 judgment exists for the recorded run. B4 is analysed on its own in Part 9e.
print("B4 added as a control condition; BASES is untouched.")
print(f"  label       : {BASELINE_LABELS['B4']}")
print(f"  retrieval   : {BASELINE_SPEC['B4']['retrieval']}")
print(f"  levels shown: {BASELINE_SPEC['B4']['confidence_shown']}  (same as B3)")
_d34 = BASELINES["B5"]["sys"] != BASELINES["B3"]["sys"]
_d35 = BASELINES["B4"]["sys"] != BASELINES["B3"]["sys"]
assert BASELINES["B4"]["kind"] == BASELINES["B3"]["kind"] == BASELINES["B5"]["kind"], \
    "B3, B5 and B4 must share one retrieval mode or the contrast is confounded"
assert BASELINES["B4"]["conf"] is False, "B4 must not be shown Confidence Levels"
assert _d34 and _d35, "B4 and B5 must each differ from B3 by their instruction only"
print("  checked     : B3, B5 and B4 share one retrieval mode; only the prompt differs.")


B4 added as a control condition; BASES is untouched.
  label       : Multi-hop KG-RAG + self-assessment prompt (control)
  retrieval   : identical to B3 and B5
  levels shown: False  (same as B3)
  checked     : B3, B5 and B4 share one retrieval mode; only the prompt differs.


In [89]:
# ---------- Part 8 (extra) - verify the paper's central control ----------
# The manuscript claims B3, B4 and B5 are evaluated on the same retrieval. Everything in
# Section 4.4-4.5 rests on that claim, so it should be verified rather than assumed.
#
# B3 and B4 share a retrieval configuration exactly (kind kgN, conf False), so the
# interesting comparison is B3 vs B5: B5 renders a Confidence Level on each item, which
# also enters the tie-break used to truncate the relational items. If that tie-break ever
# reorders the candidates differently, the two conditions would not see the same items.
#
# No API calls. Strips the level annotation, then compares (a) the item SEQUENCE and
# (b) the item SET, per query.
import re as _re
_LVL = _re.compile(r"\s*\[(?:Verified|Disputed|Conflicted)\]\s*$")
 
def _lines(q, cfg):
    body, _ = retrieve(q, multihop=(cfg["kind"] == "kgN"), with_confidence=cfg["conf"])
    return [_LVL.sub("", l).rstrip() for l in body.splitlines() if l.startswith("- ")]
 
_seq_ok = _set_ok = 0
_bad = []
print(f"{'q':>3}  {'B3 n':>5}{'B5 n':>6}  {'same sequence':>15}{'same set':>10}")
for _i, _q in enumerate(QUERIES):
    a = _lines(_q, BASELINES["B3"])
    c = _lines(_q, BASELINES["B5"])
    s_eq, t_eq = (a == c), (set(a) == set(c))
    _seq_ok += s_eq; _set_ok += t_eq
    if not t_eq:
        _bad.append(_i + 1)
    print(f"{_i+1:>3}  {len(a):>5}{len(c):>6}  {('yes' if s_eq else 'NO'):>15}"
          f"{('yes' if t_eq else 'NO'):>10}")
 
print(f"\nidentical item SET      : {_set_ok}/{len(QUERIES)} queries")
print(f"identical item SEQUENCE : {_seq_ok}/{len(QUERIES)} queries")
if _bad:
    print(f"queries where the item set differs: {_bad}")
    _q0 = QUERIES[_bad[0] - 1]
    _a, _c = set(_lines(_q0, BASELINES["B3"])), set(_lines(_q0, BASELINES["B5"]))
    print(f"\nfirst mismatch, q{_bad[0]}:")
    for _x in sorted(_a - _c)[:4]: print("   only B3 :", _x[:110])
    for _x in sorted(_c - _a)[:4]: print("   only B5 :", _x[:110])
else:
    print("VERIFIED: B3 and B5 are shown the same evidence items on every query;")
    print("they differ only in whether each item carries its Confidence Level.")
 
# B4 shares B3's configuration exactly - assert rather than measure.
assert (BASELINES["B4"]["kind"] == BASELINES["B3"]["kind"]
        and BASELINES["B4"]["conf"] == BASELINES["B3"]["conf"]), \
    "B4 must share B3's retrieval configuration"
print("B4 shares B3's retrieval configuration (kind and confidence flag) by construction.")
 
MANIFEST["retrieval_control_check"] = {
    "queries": len(QUERIES), "identical_item_set": _set_ok,
    "identical_item_sequence": _seq_ok, "mismatched_queries": _bad,
    "note": ("B3 vs B5 after stripping the Confidence Level annotation; B4 shares B3's "
             "retrieval configuration by construction.")}

  q   B3 n  B5 n    same sequence  same set
  1     25    25              yes       yes
  2      9     9              yes       yes
  3     16    16              yes       yes
  4     25    25              yes       yes
  5     21    21              yes       yes
  6     25    25              yes       yes
  7     20    20              yes       yes
  8     22    22              yes       yes
  9     19    19              yes       yes
 10     21    21              yes       yes
 11     23    23              yes       yes
 12     25    25              yes       yes
 13     25    25              yes       yes
 14     10    10              yes       yes
 15     22    22              yes       yes
 16     11    11              yes       yes
 17     25    25              yes       yes
 18      7     7              yes       yes
 19     25    25              yes       yes
 20      5     5              yes       yes

identical item SET      : 20/20 queries
identical item SEQUENCE : 20/20 que

`cell 34`

## Part 9 — Evaluation setup  *(§4.3.1)*

**Five** retrieval conditions over 20 queries, graded 1–5 on three criteria — factual
accuracy, causal validity, actionability — by three judges from different model families.

| | Retrieval | Told to look for disagreement | Confidence Levels shown |
|---|---|---|---|
| **B1** Naive RAG | text chunks from the source PDFs | no | no |
| **B2** Single-hop KG-RAG | knowledge graph, one hop | no | no |
| **B3** Multi-hop KG-RAG without auditing | multi-hop causal traversal | no | no |
| **B4** Multi-hop + self-assessment prompt *(control)* | the same multi-hop traversal | **yes** | no |
| **B5** Multi-hop + Self-Auditing *(proposed)* | the same multi-hop traversal | no | **yes** |

B3, B4 and B5 issue an identical retrieval and differ only in what the generator is told,
which is what makes the two contrasts clean: **B5 − B3** is the auditing layer against
nothing, **B5 − B4** is auditing against prompting.

No condition implements GraphRAG's community-summary indexing, so none is named GraphRAG;
`BASELINE_SPEC` in Part 8 is the definition Table 6 of the manuscript is rewritten from.

**Two evaluations exist, and the notebook reports one of them.** The *recorded study* —
`judge_inputs.json` and `RQ2_judge_scores_canonical.json` — was produced before the §3.4
retrieval defect was found and before a common answer budget was imposed, so its B3/B5
comparison is not interpretable. The *regenerated* evaluation in Part 9b runs the same
design on the corrected retrieval. `REPORT` in Part 0 chooses; it is set to `regenerated`,
and the recorded study is kept as the reference the comparison in Part 9b is made against.

This Part defines only the inputs, the reliability functions and the judging rubric.
Nothing here reads a score, so Part 9b can run before any result is computed and one
top-to-bottom execution describes a single evaluation throughout.

In [36]:
# ====== cell 35 ======
# ------------------- the recorded evaluation, and the names it defines -------------------
# The recorded study is an INPUT of this notebook: 80 answers (4 conditions x 20 queries) graded by 3 judges, produced
# before the §3.4 retrieval defect was found. It is always loaded, because Part 9b
# compares against it, and because QUERIES - the 20 evaluation queries - lives in the same
# file. Which evaluation the manuscript REPORTS is decided by REPORT in Part 0 and applied
# in "Part 9c - reported evaluation", after Part 9b has had a chance to produce the
# regenerated one. Nothing between here and there reads a reported score.
_rec_raw = json.load(open(FILES["judge"], encoding="utf-8")) if FILES["judge"] else []
ANSWER_BLOB   = json.load(open(FILES["answers"], encoding="utf-8")) if FILES["answers"] else {}
QUERIES          = ANSWER_BLOB.get("queries", [])
_rec_ans = ANSWER_BLOB.get("answers", {})

# The recorded study was numbered before the prompting control existed: its B4 IS the
# proposed system, which is B5 here, and nothing in it corresponds to today's B4. Rather
# than carry a translation table through every later cell - where one forgotten lookup
# would compare the old proposed system against the new control - the recorded rows are
# converted ONCE, here, and afterwards every part of the notebook speaks one numbering.
REC_RENUMBER = {"B4": "B5"}          # recorded label -> this notebook's label
RECORDED_ROWS = [dict(r, baseline=REC_RENUMBER.get(r["baseline"], r["baseline"]))
                 for r in _rec_raw]
RECORDED_ANSWERS = {REC_RENUMBER.get(b, b): v for b, v in _rec_ans.items()}
if _rec_raw:
    print(f"recorded study renumbered: {REC_RENUMBER} "
          "(its B4 was the proposed system; today's B4, the prompting control, has no "
          "recorded counterpart)")

# All five conditions are generated and judged in the main loop, so Table 11 carries
# the proposed system rather than leaving it to an appendix cell.
#   B1 Naive RAG | B2 single-hop KG-RAG | B3 multi-hop, no auditing
#   B4 multi-hop + self-assessment PROMPT (control) | B5 multi-hop + Self-Auditing
# The two contrasts that matter, both at identical retrieval:
#   B5 - B3  the auditing layer against nothing
#   B5 - B4  auditing against prompting
BASES = ["B1", "B2", "B3", "B4", "B5"]

# The recorded study predates this numbering: its B4 is the proposed system, which is
# B5 here, and it has no counterpart for today's B4 (the prompting control did not
# exist then). Any comparison against the recorded run must go through this map, or it
# silently compares the old proposed system against the new control.
# What each run actually contains, after the renumbering above. The recorded study has no
# prompting control, so demanding B4 of it would make REPORT = "recorded" impossible.
REC_BASES = ["B1", "B2", "B3", "B5"]

# Queries dropped from the WHOLE panel, 1-based. A hosted judge can refuse an item
# permanently - a safety classifier firing on benign agronomy text is the realistic case,
# and it is not retryable at temperature 0. When that happens the choice is between an
# unbalanced grid and a smaller one, and a smaller one is the only defensible option:
# excluding the QUERY keeps every condition scored by every judge on exactly the same
# items, which is the property every paired comparison in this notebook depends on.
# Excluding the CELL would leave one condition averaged over fewer items than the others.
#
# Leave empty unless the completeness check names a query. Any exclusion must be stated
# in 3.5 with the reason and the judge involved - it changes n, and n is reported.
EXCLUDE_QUERIES = [11]   # J2 (claude-sonnet-4-6) refused B5|11 on every attempt
                         # (stop_reason=refusal); the answer is ordinary text about
                         # nutrient concentration and rhizome enlargement.
mean3 = lambda r: (r["FA"] + r["CV"] + r["AC"]) / 3

# scipy's method="auto" switches between the exact null distribution and the normal
# approximation depending on the version and on whether ties are present, so the same data
# can yield 4.4e-05 or 9.5e-07 on two machines. The paired differences here DO contain
# ties, and the exact distribution assumes none, so the tie-corrected approximation is the
# valid choice. Pinned here, ahead of every cell that runs a signed-rank test.
WILCOXON_METHOD = "approx"

print(f"recorded study : {len(RECORDED_ROWS)} judgments, "
      f"judges {sorted({r['judge'] for r in RECORDED_ROWS})}")
print(f"queries        : {len(QUERIES)}")
print(f"answers        : {{{', '.join(f'{b}: {len(v)}' for b, v in RECORDED_ANSWERS.items())}}}")
if not QUERIES:
    print("[!] judge_inputs.json not found - the evaluation cannot run")


recorded study renumbered: {'B4': 'B5'} (its B4 was the proposed system; today's B4, the prompting control, has no recorded counterpart)
recorded study : 240 judgments, judges ['J1', 'J2', 'J3']
queries        : 20
answers        : {B1: 20, B2: 20, B3: 20, B5: 20}


In [37]:
# ====== cell 36 ======
# ------------------------------ Krippendorff's alpha ------------------------------
def krippendorff_alpha(units, metric="ordinal"):
    """units: one list of coder values per unit; None for a missing judgment."""
    vals = sorted({v for u in units for v in u if v is not None})
    idx = {v: i for i, v in enumerate(vals)}
    o = defaultdict(float)
    for u in units:
        u = [x for x in u if x is not None]
        if len(u) < 2:
            continue
        for i, a in enumerate(u):
            for j, b in enumerate(u):
                if i != j:
                    o[(a, b)] += 1.0 / (len(u) - 1)
    nc = {c: sum(o[(c, k)] for k in vals) for c in vals}
    n = sum(nc.values())
    if n < 2:
        return float("nan")
    if metric == "interval":
        d2 = lambda c, k: (c - k) ** 2
    else:                                   # ordinal metric (Krippendorff 2011)
        def d2(c, k):
            i, j = sorted((idx[c], idx[k]))
            s = sum(nc[vals[g]] for g in range(i, j + 1)) - (nc[vals[i]] + nc[vals[j]]) / 2.0
            return s * s
    Do = sum(o[(c, k)] * d2(c, k) for c in vals for k in vals)
    De = sum(nc[c] * (nc[k] - (1 if c == k else 0)) / (n - 1) * d2(c, k)
             for c in vals for k in vals)
    return float("nan") if De == 0 else 1.0 - Do / De

# self-test on a textbook-style perfect-agreement case
assert abs(krippendorff_alpha([[1, 1], [2, 2], [3, 3], [4, 4]]) - 1.0) < 1e-9
print("Krippendorff implementation: self-test passed")
# ---- agreement statistics: what the judges agree on, and what they do not ----
# Krippendorff's alpha is a SINGLE-RATER ABSOLUTE-agreement coefficient: a judge who is
# consistently one point stricter than the others contributes disagreement even when the
# two rank every answer identically. That is the correct definition for the question
# "would another judge have given this number", and it is the number to report first.
#
# It is not the only question the evaluation asks. Every comparison in this section is a
# paired within-query difference, which a judge's overall severity cancels out of, so the
# relevant reliability there is agreement on ORDERING. That is what the consistency form
# of the intraclass correlation measures, and it is reported alongside rather than instead.
#
# Rescaling the judges and recomputing alpha would raise the number, but the result would
# no longer be alpha and would not reproduce from the deposited scores, so it is not done.
from scipy.stats import spearmanr
from itertools import combinations

def agreement_stats(M):
    """M: n units x k raters, complete. Returns alpha's companions, not replacements."""
    M = np.asarray(M, float)
    n, k = M.shape
    gm = M.mean()
    MSR = k * ((M.mean(1) - gm) ** 2).sum() / (n - 1)          # between units
    MSC = n * ((M.mean(0) - gm) ** 2).sum() / (k - 1)          # between raters (severity)
    MSE = ((M - M.mean(1, keepdims=True) - M.mean(0, keepdims=True) + gm) ** 2).sum() \
          / ((n - 1) * (k - 1))
    out = {
        # single rater - the like-for-like comparison with Krippendorff's alpha
        "ICC_A1": (MSR - MSE) / (MSR + (k - 1) * MSE + k * (MSC - MSE) / n),
        "ICC_C1": (MSR - MSE) / (MSR + (k - 1) * MSE),
        # the mean of k raters - the quantity this section actually reports
        "ICC_Ak": (MSR - MSE) / (MSR + (MSC - MSE) / n),
        "ICC_Ck": (MSR - MSE) / MSR,
    }
    # Kendall's W over ranks within each rater, corrected for ties
    R = np.zeros_like(M); Tcorr = 0.0
    for c in range(k):
        col = M[:, c]
        order = np.argsort(col, kind="mergesort")
        ranks = np.empty(n, float); i = 0
        while i < n:
            j = i
            while j + 1 < n and col[order[j + 1]] == col[order[i]]:
                j += 1
            ranks[order[i:j + 1]] = (i + j) / 2.0 + 1
            t = j - i + 1
            if t > 1:
                Tcorr += t ** 3 - t
            i = j + 1
        R[:, c] = ranks
    Rs = R.sum(1)
    out["kendall_W"] = 12 * ((Rs - Rs.mean()) ** 2).sum() / (k ** 2 * (n ** 3 - n) - k * Tcorr)
    rho = [spearmanr(M[:, i], M[:, j]).statistic for i, j in combinations(range(k), 2)]
    out["spearman_mean"] = float(np.mean(rho))
    out["spearman_pairs"] = {f"{i+1}-{j+1}": float(r)
                             for (i, j), r in zip(combinations(range(k), 2), rho)}
    out["rater_means"] = [float(x) for x in M.mean(0)]
    out["rater_sds"] = [float(x) for x in M.std(0, ddof=1)]
    return out

def print_agreement(stats, label, judges):
    print(f"\nAgreement beyond absolute level - {label}")
    print(f"  rater means : " + "  ".join(f"{j} {m:.2f}" for j, m in zip(judges, stats["rater_means"])))
    print(f"  single rater: ICC(A,1)={stats['ICC_A1']:.3f} absolute   "
          f"ICC(C,1)={stats['ICC_C1']:.3f} consistency")
    print(f"  mean of {len(judges)}   : ICC(A,{len(judges)})={stats['ICC_Ak']:.3f} absolute   "
          f"ICC(C,{len(judges)})={stats['ICC_Ck']:.3f} consistency")
    print(f"  rank agreement: Kendall W={stats['kendall_W']:.3f}   "
          f"mean Spearman rho={stats['spearman_mean']:.3f}  {stats['spearman_pairs']}")
    print("  ICC(A,1) is the coefficient comparable with Krippendorff's alpha; the "
          "consistency\n  forms ignore a rater's overall severity, which the paired tests "
          "cancel out anyway.")

# self-test: three raters who rank identically but differ by a constant offset.
# Consistency must be 1; absolute agreement must not be.
_t = np.array([[1., 2., 3.], [2., 3., 4.], [3., 4., 5.], [4., 5., 6.], [5., 6., 7.]])
_s = agreement_stats(_t)
assert abs(_s["ICC_C1"] - 1.0) < 1e-9 and abs(_s["kendall_W"] - 1.0) < 1e-9
assert _s["ICC_A1"] < 0.9
print("agreement_stats: self-test passed (pure severity shift -> ICC(C,1)=1, ICC(A,1)<1)")

Krippendorff implementation: self-test passed
agreement_stats: self-test passed (pure severity shift -> ICC(C,1)=1, ICC(A,1)<1)


In [38]:
# ====== cell 37 ======
# ------------------- the judging rubric and its transports -------------------
# Defined once, here, because Part 9b calls them and Part 9b now runs before any results
# cell. RUBRIC is the FA/CV/AC prompt of the recorded study, unchanged, so the regenerated
# judgments are graded on exactly the criteria §3.5 defines. Part 9d adds a fourth
# criterion in a separate pass with its own prompt; this one is not touched.
RUBRIC = ("You are grading an agronomy answer. Score three criteria 1-5:\n"
          "FA factual accuracy, CV causal validity, AC actionability.\n"
          'Return strict JSON: {"FA":n,"CV":n,"AC":n}')

def parse_scores(raw):
    for pat in (r"\{[^{}]*\"FA\"[^{}]*\}", r"\{.*\}"):
        m = re.search(pat, raw, re.S)
        if m:
            try:
                d_ = json.loads(m.group())
                return {"FA": float(d_["FA"]), "CV": float(d_["CV"]), "AC": float(d_["AC"])}
            except Exception:
                pass
    raise RuntimeError(f"unparsable: {raw[:100]!r}")

def judge_call(kind, model, prompt):
    """One FA/CV/AC judgment. 'local' is handled by Part 9b's own wrapper."""
    if kind == "anthropic":
        import anthropic
        m = anthropic.Anthropic().messages.create(
            model=model, max_tokens=512, temperature=0, system=RUBRIC,
            messages=[{"role": "user", "content": prompt}])
        if m.stop_reason not in ("end_turn", "stop_sequence"):
            raise RuntimeError(f"incomplete (stop_reason={m.stop_reason})")
        raw = "".join(b.text for b in m.content if getattr(b, "type", "") == "text")
    else:
        if kind == "openai":
            from openai import OpenAI; cli = OpenAI()
        else:
            from groq import Groq;     cli = Groq()
        r = cli.chat.completions.create(
            model=model, temperature=0, max_tokens=512,
            messages=[{"role": "system", "content": RUBRIC},
                      {"role": "user", "content": prompt}])
        raw = (r.choices[0].message.content or "") if r.choices else ""
    if not raw.strip():
        raise RuntimeError("empty response")
    return parse_scores(raw)

print("judging rubric and transports ready")


judging rubric and transports ready


`cell 38`

## Part 9b — Regenerating the evaluation with the corrected retriever

The 80 answers in `judge_inputs.json` (four conditions, before the prompting control
existed) were produced **before** the §3.4 retrieval fix, when
no query reached a parameter node and therefore no answer could carry a Confidence Level.
B5 was, in substance, B3. Table 11 as recorded cannot support a claim about the auditing
layer's effect.

These two cells regenerate the evaluation end to end with the current retriever and judge
it again, writing to new files. `judge_inputs.json` and `RQ2_judge_scores_canonical.json`
are never modified: the recorded study stays available, and the two runs are printed side
by side so the manuscript can say which one it reports and why.

Set `REGEN_ANSWERS = True` once (100 generation calls), then `REGEN_JUDGE = True` once
(300 judging calls). Both are cached; every later execution is free.

Nothing is switched here. `REPORT` in Part 0 decides which run the notebook reports, and
Part 9c binds `JUDGE_ROWS` / `ANSWERS` / `CELL` accordingly for Part 9c, Part 9d, Part 10
and Part 12. This Part runs first precisely so that the choice is made once, after both
runs exist, rather than rebound midway through the results.

In [40]:
# ====== cell 39 ======
# ---------------- regenerate the answers with the corrected retriever ----------------
REGEN_ANSWERS = True           # True: len(BASES) x len(QUERIES) generation calls, cached
MODEL_GEN     = "gpt-4o"        # the generator; recorded separately in the manifest
ANS_CACHE     = OUT_DIR / "answers_regenerated.json"
CHUNK_CACHE   = OUT_DIR / "pdf_chunks.json"

# B1 is TF-IDF over raw source text, so it needs the PDFs rather than the triplets. The
# chunk index is built once from the deposited PDFs and cached, which keeps B1 exactly as
# BASELINE_SPEC describes it instead of quietly substituting a graph-derived baseline.
def build_chunk_index(pdf_dir):
    if CHUNK_CACHE.exists():
        return json.load(open(CHUNK_CACHE, encoding="utf-8"))
    chunks = []
    pdfs = sorted(Path(pdf_dir).glob("S*.pdf"),
                  key=lambda p: int(re.match(r"^S(\d+)", p.name).group(1)))
    for p in pdfs:
        sid = p.name[:3]
        for ch in chunk_text(read_pdf(p), size=1800, overlap=200):
            if ch.strip():
                chunks.append({"source_id": sid, "text": ch})
    json.dump(chunks, open(CHUNK_CACHE, "w", encoding="utf-8"), ensure_ascii=False)
    return chunks

_B1 = None
def retrieve_chunks(query, k=5):
    """B1: TF-IDF over raw PDF text chunks, top k."""
    global _B1
    if _B1 is None:
        from sklearn.feature_extraction.text import TfidfVectorizer
        ch = build_chunk_index(DATA_DIR)
        if not ch:
            return "(no source text available)"
        v = TfidfVectorizer(stop_words="english").fit([c["text"] for c in ch])
        _B1 = (ch, v, v.transform([c["text"] for c in ch]))
    ch, v, M = _B1
    from sklearn.metrics.pairwise import cosine_similarity
    sim = cosine_similarity(v.transform([query]), M)[0]
    top = sim.argsort()[::-1][:k]
    return "\n\n".join(f"[{ch[i]['source_id']}] {ch[i]['text'][:1200]}" for i in top)

def evidence_for(baseline, query):
    cfg = BASELINES[baseline]
    if cfg["kind"] == "chunk":
        return retrieve_chunks(query), []
    return retrieve(query, multihop=(cfg["kind"] == "kgN"), with_confidence=cfg["conf"])

# The cache is keyed by the generator AND the prompt version. Editing a system prompt
# without bumping PROMPT_VERSION would silently reuse answers written under the old one.
REGEN = json.load(open(ANS_CACHE, encoding="utf-8")) if ANS_CACHE.exists() else {}
if (REGEN.get("model") != MODEL_GEN or REGEN.get("queries") != QUERIES
        or REGEN.get("prompt_version") != PROMPT_VERSION):
    if REGEN:
        print("[cache] regenerated answers were made with a different model or query set "
              "- discarded")
    REGEN = {"model": MODEL_GEN, "prompt_version": PROMPT_VERSION, "queries": QUERIES,
             "answers": {b: {} for b in BASES}, "levels": {b: {} for b in BASES}}
REGEN.setdefault("answers", {b: {} for b in BASES})
REGEN.setdefault("levels", {b: {} for b in BASES})

def _complete(store):
    return bool(QUERIES) and all(len(store.get(b, {})) == len(QUERIES) for b in BASES)

todo = [(b, i) for b in BASES for i in range(len(QUERIES))
        if str(i) not in REGEN["answers"].get(b, {})]
HAVE_REGEN = _complete(REGEN["answers"])      # defined before any call, so a failure here
print(f"answers to generate: {len(todo)} of {len(BASES) * len(QUERIES)}")  # cannot cascade

if REGEN_ANSWERS and todo:
    if not HAS_OPENAI:
        raise RuntimeError("OPENAI_API_KEY is not set - re-run Part 0 or leave "
                           "REGEN_ANSWERS = False")
    from openai import OpenAI
    cli = OpenAI()
    for n_, (b, i) in enumerate(todo, 1):
        ev, lv = evidence_for(b, QUERIES[i])
        try:
            r = cli.chat.completions.create(
                model=MODEL_GEN, temperature=0.0, max_tokens=900,
                messages=[{"role": "system", "content": BASELINES[b]["sys"]},
                          {"role": "user", "content":
                           f"Question: {QUERIES[i]}\n\nRetrieved evidence:\n{ev}"}])
            txt = (r.choices[0].message.content or "").strip()
        except Exception as e:
            print(f"  [warn] {b} q{i+1}: {str(e)[:110]}"); continue
        REGEN["answers"].setdefault(b, {})[str(i)] = txt
        REGEN["levels"].setdefault(b, {})[str(i)] = dict(Counter(lv))
        if n_ % 10 == 0 or n_ == len(todo):
            print(f"  {n_}/{len(todo)}")
            json.dump(REGEN, open(ANS_CACHE, "w", encoding="utf-8"), ensure_ascii=False)
    json.dump(REGEN, open(ANS_CACHE, "w", encoding="utf-8"), ensure_ascii=False)
elif not todo:
    print("all answers already generated")
else:
    print("REGEN_ANSWERS = False - set it True once to produce the new evaluation set")

HAVE_REGEN = _complete(REGEN["answers"])
if HAVE_REGEN:
    print(f"\nregenerated set complete: {len(BASES)} x {len(QUERIES)} answers, model {MODEL_GEN}")
    print(f"{'':<4}{'mean answer length':>20}{'Conflicted items':>18}{'Disputed items':>16}")
    for b in BASES:
        L = REGEN["answers"][b]
        lv = REGEN["levels"].get(b, {})
        c = sum(d.get("Conflicted", 0) for d in lv.values())
        d_ = sum(d.get("Disputed", 0) for d in lv.values())
        print(f"{b:<4}{int(np.mean([len(v) for v in L.values()])):>20}{c:>18}{d_:>16}")
    print("\nB3 and B5 retrieve the same subgraph; only B5 is shown the Confidence Levels, "
          "so\nthe difference between them is the auditing layer and nothing else.")
    MANIFEST["regenerated_answers"] = {"model": MODEL_GEN, "n": len(BASES) * len(QUERIES),
                                       "levels": REGEN["levels"]}


answers to generate: 0 of 100
all answers already generated

regenerated set complete: 5 x 20 answers, model gpt-4o
      mean answer length  Conflicted items  Disputed items
B1                   381                 0               0
B2                   371                68               0
B3                   648                72               0
B4                   558                72               0
B5                   657                72               0

B3 and B5 retrieve the same subgraph; only B5 is shown the Confidence Levels, so
the difference between them is the auditing layer and nothing else.


In [41]:
# ====== cell 40 ======
# ---------------- judge the regenerated answers with the same rubric ----------------
REGEN_JUDGE = True              # True: judging calls for whatever is not yet cached
                                # (what gets REPORTED is Part 0's REPORT, not a switch here)
RJG_CACHE = OUT_DIR / "judge_scores_regenerated.json"

# J3 is the open-weight judge (§3.5: three judges, two proprietary vendors plus one
# open-weight model). It runs LOCALLY through an OpenAI-compatible server - LM Studio,
# Ollama, llama.cpp or vLLM all expose one - rather than through a hosted API.
#
# This is the reproducible choice, not merely a fallback. Groq retired
# llama-3.3-70b-versatile for free and developer tiers on 2026-08-16, and the previous
# run lost J3 to eighty consecutive 404s. Open weights served locally cannot be
# deprecated out from under the deposit: a reviewer downloads the same checkpoint and
# gets the same judge. The model actually used is discovered at run time and recorded
# in the manifest, because "whatever was loaded in the GUI" is not a method description.
LOCAL_BASES = [os.environ.get("LOCAL_LLM_BASE_URL", ""),
               "http://localhost:1234/v1",     # LM Studio default
               "http://127.0.0.1:1234/v1",
               "http://localhost:11434/v1",    # Ollama
               "http://localhost:8000/v1"]     # vLLM / llama.cpp
# llama first: it is not a reasoning model, so it does not spend hundreds of tokens on a
# <think> block before emitting three integers, and it is the same family as the judge
# the recorded study used. Reasoning models are kept in the list as fallbacks.
LOCAL_PREFERENCE = ["llama", "gpt-oss", "mistral", "gemma", "qwen", "phi"]

# Discovery alone is not reproducibility. A reviewer with a different model loaded would
# get a different J3, the manifest would faithfully record a different panel, and the
# reliability coefficients would not be comparable with the deposited ones. So the
# deposited checkpoint is NAMED, and a mismatch is an error rather than a silent
# substitution. Set it to None to explore with whatever is loaded - the manifest will say
# which model answered either way.
LOCAL_MODEL_REQUIRED = "meta-llama-3.1-8b-instruct"
LOCAL_SKIP = ("embed", "embedding", "rerank", "whisper", "clip", "bge", "nomic")

def resolve_local_judge(bases=LOCAL_BASES, preference=LOCAL_PREFERENCE):
    """Find a local OpenAI-compatible server and the chat model it has loaded."""
    from openai import OpenAI
    for base in [b for b in bases if b]:
        try:
            cli = OpenAI(base_url=base, api_key="local", timeout=10)
            ids = [m.id for m in cli.models.list().data]
        except Exception:
            continue
        chat = [m for m in ids if not any(s in m.lower() for s in LOCAL_SKIP)]
        if not chat:
            return None, None, f"{base} reachable but only non-chat models: {ids[:5]}"
        for want in preference:
            for m in chat:
                if want in m.lower():
                    return base, m, f"{len(chat)} chat model(s) at {base}"
        return base, chat[0], f"{len(chat)} chat model(s) at {base}; no preferred name matched"
    return None, None, ("no local server answered on " +
                        ", ".join(b for b in bases if b) +
                        " - start LM Studio's server (Developer tab -> Start Server) "
                        "or set LOCAL_LLM_BASE_URL")

LOCAL_BASE, LOCAL_MODEL, _local_note = resolve_local_judge()
print(f"J3 (local open-weight): {LOCAL_MODEL or 'UNAVAILABLE'}  - {_local_note}")
if LOCAL_MODEL_REQUIRED and LOCAL_MODEL and LOCAL_MODEL != LOCAL_MODEL_REQUIRED:
    raise RuntimeError(
        f"J3 resolved to '{LOCAL_MODEL}' but this notebook is pinned to "
        f"'{LOCAL_MODEL_REQUIRED}'. Load that checkpoint in the local server, or set "
        f"LOCAL_MODEL_REQUIRED = None to run with a different open-weight judge - in "
        f"which case §3.5 must name the model you used and the reliability coefficients "
        f"are not comparable with the deposited ones.")
if LOCAL_MODEL_REQUIRED and LOCAL_MODEL:
    print(f"  pinned checkpoint confirmed: {LOCAL_MODEL_REQUIRED}")

def judge_call_any(kind, model, prompt):
    """Dispatch to the hosted judges of Part 9, or to the local server for J3."""
    if kind != "local":
        return judge_call(kind, model, prompt)
    from openai import OpenAI
    cli = OpenAI(base_url=LOCAL_BASE, api_key="local", timeout=600)
    r = cli.chat.completions.create(
        model=model, temperature=0, max_tokens=900,
        messages=[{"role": "system", "content": RUBRIC},
                  {"role": "user", "content": prompt}])
    raw = (r.choices[0].message.content or "") if r.choices else ""
    # Small open-weight models emit a reasoning block and often prose around the JSON,
    # so the local reply is cleaned before the shared parser sees it.
    raw = re.sub(r"<(think|thinking|reasoning)>.*?</\1>", " ", raw, flags=re.S | re.I)
    raw = re.sub(r"```(?:json)?|```", " ", raw)
    if not raw.strip():
        raise RuntimeError("empty response")
    try:
        return parse_scores(raw)
    except Exception:
        got = {k: float(v) for k, v in
               re.findall(r'"?\b(FA|CV|AC)\b"?\s*[:=]\s*([1-5](?:\.\d+)?)', raw, re.I)}
        if len(got) == 3:
            return got
        raise

JUDGE_MODELS = {"J1": ("openai",    "gpt-4o"),
                "J2": ("anthropic", "claude-sonnet-4-6")}
if LOCAL_MODEL:
    JUDGE_MODELS["J3"] = ("local", LOCAL_MODEL)
else:
    print("  J3 will be SKIPPED. Two judges is a smaller panel than §3.5 describes;")
    print("  either start the local server and re-run this cell, or say so in §3.5.")
N_JUDGES_PLANNED = len(JUDGE_MODELS)

# If the cell above stopped early, its result may be absent rather than False; read it
# defensively so one failure does not turn into a NameError here.
HAVE_REGEN = bool(globals().get("HAVE_REGEN"))

# A cached score is valid only for the model that produced it AND for the answer text it
# graded. Keeping the model check alone would let scores given to answers written under a
# different system prompt be reused silently - the same trap the detector cache had, and
# the reason the previous run's numbers could not be traced to one configuration. The
# model check on its own still does useful work: it lets J1 and J2 survive a change of J3
# without being re-billed.
_ANSV = REGEN.get("prompt_version", "unversioned")
# A score is also only valid for the RUBRIC that produced it. Editing an anchor changes
# what the number means. Stamped additively: entries written before this check carry no
# rubric_sha and are kept (with a note) rather than discarding 240 paid judgments; entries
# that carry a DIFFERENT sha are discarded.
RUBRIC_SHA = hashlib.sha256(RUBRIC.encode("utf-8")).hexdigest()[:12]
RG = json.load(open(RJG_CACHE, encoding="utf-8")) if RJG_CACHE.exists() else {}
_before = len(RG)
_unstamped = sum(1 for v in RG.values() if v and "rubric_sha" not in v)
RG = {k: v for k, v in RG.items()
      if v and v.get("model") == JUDGE_MODELS.get(k.split("|")[0], (None, None))[1]
      and v.get("answers_version") == _ANSV
      and v.get("rubric_sha", RUBRIC_SHA) == RUBRIC_SHA}
if _unstamped:
    print(f"[cache] {_unstamped} judgment(s) carry no rubric stamp (written before this "
          f"check). Kept; they are valid only if RUBRIC is unchanged.")
print(f"[cache] {len(RG)} judgment(s) reusable"
      + (f"  ({_before - len(RG)} discarded: different judge model or answer version)"
         if _before > len(RG) else ""))

if REGEN_JUDGE and HAVE_REGEN:
    for j, (kind, model) in JUDGE_MODELS.items():
        if kind == "openai" and not HAS_OPENAI:       print(f"{j}: no key, skipped"); continue
        if kind == "anthropic" and not HAS_ANTHROPIC: print(f"{j}: no key, skipped"); continue
        todo_j = [(b, i) for b in BASES for i in range(len(QUERIES))
                  if f"{j}|{b}|{i+1}" not in RG]
        print(f"{j} ({model}): {len(todo_j)} to score")
        done_j = fails = 0
        for b, i in todo_j:
            prompt = f"Question: {QUERIES[i]}\n\nAnswer:\n{REGEN['answers'][b][str(i)]}"
            got = None
            for attempt in range(4):
                try:
                    got = judge_call_any(kind, model, prompt)
                    got["model"] = model; got["answers_version"] = _ANSV
                    got["rubric_sha"] = RUBRIC_SHA; break
                except Exception as e:
                    if attempt == 3:
                        print(f"  [warn] {j}|{b}|{i+1}: {str(e)[:100]}")
                    else:
                        time.sleep(2 * (attempt + 1))
            if got:
                RG[f"{j}|{b}|{i+1}"] = got
                done_j += 1
                if done_j % 20 == 0:
                    print(f"    {done_j}/{len(todo_j)}")
                    json.dump(RG, open(RJG_CACHE, "w", encoding="utf-8"), ensure_ascii=False)
            else:
                fails += 1
                # A model that is gone, or a server that is down, fails on every call.
                # Stop after five rather than emitting eighty identical errors.
                if fails >= 5 and done_j == 0:
                    print(f"  [abort] {j}: {fails} failures and no score recorded - "
                          f"'{model}' is not answering.")
                    break
        json.dump(RG, open(RJG_CACHE, "w", encoding="utf-8"), ensure_ascii=False)
elif not HAVE_REGEN:
    print("no regenerated answer set - run the cell above with REGEN_ANSWERS = True first")
else:
    print("REGEN_JUDGE = False - set it True once to score the regenerated answers")

ROWS_REGEN = [{"judge": k.split("|")[0], "baseline": k.split("|")[1],
               "query": int(k.split("|")[2]), "FA": v["FA"], "CV": v["CV"], "AC": v["AC"]}
              for k, v in RG.items() if all(c in v for c in ("FA", "CV", "AC"))]
_expected = N_JUDGES_PLANNED * len(BASES) * len(QUERIES)
print(f"\nregenerated judgments: {len(ROWS_REGEN)} of {_expected}")

if ROWS_REGEN:
    CELL_R = {(r["judge"], r["baseline"], r["query"]): r for r in ROWS_REGEN}
    QIDS_R = sorted({r["query"] for r in ROWS_REGEN})
    JUD_R  = sorted({r["judge"] for r in ROWS_REGEN})

    _SKIPPED = Counter()

    def _mean(cellmap, judges, qids, b):
        """Mean rubric score for one baseline over the given judges and queries.

        Missing cells are skipped rather than raising, because this cell compares two runs
        that need not have the same shape. Every skip is counted: an unreported skip is how
        one baseline ends up averaged over more judges than another.
        """
        want = [(j, b, q) for j in judges for q in qids]
        v = [mean3(cellmap[k]) for k in want if k in cellmap]
        _SKIPPED[b] += len(want) - len(v)
        return float(np.mean(v)) if v else float("nan")

    # The recorded study is taken from the input file that Part 9 loaded it into, never
    # from CELL. CELL holds whatever REPORT names, so reading the baseline from it would
    # compare the reported run against itself.
    _rec_rows = RECORDED_ROWS
    CELL_REC = {(r["judge"], r["baseline"], r["query"]): r for r in _rec_rows}
    JUD_REC  = sorted({r["judge"] for r in _rec_rows})
    QIDS_REC = sorted({r["query"] for r in _rec_rows})

    # The two runs are only comparable over judges they share. Comparing a three-judge
    # mean against a two-judge mean moves the numbers by the missing judge's bias, not
    # by anything the systems did.
    SHARED = [j for j in JUD_R if j in JUD_REC]
    if set(SHARED) != set(JUD_REC) or set(SHARED) != set(JUD_R):
        print(f"\n[!] the recorded study has judges {JUD_REC} and this run has {JUD_R}.")
        print(f"    Both columns below are restricted to the shared judges {SHARED}.")

    print("\nrecorded study vs regenerated run"
          f"  (mean over judges {SHARED} and {len(QIDS_R)} queries)")
    print("  The recorded rows were renumbered at load, so both columns speak the same")
    print("  numbering. B4, the prompting control, did not exist then and shows '-'.")
    print(f"{'':<4}{'recorded':>10}{'regenerated':>14}{'delta':>9}   label")
    for b in BASES:
        new = _mean(CELL_R, SHARED, QIDS_R, b)
        if b not in REC_BASES:
            print(f"{b:<4}{'-':>10}{new:>14.2f}{'-':>9}   {BASELINE_LABELS[b]}"
                  "  (no recorded counterpart)")
            continue
        old = _mean(CELL_REC, SHARED, QIDS_REC, b)
        print(f"{b:<4}{old:>10.2f}{new:>14.2f}{new - old:>+9.2f}   {BASELINE_LABELS[b]}")
    if sum(_SKIPPED.values()):
        print(f"\n  [!] {sum(_SKIPPED.values())} cell(s) missing from the table above, by "
              f"baseline: {dict(_SKIPPED)}.")
        print("      The columns are therefore averaged over different numbers of items and")
        print("      the delta is not a clean comparison. Complete the run before quoting it.")

    ord_old = " < ".join(sorted(REC_BASES, key=lambda b: _mean(CELL_REC, SHARED, QIDS_REC, b)))
    ord_new = " < ".join(sorted(BASES, key=lambda b: _mean(CELL_R,   SHARED, QIDS_R,   b)))
    print(f"\n  ordering, recorded    : {ord_old}")
    print(f"  ordering, regenerated : {ord_new}")

    # The regenerated means over ALL of this run's judges, which is what the paper reports.
    print(f"\n  regenerated means over all {len(JUD_R)} judges {JUD_R}:")
    print("   " + "  ".join(f"{b} {_mean(CELL_R, JUD_R, QIDS_R, b):.2f}" for b in BASES))

    # B5 - B3 is the only within-run comparison in this table: same retrieval, same
    # generator, same prompt, Confidence Levels shown to B5 alone. The recorded and
    # regenerated columns differ in generator as well, so read across rows with care.
    d_old = _mean(CELL_REC, SHARED, QIDS_REC, "B5") - _mean(CELL_REC, SHARED, QIDS_REC, "B3")
    d_new = _mean(CELL_R,   JUD_R,  QIDS_R,   "B5") - _mean(CELL_R,   JUD_R,  QIDS_R,   "B3")
    print(f"\n  B5 - B3, recorded    : {d_old:+.2f}")
    print(f"  B5 - B3, regenerated : {d_new:+.2f}")
    print("  This is the ablation of the auditing layer: identical retrieval, Confidence")
    print("  Levels shown to the generator in B5 only. It is a within-run difference and")
    print("  is the one number in this table that is not confounded by the generator.")

    # Paired significance for the regenerated ablation, on the same footing as Part 9.
    try:
        from scipy.stats import wilcoxon
        pairs = [(np.mean([mean3(CELL_R[(j, "B5", q)]) for j in JUD_R if (j, "B5", q) in CELL_R]),
                  np.mean([mean3(CELL_R[(j, "B3", q)]) for j in JUD_R if (j, "B3", q) in CELL_R]))
                 for q in QIDS_R]
        pairs = [(x, y) for x, y in pairs if np.isfinite(x) and np.isfinite(y)]
        xs = np.array([p[0] for p in pairs]); ys = np.array([p[1] for p in pairs])
        diff = xs - ys
        if len(pairs) < 6:
            print(f"\n  [note] only {len(pairs)} paired queries - no test run")
        elif np.allclose(diff, 0):
            # Every query scored identically under B3 and B5. That is a result, not a
            # failed test: the judges could not tell the two apart at all.
            print(f"\n  B5 and B3 scored IDENTICALLY on all {len(pairs)} queries.")
            print("  No signed-rank test is defined; report the tie itself.")
            MANIFEST["regenerated_ablation"] = {"delta": 0.0, "n": len(pairs),
                                                "p": None, "all_tied": True}
        else:
            st = wilcoxon(xs, ys, alternative="greater", method=WILCOXON_METHOD,
                          zero_method="wilcox")
            n_tied = int(np.sum(diff == 0))
            print(f"\n  Wilcoxon B5 > B3 on the regenerated run: n={len(pairs)}"
                  f"{f' ({n_tied} tied)' if n_tied else ''}, p={st.pvalue:.3g}"
                  f"  ({'significant' if st.pvalue < 0.05 else 'NOT significant'} at 0.05)")
            MANIFEST["regenerated_ablation"] = {"delta": d_new, "n": len(pairs),
                                                "n_tied": n_tied, "p": float(st.pvalue)}
    except Exception as e:
        print(f"  [note] paired test skipped: {str(e)[:90]}")

    MANIFEST["regenerated_judging"] = {
        "n": len(ROWS_REGEN), "expected": _expected,
        "models": {j: m for j, (_, m) in JUDGE_MODELS.items()},
        "answers_version": _ANSV,
        "local_judge": {"base_url": LOCAL_BASE, "model": LOCAL_MODEL},
        "judges_present": JUD_R, "judges_shared_with_recorded": SHARED,
        "means_shared_judges": {b: _mean(CELL_R, SHARED, QIDS_R, b) for b in BASES}}

    # No switching here. Part 9 already loaded whichever run REPORT names, so this cell
    # only reports the comparison; rebinding CELL midway used to leave the two halves of
    # the notebook describing different runs.
    MANIFEST["regenerated_available"] = True


J3 (local open-weight): meta-llama-3.1-8b-instruct  - 4 chat model(s) at http://localhost:1234/v1
  pinned checkpoint confirmed: meta-llama-3.1-8b-instruct
[cache] 299 judgment(s) reusable
J1 (gpt-4o): 0 to score
J2 (claude-sonnet-4-6): 1 to score
  [warn] J2|B5|11: incomplete (stop_reason=refusal)
J3 (meta-llama-3.1-8b-instruct): 0 to score

regenerated judgments: 299 of 300

recorded study vs regenerated run  (mean over judges ['J1', 'J2', 'J3'] and 20 queries)
  The recorded rows were renumbered at load, so both columns speak the same
  numbering. B4, the prompting control, did not exist then and shows '-'.
      recorded   regenerated    delta   label
B1        1.96          2.63    +0.67   Naive RAG
B2        3.06          2.86    -0.21   Single-hop KG-RAG
B3        2.63          3.62    +0.99   Multi-hop KG-RAG without auditing
B4           -          3.31        -   Multi-hop KG-RAG + self-assessment prompt (control)  (no recorded counterpart)
B5        4.24          3.54    -0.

`cell 41`

## Part 9c — Reported evaluation  *(§4.3.1, Table 11)*

The table, Krippendorff's α, the agreement statistics and the signed-rank tests, all
computed on whichever evaluation `REPORT` names.

α uses the **ordinal** metric, which is what an ordered 1–5 rubric calls for; the interval
value is printed beside it. α is a single-rater *absolute*-agreement coefficient, so a
judge who is consistently stricter counts as disagreement; the consistency statistics
printed with it describe agreement on *ordering*, which is what the paired tests use.

In [43]:
# ====== cell 42 ======
# ------------------- which evaluation this notebook reports -------------------
# Applied after Part 9b, so that one top-to-bottom execution has the regenerated files in
# hand by the time it gets here. Everything downstream - the table below, alpha, the
# agreement statistics, Part 9d's UR, and Figures F8/F9 and Tables T11-T13 in Part 12 -
# reads CELL, so all of them describe the run named here and no other.
#
# REPORT is honoured or the cell RAISES. There is deliberately no fallback: a notebook
# that quietly reports the recorded study while REPORT says "regenerated" produces a
# manuscript whose numbers cannot be traced to its own configuration, and the only
# warning is a line of stdout that scrolls past. Failing here costs one re-execution of
# Part 9b; not failing here costs a retraction.

def _require_rectangular(rows, label, bases):
    """Every (judge, baseline, query) cell present, or refuse to report.

    An incomplete panel is the same substitution one level down: if J3 scored 40 of its
    80 items, B3's mean is taken over three judges and B5's over two, and the difference
    between them is then partly a difference of judges. Part 9b's comparison skips
    missing cells silently, so nothing downstream would announce it.
    """
    js = sorted({r["judge"] for r in rows})
    qs = sorted({r["query"] for r in rows})
    have = {(r["judge"], r["baseline"], r["query"]) for r in rows}
    missing = [(j, b, q) for j in js for b in bases for q in qs if (j, b, q) not in have]
    if missing:
        shown = ", ".join(f"{j}|{b}|{q}" for j, b, q in missing[:8])
        raise RuntimeError(
            f"the {label} evaluation is incomplete: {len(missing)} of "
            f"{len(js) * len(bases) * len(qs)} cells are missing ({shown}"
            f"{', ...' if len(missing) > 8 else ''}). Reporting it would average different "
            f"baselines over different judges. Re-run Part 9b until every judge has scored "
            f"every (baseline, query) pair. If a judge refuses an item PERMANENTLY - a "
            f"hosted safety classifier does this and it does not clear on retry - put that "
            f"query number in EXCLUDE_QUERIES (Part 9 setup) instead: "
            f"{sorted({q for _, _, q in missing})}")
    return js, qs

if REPORT == "regenerated":
    if not (ROWS_REGEN and REGEN.get("answers")):
        raise RuntimeError(
            "REPORT = 'regenerated' but Part 9b produced no regenerated "
            f"{'answers' if not REGEN.get('answers') else 'judgments'}. "
            "Run Part 9b with REGEN_ANSWERS = True and REGEN_JUDGE = True until it reports "
            "a full set, then re-execute from there. Set REPORT = 'recorded' in Part 0 only "
            "if you intend to report the pre-fix study, which cannot support any claim "
            "about the auditing layer (see Part 9b).")
    JUDGE_ROWS, ANSWERS, EVAL_SOURCE = ROWS_REGEN, REGEN["answers"], "regenerated"
    print(f"[report] regenerated evaluation "
          f"(generator {REGEN.get('model')}, prompts {REGEN.get('prompt_version')}, "
          f"{len(JUDGE_ROWS)} judgments)")

elif REPORT == "recorded":
    if not RECORDED_ROWS:
        raise RuntimeError(
            "REPORT = 'recorded' but no recorded judgments were loaded. Point FILES['judge'] "
            "at RQ2_judge_scores_canonical.json, or set REPORT = 'regenerated'.")
    JUDGE_ROWS, ANSWERS, EVAL_SOURCE = RECORDED_ROWS, RECORDED_ANSWERS, "recorded study"
    print("[report] recorded study, by request")

else:
    raise ValueError(f"REPORT must be 'regenerated' or 'recorded', not {REPORT!r}")

# The recorded study is complete over four conditions, the regenerated run over five.
REPORT_BASES = REC_BASES if REPORT == "recorded" else BASES

if EXCLUDE_QUERIES:
    _n0 = len(JUDGE_ROWS)
    JUDGE_ROWS = [r for r in JUDGE_ROWS if r["query"] not in EXCLUDE_QUERIES]
    _left = len({r["query"] for r in JUDGE_ROWS})
    print(f"[exclude] quer{'y' if len(EXCLUDE_QUERIES) == 1 else 'ies'} "
          f"{EXCLUDE_QUERIES} dropped from the whole panel: "
          f"{_n0 - len(JUDGE_ROWS)} judgment(s) removed, {_left} quer"
          f"{'y' if _left == 1 else 'ies'} remain. The answers are still generated and "
          f"cached; they are simply not scored. State this in 3.5.")
JUDGES, QIDS = _require_rectangular(JUDGE_ROWS, EVAL_SOURCE, REPORT_BASES)
if REPORT == "recorded":
    print(f"[note] the recorded study covers {REPORT_BASES} - it has no prompting "
          "control, so Part 9e cannot run and B5 - B4 is undefined for it.")
CELL = {(r["judge"], r["baseline"], r["query"]): r for r in JUDGE_ROWS}

# A short panel is not an error - it is a legitimate degraded state - but it changes what
# the paper may claim, so it is stated here and carried into the manifest rather than left
# for the reader to infer from a judge column that happens to be missing.
PANEL_SHORT = len(JUDGES) < 3
if PANEL_SHORT:
    print(f"\n[!] {len(JUDGES)} judges, not the three §3.5 describes: {JUDGES}.")
    print("    Krippendorff's alpha, the ICCs and Kendall's W are all panel-size")
    print("    dependent, so these are NOT comparable with the recorded study's")
    print("    coefficients and §3.5/§4.3.1 must be rewritten to the panel actually used.")
    print("    Part 13 will refuse to write §3.5's model sentence for the same reason.")

print(f"\nreporting  : {EVAL_SOURCE}   judges {JUDGES}   {len(JUDGE_ROWS)} judgments "
      f"({len(JUDGES)} x {len(REPORT_BASES)} x {len(QIDS)}, complete)")
MANIFEST["reporting_source"] = REPORT
MANIFEST["judging_source"] = EVAL_SOURCE
MANIFEST["reported_panel"] = {"judges": JUDGES, "n_queries": len(QIDS),
                              "excluded_queries": list(EXCLUDE_QUERIES),
                              "excluded_reason": ("a hosted judge refused the item "
                                                  "permanently; the query is dropped from "
                                                  "the whole panel to keep the grid "
                                                  "rectangular") if EXCLUDE_QUERIES else None,
                              "n_judgments": len(JUDGE_ROWS), "rectangular": True,
                              "short_panel": PANEL_SHORT}


[report] regenerated evaluation (generator gpt-4o, prompts graded-evidence-kept-v3-deterministic, 299 judgments)
[exclude] query [11] dropped from the whole panel: 14 judgment(s) removed, 19 queries remain. The answers are still generated and cached; they are simply not scored. State this in 3.5.

reporting  : regenerated   judges ['J1', 'J2', 'J3']   285 judgments (3 x 5 x 19, complete)


In [44]:
# ====== cell 43 ======
# --------------------------- primary results (§4.3.1, Table) ---------------------------
import numpy as np
from scipy.stats import wilcoxon

if JUDGE_ROWS:
    # The label follows the data rather than being asserted: the selector cell above bound
    # JUDGE_ROWS/CELL/QIDS/JUDGES to whichever run Part 0's REPORT names, and recorded its
    # choice in EVAL_SOURCE. Nothing downstream rebinds them.
    _SRC = EVAL_SOURCE            # set by the loader in the first cell of this Part
    print(f"Multi-LLM judge evaluation - {_SRC}")
    print(f"{len(JUDGES)} judges x {len(QIDS)} queries x {len(REPORT_BASES)} baselines "
          f"= {len(JUDGE_ROWS)} judgments\n")

    LBL = globals().get("BASELINE_LABELS", {b: b for b in REPORT_BASES})
    print(f"{'System':<8}" + "".join(f"{j:>8}" for j in JUDGES) + f"{'Mean':>9}   label")
    means = {}
    for b in REPORT_BASES:
        row = [np.mean([mean3(CELL[(j, b, q)]) for q in QIDS]) for j in JUDGES]
        means[b] = float(np.mean(row))
        print(f"{b:<8}" + "".join(f"{v:>8.2f}" for v in row) + f"{means[b]:>9.2f}"
              + f"   {LBL.get(b, b)}")
    print(f"\nordering (ascending): {' < '.join(sorted(REPORT_BASES, key=lambda b: means[b]))}")

    print(f"\n{'System':<8}{'FA':>8}{'CV':>8}{'AC':>8}")
    per_crit = {}
    for b in REPORT_BASES:
        vals = [float(np.mean([CELL[(j, b, q)][c] for j in JUDGES for q in QIDS]))
                for c in ("FA", "CV", "AC")]
        per_crit[b] = dict(zip(("FA", "CV", "AC"), vals))
        print(f"{b:<8}" + "".join(f"{v:>8.2f}" for v in vals))

    print("\nInter-judge reliability (Krippendorff's alpha)")
    alphas = {}
    for c in ("FA", "CV", "AC"):
        u = [[CELL[(j, b, q)][c] for j in JUDGES] for b in REPORT_BASES for q in QIDS]
        alphas[c] = krippendorff_alpha(u)
        print(f"  {c:<8} ordinal={alphas[c]:.3f}   interval={krippendorff_alpha(u,'interval'):.3f}")
    u = [[mean3(CELL[(j, b, q)]) for j in JUDGES] for b in REPORT_BASES for q in QIDS]
    alphas["overall"] = krippendorff_alpha(u)
    print(f"  {'overall':<8} ordinal={alphas['overall']:.3f}   "
          f"interval={krippendorff_alpha(u,'interval'):.3f}")
    _M = np.array([[mean3(CELL[(j, b, q)]) for j in JUDGES] for b in REPORT_BASES for q in QIDS])
    agree = agreement_stats(_M)
    print_agreement(agree, "overall rubric mean", JUDGES)
    agree_crit = {}
    for c in ("FA", "CV", "AC"):
        s = agreement_stats([[CELL[(j, b, q)][c] for j in JUDGES] for b in REPORT_BASES for q in QIDS])
        agree_crit[c] = s
        print(f"  {c}: ICC(A,1)={s['ICC_A1']:.3f}  ICC(C,1)={s['ICC_C1']:.3f}  "
              f"ICC(C,{len(JUDGES)})={s['ICC_Ck']:.3f}  W={s['kendall_W']:.3f}")

    def holm(pvals):
        adj, prev = {}, 0.0
        for i, (k, p) in enumerate(sorted(pvals.items(), key=lambda kv: kv[1])):
            prev = max(prev, min(1.0, p * (len(pvals) - i)))
            adj[k] = prev
        return adj


    print("\nWilcoxon signed-rank, one-sided (B5 > X), Holm-corrected")
    print(f"  method = {WILCOXON_METHOD} (pinned; ties in the paired differences make the "
          "exact test invalid)")
    tie_report = {}
    for b in ("B1", "B2", "B3"):
        d = np.array([np.mean([mean3(CELL[(j, "B5", q)]) for j in JUDGES])
                      - np.mean([mean3(CELL[(j, b, q)]) for j in JUDGES]) for q in QIDS])
        tie_report[b] = {"n": len(d), "zeros": int((d == 0).sum()),
                         "tied_magnitudes": int(len(d) - len(set(np.abs(d))))}
        print(f"    B5-{b}: n={len(d)}, zero differences={tie_report[b]['zeros']}, "
              f"tied |differences|={tie_report[b]['tied_magnitudes']}")

    fam9 = {}
    for b in ("B1", "B2", "B3"):
        for c in ("FA", "CV", "AC"):
            x = [np.mean([CELL[(j, "B5", q)][c] for j in JUDGES]) for q in QIDS]
            y = [np.mean([CELL[(j, b, q)][c]    for j in JUDGES]) for q in QIDS]
            fam9[f"B5>{b} [{c}]"] = wilcoxon(x, y, alternative="greater",
                                             method=WILCOXON_METHOD).pvalue
    h9 = holm(fam9)
    print("  9-metric family:")
    for k in sorted(fam9, key=lambda k: fam9[k]):
        print(f"    {k:<16} raw p={fam9[k]:.3g}  Holm={h9[k]:.3g}"
              + ("  *" if h9[k] < 0.05 else ""))

    fam3 = {}
    paired = {b: [np.mean([mean3(CELL[(j, b, q)]) for j in JUDGES]) for q in QIDS] for b in REPORT_BASES}
    for b in ("B1", "B2", "B3"):
        fam3[f"B5>{b}"] = wilcoxon(paired["B5"], paired[b], alternative="greater",
                                   method=WILCOXON_METHOD).pvalue
    h3 = holm(fam3)
    print("  overall-mean family:")
    for k in sorted(fam3, key=lambda k: fam3[k]):
        print(f"    {k:<16} raw p={fam3[k]:.3g}  Holm={h3[k]:.3g}"
              + ("  *" if h3[k] < 0.05 else ""))

    rng = np.random.default_rng(SEED)
    bs = [np.mean(rng.choice(paired["B5"], len(QIDS))) for _ in range(10000)]
    print(f"\nB5 mean = {np.mean(paired['B5']):.2f}, bootstrap 95% CI = "
          f"[{np.percentile(bs,2.5):.2f}, {np.percentile(bs,97.5):.2f}]")

    MANIFEST["evaluation"] = {
        "source": _SRC, "n_judgments": len(JUDGE_ROWS),
        "judges": JUDGES, "means": means, "means_by_criterion": per_crit,
        "krippendorff_ordinal": {k: float(v) for k, v in alphas.items()},
        "wilcoxon_method": WILCOXON_METHOD, "wilcoxon_ties": tie_report,
        "wilcoxon_holm_9": {k: float(v) for k, v in h9.items()},
        "wilcoxon_holm_overall": {k: float(v) for k, v in h3.items()},
        "agreement": {"overall": agree, "by_criterion": agree_crit},
        "b5_bootstrap_ci95": [float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))]}
else:
    print("no recorded judgments available")

Multi-LLM judge evaluation - regenerated
3 judges x 19 queries x 5 baselines = 285 judgments

System        J1      J2      J3     Mean   label
B1          2.82    2.39    2.68     2.63   Naive RAG
B2          3.00    2.25    3.33     2.86   Single-hop KG-RAG
B3          3.58    2.95    4.35     3.63   Multi-hop KG-RAG without auditing
B4          3.40    2.75    3.81     3.32   Multi-hop KG-RAG + self-assessment prompt (control)
B5          3.58    2.89    4.11     3.53   Multi-hop KG-RAG with Self-Auditing (proposed)

ordering (ascending): B1 < B2 < B4 < B5 < B3

System        FA      CV      AC
B1          3.04    2.49    2.37
B2          3.16    2.61    2.81
B3          3.86    3.49    3.53
B4          3.74    3.19    3.04
B5          3.77    3.49    3.32

Inter-judge reliability (Krippendorff's alpha)
  FA       ordinal=0.127   interval=0.161
  CV       ordinal=0.491   interval=0.452
  AC       ordinal=0.166   interval=0.164
  overall  ordinal=0.378   interval=0.397

Agreement bey

`cell 44`

## Part 9d — Uncertainty reporting, disclosure and groundedness  *(§4.3.3)*

A fourth criterion, scored in a **separate pass** over the queries whose retrieved evidence
carries a documented disagreement, together with two measurements that need no model at
all: how often an answer uses the vocabulary of disagreement, and whether the numbers it
quotes appear in the evidence it was given.

In [46]:
# ====== cell 45 ======
# ======= Part 9d - uncertainty reporting (UR), a separate rubric over disputed queries =======
# WHY a fourth criterion. FA, CV and AC ask whether an answer is right, well-reasoned and
# usable. None of them asks whether it told the truth about what the literature does NOT
# agree on, which is the only thing the Self-Auditing layer changes. Under FA/CV/AC an
# answer that silently picks one side of a documented disagreement scores exactly like one
# that reports the disagreement - so B5 - B3 measured nothing, by construction.
#
# UR is scored in a SEPARATE pass. The FA/CV/AC prompt in Part 9 is left untouched, so
# those three stay comparable with the recorded study.
#
# The judge cannot grade UR from the answer alone: silence is honest when there was
# nothing to disclose. Each UR call therefore carries a REFERENCE list of the
# disagreements the audited corpus actually holds for that query, taken from the B5
# subgraph - the fullest retrieval - and used for every baseline, so the conditions
# are graded against the same facts rather than against their own evidence.
RUN_UR = True                 # True: (queries with a disagreement) x len(REPORT_BASES) x judges
# UR is defined only for the regenerated evaluation. The reference list of disagreements is
# read from the CURRENT graph, while REPORT="recorded" supplies answers written before the
# retrieval fix - grading those against today's evidence would score them for failing to
# disclose something their generator was never shown.
if REPORT != "regenerated":
    RUN_UR = False
    print(f"[skip] REPORT = {REPORT!r}: UR is defined only for the regenerated run, "
          "because its reference comes from the current graph.")
UR_CACHE = OUT_DIR / "ur_scores.json"

# ---------------------------------------------------------------- the reference
def disputed_items(query, cap=8):
    """The non-Verified items the audited graph holds for this query."""
    body, _ = retrieve(query, multihop=True, with_confidence=True)
    out = []
    for line in body.splitlines():
        if line.startswith("- ") and ("[Conflicted]" in line or "[Disputed]" in line):
            out.append(line[2:].strip())
    return out[:cap]

DISPUTED = {q: disputed_items(q) for q in QUERIES}
_with = [q for q in QUERIES if DISPUTED[q]]
print(f"queries whose audited evidence carries a disagreement: {len(_with)}/{len(QUERIES)}")
print("UR is scored on those only; on the rest, silence is correct and grading it would")
print("reward gratuitous hedging.\n")

# ------------------------------------------- deterministic disclosure rate (no API calls)
# A blunt, reviewer-checkable companion to the judged score: does the answer use the
# vocabulary of disagreement at all? It cannot see whether the disclosure is accurate,
# which is what the judged UR score is for.
DISCLOSE = re.compile(
    r"conflict(?:ed|ing|s)?|disput(?:ed|es?)|disagree|contradict|inconsisten|"
    r"vary (?:across|between|among)|differ (?:across|between|among)|"
    r"not settled|no consensus|sources? (?:do not|don't) agree", re.I)

def disclosure_rate(answers):
    out = {}
    for b in REPORT_BASES:
        hit = tot = 0
        for i, q in enumerate(QUERIES):
            if not DISPUTED[q]:
                continue
            tot += 1
            hit += bool(DISCLOSE.search(answers.get(b, {}).get(str(i), "")))
        out[b] = (hit, tot)
    return out

# DISPUTED comes from the CURRENT graph, so "did this answer disclose?" is only a
# measurement for answers that were written from that graph. Under REPORT = "recorded"
# it is not one, exactly as for UR and groundedness, and the table is not printed.
#
# Under REPORT = "regenerated" the recorded column IS still printed, but as a labelled
# diagnostic rather than as the recorded study's disclosure rate: it says how often the
# PRE-FIX answers happened to disclose the disagreements today's audit finds. That number
# being near zero is the paper's own argument - the pre-fix retrieval never surfaced the
# conflicts, so B5 could not have disclosed them - and it is not a reproduction of
# anything the recorded study measured.
if REPORT != "regenerated":
    _rec = _reg = {}
    print(f"[skip] REPORT = {REPORT!r}: the disclosure count is scored against the "
          "disputed set derived from the current graph, which the recorded answers were "
          "never retrieved from.")
else:
    print(f"{'':<4}{'pre-fix answers':>24}{'this run':>22}")
    print(f"{'':<4}{'(diagnostic only)':>24}{'disclosed / disputed':>22}")
    _rec = disclosure_rate(RECORDED_ANSWERS) if RECORDED_ANSWERS else {}
    _reg = disclosure_rate(ANSWERS)
    for b in REPORT_BASES:
        r = _rec.get(b); g = _reg.get(b)
        f = lambda t: f"{t[0]}/{t[1]}  ({t[0]/t[1]:.0%})" if t and t[1] else "-"
        print(f"{b:<4}{f(r):>24}{f(g):>22}   {BASELINE_LABELS[b]}")
    print("\nLeft column: how often the pre-fix answers disclosed the disagreements THIS")
    print("audit finds. It is not the recorded study's own measurement - that study never")
    print("surfaced these conflicts - and must not be quoted as one.")
    print("This is a lexical count, not a judgement of accuracy. The scored UR below is.")
MANIFEST["ur_disclosure"] = {"pre_fix_answers_diagnostic": _rec, "this_run": _reg,
                             "queries_with_disagreement": len(_with),
                             "reported": REPORT == "regenerated"}

# ------------------------------- groundedness (no API calls) -------------------------
# Does the answer's arithmetic come from the retrieved evidence or from the model? Every
# number in an answer is checked against the numbers in the evidence that answer was
# given. This is the measurement the FA/CV/AC rubric cannot make: a fluent, confident,
# entirely ungrounded answer scores well on all three.
_NUM = re.compile(r"(?<![\w.])(\d+(?:\.\d+)?)(?![\w.])")
_LIST = {str(i) for i in range(1, 11)}          # markdown list numbering, not data

def groundedness(answers):
    out = {}
    for b in REPORT_BASES:
        cfg = BASELINES[b]
        if cfg["kind"] == "chunk" and "retrieve_chunks" not in globals():
            out[b] = None                  # B1's evidence needs the PDF chunk index
            continue
        used = found = withnum = 0
        for i, q in enumerate(QUERIES):
            a = answers.get(b, {}).get(str(i), "")
            if cfg["kind"] == "chunk":
                ev = retrieve_chunks(q)
            else:
                ev, _ = retrieve(q, multihop=(cfg["kind"] == "kgN"),
                                 with_confidence=cfg["conf"])
            an = set(_NUM.findall(a)) - _LIST
            if not an:
                continue
            withnum += 1
            en = set(_NUM.findall(ev))
            used += len(an); found += len(an & en)
        out[b] = (found, used, withnum)
    return out

# Same dependency as Part 9f: groundedness re-retrieves the evidence from the current
# graph, so it is a measurement only when the answers came from that graph.
if REPORT == "regenerated":
    _gr = groundedness(ANSWERS)
else:
    _gr = {b: None for b in REPORT_BASES}
    print(f"[skip] REPORT = {REPORT!r}: groundedness compares an answer's numbers with "
          "the evidence it was given, and the recorded answers' evidence predates the "
          "retrieval fix and was not deposited.")
print(f"\nGroundedness - numbers in the answer that appear in its own retrieved evidence")
print(f"{'':<4}{'answers with numbers':>22}{'numbers':>9}{'grounded':>10}{'rate':>8}")
for b in REPORT_BASES:
    if _gr[b] is None:
        print(f"{b:<4}{'-':>22}{'-':>9}{'-':>10}{'-':>8}   {BASELINE_LABELS[b]}"
              "  (chunk index not built)")
        continue
    f_, u_, w_ = _gr[b]
    print(f"{b:<4}{f'{w_}/{len(QUERIES)}':>22}{u_:>9}{f_:>10}"
          f"{(f_ / u_ if u_ else 0):>8.0%}   {BASELINE_LABELS[b]}")
print("A number counted as grounded appears verbatim in the evidence; a rounded or")
print("re-derived value is counted as ungrounded, so this is a conservative floor.")
MANIFEST["groundedness"] = {b: (None if _gr[b] is None else
                            {"grounded": _gr[b][0], "used": _gr[b][1],
                             "answers_with_numbers": _gr[b][2]}) for b in REPORT_BASES}

# ---------------------------------------------------------------- the judged score
UR_RUBRIC = """You grade ONE property of an agronomy answer: how honestly it reports
disagreement in the underlying literature. Ignore fluency, length and general helpfulness.

You are given the question, a REFERENCE list of findings on which the source documents
genuinely disagree, and the ANSWER.

Score UR from 1 to 5:
5 - names the disputed quantity, gives the divergent values or ranges, and presents none
    of them as settled.
4 - states that the sources disagree on the relevant quantity, without the divergent values.
3 - hedges in general terms ("values vary", "results differ") without tying the hedge to
    the disputed quantity.
2 - gives a value from the disputed set and presents it as established, with no hint of
    disagreement.
1 - asserts a disputed value as settled fact, or invents a value not in the sources and
    presents it confidently.

Do not reward an answer for length or for extra advice. An answer that declines to give a
number BECAUSE the sources disagree, and says so, scores 5.

Return strict JSON and nothing else: {"UR": n, "why": "<12 words or fewer>"}"""

def parse_ur(raw):
    raw = re.sub(r"<(think|thinking|reasoning)>.*?</\1>", " ", raw, flags=re.S | re.I)
    raw = re.sub(r"```(?:json)?|```", " ", raw)
    m = re.search(r'\{[^{}]*"UR"[^{}]*\}', raw, re.S) or re.search(r"\{.*\}", raw, re.S)
    if m:
        try:
            d = json.loads(m.group())
            return {"UR": float(d["UR"]), "why": str(d.get("why", ""))[:80]}
        except Exception:
            pass
    m = re.search(r'"?\bUR\b"?\s*[:=]\s*([1-5](?:\.\d+)?)', raw, re.I)
    if m:
        return {"UR": float(m.group(1)), "why": ""}
    raise RuntimeError(f"unparsable: {raw[:100]!r}")

def ur_call(kind, model, prompt):
    """Same transports as Part 9b, but the UR rubric and the UR parser."""
    if kind == "local":
        from openai import OpenAI
        cli = OpenAI(base_url=LOCAL_BASE, api_key="local", timeout=600)
        r = cli.chat.completions.create(
            model=model, temperature=0, max_tokens=900,
            messages=[{"role": "system", "content": UR_RUBRIC},
                      {"role": "user", "content": prompt}])
        raw = (r.choices[0].message.content or "") if r.choices else ""
    elif kind == "anthropic":
        import anthropic
        msg = anthropic.Anthropic().messages.create(
            model=model, max_tokens=400, temperature=0, system=UR_RUBRIC,
            messages=[{"role": "user", "content": prompt}])
        if msg.stop_reason not in ("end_turn", "stop_sequence"):
            raise RuntimeError(f"incomplete (stop_reason={msg.stop_reason})")
        raw = "".join(b.text for b in msg.content if getattr(b, "type", "") == "text")
    else:
        from openai import OpenAI
        r = OpenAI().chat.completions.create(
            model=model, temperature=0, max_tokens=400,
            messages=[{"role": "system", "content": UR_RUBRIC},
                      {"role": "user", "content": prompt}])
        raw = (r.choices[0].message.content or "") if r.choices else ""
    if not raw.strip():
        raise RuntimeError("empty response")
    return parse_ur(raw)

# Like the FA/CV/AC cache, a UR score is valid only for the judge model that produced it
# AND for the answer text it graded.
# A UR score depends on three things, not one: the answer, the rubric, and the REFERENCE
# list of disagreements the judge was shown - and that reference is derived from the graph,
# so a change in retrieval silently changes what "correct disclosure" means. All three are
# stamped. Entries without the new fields are kept (they were paid for) but reported.
UR_RUBRIC_SHA = hashlib.sha256(UR_RUBRIC.encode("utf-8")).hexdigest()[:12]
DISPUTED_SHA = hashlib.sha256(
    json.dumps(DISPUTED, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()[:12]
_URV = (f"{EVAL_SOURCE}:{REGEN.get('prompt_version', 'unversioned')}:"
        f"UR={UR_RUBRIC_SHA}:REF={DISPUTED_SHA}")
UR = json.load(open(UR_CACHE, encoding="utf-8")) if UR_CACHE.exists() else {}
UR = {k: v for k, v in UR.items()
      if v and v.get("model") == JUDGE_MODELS.get(k.split("|")[0], (None, None))[1]
      and v.get("answers_version") == _URV}
# RUN_UR = False only stops NEW calls. The results block below reads this dict, so a
# cache left from an earlier regenerated run would be printed as though it belonged to
# whatever REPORT names now. Emptying it here is what actually makes the skip a skip.
# (The file is untouched: the scores are not deleted, only ignored for this run.)
if REPORT != "regenerated" and UR:
    print(f"[skip] {len(UR)} cached UR judgment(s) ignored: they grade regenerated "
          f"answers and REPORT is {REPORT!r}. ur_scores.json is left on disk.")
    UR = {}
# Score the answers this notebook reports, which the loader in Part 9 has already put
# in ANSWERS; falling back to REGEN would score a different set when REPORT is
# "recorded".
_answers = ANSWERS
# EXCLUDE_QUERIES applies here too: an item the panel does not score must not be scored
# on the UR pass either, or UR and the primary rubric would rest on different query sets.
_todo = [(j, b, i) for j in JUDGE_MODELS for b in REPORT_BASES
         for i, q in enumerate(QUERIES) if DISPUTED[q]
         and (i + 1) not in EXCLUDE_QUERIES
         and f"{j}|{b}|{i+1}" not in UR]
print(f"\nUR judgments to make: {len(_todo)} "
      f"({len(JUDGE_MODELS)} judges x {len(REPORT_BASES)} baselines x {len(_with)} queries)")

if RUN_UR and _todo:
    for j, (kind, model) in JUDGE_MODELS.items():
        todo_j = [(b, i) for jj, b, i in _todo if jj == j]
        if not todo_j:
            continue
        print(f"{j} ({model}): {len(todo_j)} to score")
        done = fails = 0
        for b, i in todo_j:
            q = QUERIES[i]
            ref = "\n".join(f"  - {x}" for x in DISPUTED[q])
            prompt = (f"QUESTION:\n{q}\n\nREFERENCE - the sources disagree on these:\n{ref}"
                      f"\n\nANSWER:\n{_answers[b][str(i)]}")
            got = None
            for attempt in range(4):
                try:
                    got = ur_call(kind, model, prompt)
                    got["model"] = model; got["answers_version"] = _URV; break
                except Exception as e:
                    if attempt == 3:
                        print(f"  [warn] {j}|{b}|{i+1}: {str(e)[:100]}")
                    else:
                        time.sleep(2 * (attempt + 1))
            if got:
                UR[f"{j}|{b}|{i+1}"] = got; done += 1
                if done % 20 == 0:
                    print(f"    {done}/{len(todo_j)}")
                    json.dump(UR, open(UR_CACHE, "w", encoding="utf-8"), ensure_ascii=False)
            else:
                fails += 1
                if fails >= 5 and done == 0:
                    print(f"  [abort] {j}: {fails} failures, no score recorded."); break
        json.dump(UR, open(UR_CACHE, "w", encoding="utf-8"), ensure_ascii=False)
elif not _todo:
    print("all UR judgments already cached")
else:
    print("RUN_UR = False - set it True once to score uncertainty reporting")

# ---------------------------------------------------------------- results
UR_ROWS = [{"judge": k.split("|")[0], "baseline": k.split("|")[1],
            "query": int(k.split("|")[2]), "UR": v["UR"]}
           for k, v in UR.items()
           if "UR" in v and int(k.split("|")[2]) not in EXCLUDE_QUERIES]
if UR_ROWS:
    JU = sorted({r["judge"] for r in UR_ROWS})
    QU = sorted({r["query"] for r in UR_ROWS})
    UCELL = {(r["judge"], r["baseline"], r["query"]): r["UR"] for r in UR_ROWS}

    # The same guard Part 9c applies to the primary rubric. umean() averages whatever
    # cells exist, so an incomplete panel - J3 dying halfway is the realistic case - would
    # give B3 a three-judge mean and B5 a two-judge one, and the difference between them
    # would then be partly a difference of judges. UR is the paper's headline result; it
    # does not get reported off an unbalanced grid.
    _umiss = [(j, b, q) for j in JU for b in REPORT_BASES for q in QU
              if (j, b, q) not in UCELL]
    if _umiss:
        _shown = ", ".join(f"{j}|{b}|{q}" for j, b, q in _umiss[:8])
        raise RuntimeError(
            f"the UR panel is incomplete: {len(_umiss)} of "
            f"{len(JU) * len(REPORT_BASES) * len(QU)} cells are missing ({_shown}"
            f"{', ...' if len(_umiss) > 8 else ''}). Averaging it would compare baselines "
            f"over different judges. Re-run this cell with RUN_UR = True until every "
            f"judge has scored every (condition, disputed query) pair.")
    print(f"UR panel complete: {len(JU)} judges x {len(REPORT_BASES)} conditions x "
          f"{len(QU)} disputed queries = {len(JU) * len(REPORT_BASES) * len(QU)}")

    def umean(b, judges=None):
        v = [UCELL[(j, b, q)] for j in (judges or JU) for q in QU if (j, b, q) in UCELL]
        return float(np.mean(v)) if v else float("nan")
    print(f"\nUncertainty reporting (1-5), {len(UR_ROWS)} judgments over {len(QU)} "
          f"disputed queries, judges {JU}")
    print(f"{'':<4}{'UR':>7}" + "".join(f"{j:>7}" for j in JU) + "   label")
    for b in REPORT_BASES:
        print(f"{b:<4}{umean(b):>7.2f}"
              + "".join(f"{umean(b, [j]):>7.2f}" for j in JU)
              + f"   {BASELINE_LABELS[b]}")
    d = umean("B5") - umean("B3")
    print(f"\n  UR, B5 - B3 : {d:+.2f}   the auditing layer's effect on honest disclosure,")
    print("                 same retrieval and generator, Confidence Levels shown to B5 only")
    try:
        from scipy.stats import wilcoxon
        pr = [(np.mean([UCELL[(j, "B5", q)] for j in JU if (j, "B5", q) in UCELL]),
               np.mean([UCELL[(j, "B3", q)] for j in JU if (j, "B3", q) in UCELL]))
              for q in QU]
        pr = [(x, y) for x, y in pr if np.isfinite(x) and np.isfinite(y)]
        xs = np.array([p[0] for p in pr]); ys = np.array([p[1] for p in pr])
        if len(pr) >= 6 and not np.allclose(xs - ys, 0):
            st = wilcoxon(xs, ys, alternative="greater", method=WILCOXON_METHOD,
                          zero_method="wilcox")
            print(f"  Wilcoxon B5 > B3 on UR: n={len(pr)}, p={st.pvalue:.3g}"
                  f"  ({'significant' if st.pvalue < 0.05 else 'NOT significant'} at 0.05)")
            MANIFEST["ur_ablation"] = {"delta": d, "n": len(pr), "p": float(st.pvalue)}
        elif len(pr) >= 6:
            print(f"  B5 and B3 scored identically on all {len(pr)} disputed queries.")
    except Exception as e:
        print(f"  [note] paired test skipped: {str(e)[:90]}")
    if len(JU) > 1:
        units = [[UCELL.get((j, b, q)) for j in JU] for b in REPORT_BASES for q in QU]
        print(f"\n  Krippendorff alpha on UR (ordinal): "
              f"{krippendorff_alpha(units, 'ordinal'):.3f}")
        _MU = np.array([[UCELL[(j, b, q)] for j in JU] for b in REPORT_BASES for q in QU
                        if all((j, b, q) in UCELL for j in JU)])
        ur_agree = agreement_stats(_MU)
        print_agreement(ur_agree, "uncertainty reporting (UR)", JU)
        MANIFEST["ur_agreement"] = ur_agree
    MANIFEST["ur_scores"] = {"n": len(UR_ROWS), "judges": JU,
                             "queries": len(QU), "means": {b: umean(b) for b in REPORT_BASES}}


queries whose audited evidence carries a disagreement: 14/20
UR is scored on those only; on the rest, silence is correct and grading it would
reward gratuitous hedging.

             pre-fix answers              this run
           (diagnostic only)  disclosed / disputed
B1                1/14  (7%)            0/14  (0%)   Naive RAG
B2                0/14  (0%)            1/14  (7%)   Single-hop KG-RAG
B3                1/14  (7%)            0/14  (0%)   Multi-hop KG-RAG without auditing
B4                0/14  (0%)            1/14  (7%)   Multi-hop KG-RAG + self-assessment prompt (control)
B5                0/14  (0%)          10/14  (71%)   Multi-hop KG-RAG with Self-Auditing (proposed)

Left column: how often the pre-fix answers disclosed the disagreements THIS
audit finds. It is not the recorded study's own measurement - that study never
surfaced these conflicts - and must not be quoted as one.
This is a lexical count, not a judgement of accuracy. The scored UR below is.

Groundedn

`cell 46`

## Part 9e — The three-way contrast: nothing, prompting, auditing  *(§4.1.2, §4.3.3)*

B1–B3 are all this system with a component removed, so none of them answers the
question a referee asks first: **would simply telling the generator to look for
disagreement do the same job as auditing the knowledge base?** B4 answers it.

B4 receives exactly the evidence B3 and B5 receive and is shown **no** Confidence Levels;
it gets an instruction instead, written to be as strong as B5's. So:

- **B5 − B3** — the auditing layer against nothing
- **B5 − B4** — auditing against prompting *(the only contrast that is not an ablation)*
- **B4 − B3** — what the instruction alone buys

Nothing is generated or judged here. All five conditions come from Part 9b's answer set
and Part 9d's UR pass; this Part only reads them, so it must run after Part 9d.


In [107]:
# ====== cell 47 ======
# ===== Part 9e - the three-way contrast: nothing, prompting, auditing =====
# B3, B4 and B5 issue the SAME retrieval and differ only in what the generator is told:
#   B3  nothing
#   B4  an instruction to look for disagreement and report it   (the prompting control)
#   B5  the audited Confidence Levels themselves                (the proposed system)
# So B5 - B3 is the auditing layer against nothing, and B5 - B4 is auditing against
# prompting - the comparison a referee asks for first, and the one B1-B3 cannot
# make because they are all this system with a component removed.
#
# Nothing is generated or judged here: all five conditions come from Part 9b's answer set
# and Part 9d's UR pass. This Part only reads them.

_TRI = [b for b in ("B3", "B4", "B5") if b in REPORT_BASES]
if len(_TRI) < 3 or not JUDGE_ROWS or not QUERIES:
    print("[skip] Part 9e needs B3, B4 and B5 scored over a non-empty query set; "
          "run Parts 9b and 9d first.")
else:
    _qd = [i for i, q in enumerate(QUERIES)
           if DISPUTED[q] and (i + 1) not in EXCLUDE_QUERIES]

    # ---------------------------------------------------- primary rubric, the three-way
    print(f"Primary rubric (FA/CV/AC), {len(QIDS)} queries, judges {JUDGES}")
    print(f"{'':<4}{'FA':>8}{'CV':>8}{'AC':>8}{'mean':>9}   label")
    _rm = {}
    for b in _TRI:
        _crit = [float(np.mean([CELL[(j, b, q)][c] for j in JUDGES for q in QIDS]))
                 for c in ("FA", "CV", "AC")]
        _rm[b] = float(np.mean([mean3(CELL[(j, b, q)]) for j in JUDGES for q in QIDS]))
        print(f"{b:<4}" + "".join(f"{x:>8.2f}" for x in _crit)
              + f"{_rm[b]:>9.2f}   {BASELINE_LABELS[b]}")

    # ---------------------------------------------------------------- uncertainty reporting
    _um = {}
    if "UCELL" in globals() and UCELL:
        print(f"\nUncertainty reporting (UR), {len(QU)} disputed queries")
        print(f"{'':<4}{'UR':>8}" + "".join(f"{j:>8}" for j in JU) + "   label")
        for b in _TRI:
            # NOT `per`: Part 6 binds that name to the injection Counter, and Part 12
            # reads it. Rebinding it here silently broke Part 12 on a full Run All.
            _perj = [float(np.mean([UCELL[(j, b, q)] for q in QU if (j, b, q) in UCELL]))
                     for j in JU]
            _um[b] = float(np.mean(_perj))
            print(f"{b:<4}{_um[b]:>8.2f}" + "".join(f"{x:>8.2f}" for x in _perj)
                  + f"   {BASELINE_LABELS[b]}")
    else:
        print("\n[note] no UR scores - run Part 9d first")

    # ------------------------------- the judge-free disclosure count, with a paired test
    # This is the strongest evidence in the cell and it does not depend on a judge at all.
    # The UR score is an LLM reading an answer; this is a count of whether the answer used
    # the vocabulary of disagreement on a query where the audited evidence says there IS
    # one. Paired over the same queries, so McNemar, exact.
    from scipy.stats import binomtest

    # The second net is deliberately OVER-inclusive: it counts generic hedging that names
    # no disputed quantity, which the UR rubric scores as 3 and not as disclosure. It is a
    # SENSITIVITY FLOOR, not a measurement - if B4 matches B3 even here, the instruction
    # did not change disclosure behaviour. Its rate must never be quoted as a disclosure
    # rate. Pure discourse connectives ("however", "although") are excluded: they carry no
    # uncertainty semantics and including them would be indefensible.
    _WIDE = re.compile(r"conflict|disput|disagree|contradict|inconsisten|vary|varies|"
                       r"variation|differ|range from|ranges from|not settled|no consensus|"
                       r"unclear|uncertain|depend(?:s|ing) on|may differ|"
                       r"some sources|other sources", re.I)

    def _mcnemar(rx, a, b):
        f = {x: {i: bool(rx.search(ANSWERS.get(x, {}).get(str(i), ""))) for i in _qd}
             for x in (a, b)}
        nb_ = sum(1 for i in _qd if f[a][i] and not f[b][i])
        nc_ = sum(1 for i in _qd if f[b][i] and not f[a][i])
        if nb_ + nc_ == 0:
            return sum(f[a].values()), sum(f[b].values()), 0, 0, None
        p = float(binomtest(nb_, nb_ + nc_, 0.5, alternative="greater").pvalue)
        return sum(f[a].values()), sum(f[b].values()), nb_, nc_, p

    DISC = {}
    _rate = lambda n: f"{n}/{len(_qd)} ({n/len(_qd):.0%})" if _qd else f"{n}/0"
    for tag, rx in () if not _qd else (("vocabulary-only disclosure (S1, sensitivity; NOT the reported measure)", DISCLOSE),
                    ("over-inclusive net (sensitivity floor, do NOT quote as a rate)",
                     _WIDE)):
        print(f"\n{tag}, over the {len(_qd)} disputed queries (judge-free)")
        fam = {}
        for a, b in (("B5", "B3"), ("B5", "B4"), ("B4", "B3")):
            na, nbb, x, y, p = _mcnemar(rx, a, b)
            line = f"  {a} {_rate(na)}  vs  {b} {_rate(nbb)}"
            if p is None:
                print(line + "   no discordant pairs")
            else:
                fam[f"{a}>{b}"] = p
                print(line + f"   McNemar exact one-sided p={p:.4f}"
                             f"  (discordant {x}+{y})")
        adj, prev = {}, 0.0
        for n_, (k, p) in enumerate(sorted(fam.items(), key=lambda kv: kv[1])):
            prev = max(prev, min(1.0, p * (len(fam) - n_))); adj[k] = prev
        if adj:
            print("  Holm: " + "  ".join(f"{k} {v:.4f}{'*' if v < 0.05 else ''}"
                                         for k, v in sorted(adj.items())))
        DISC[tag] = {"raw": fam, "holm": adj}
    if not _qd:
        print("\n[note] no query carries a confirmed disagreement, so there is nothing to "
              "disclose and the disclosure test is undefined on this run.")
    print("\nThe second net is the robustness check, not a second measurement: if B4")
    print("matches B3 there too, its instruction did not change disclosure behaviour and")
    print("the narrow result is not an artefact of the narrow regex. Quote the narrow")
    print("rate in the paper and cite the wide one only as the sensitivity check.")

    # ----------------------------------------------------------------- the two contrasts
    _pv = {}
    if _um:
        print("\n" + "=" * 74)
        print(f"  B5 - B3 on UR : {_um['B5'] - _um['B3']:+.2f}   auditing vs nothing")
        print(f"  B5 - B4 on UR : {_um['B5'] - _um['B4']:+.2f}   AUDITING vs PROMPTING, "
              "identical retrieval")
        print(f"  B4 - B3 on UR : {_um['B4'] - _um['B3']:+.2f}   what the instruction "
              "alone buys")
        print("=" * 74)
        for a, b in (("B5", "B4"), ("B4", "B3"), ("B5", "B3")):
            x = [float(np.mean([UCELL[(j, a, q)] for j in JU if (j, a, q) in UCELL]))
                 for q in QU]
            y = [float(np.mean([UCELL[(j, b, q)] for j in JU if (j, b, q) in UCELL]))
                 for q in QU]
            d = np.array(x) - np.array(y)
            if len(QU) < 6 or np.allclose(d, 0):
                print(f"  {a} > {b}: no test (n={len(QU)}, all tied={bool(np.allclose(d,0))})")
                continue
            p = float(wilcoxon(x, y, alternative="greater",
                               method=WILCOXON_METHOD).pvalue)
            _pv[f"{a}>{b}"] = p
            print(f"  Wilcoxon {a} > {b} on UR: n={len(QU)}, p={p:.4g}"
                  f"  ({'significant' if p < 0.05 else 'NOT significant'} at 0.05)")

    print("\nHow to write this up:")
    _dp = DISC.get("vocabulary-only disclosure (S1, sensitivity; NOT the reported measure)", {}).get("holm", {}).get("B5>B4", 1.0)
    _up = _pv.get("B5>B4", 1.0)
    if _dp < 0.05 and _up >= 0.05:
        print("  Split verdict, and the judge-free measure is the stronger one. On the")
        print("  judged UR scale B5 exceeds the prompting control but does not reach")
        print(f"  significance (p={_up:.3f}, n={len(QU)}); on the disclosure count it does")
        print(f"  (Holm p={_dp:.3f}). Report BOTH, in that order, and say plainly that the")
        print("  UR test is underpowered at this n. Then give the mechanism from the answer")
        print("  text: instructed to look for disagreement, the generator surfaces the")
        print("  divergent values but reconciles them; the audited levels supply the")
        print("  judgment that they are unresolved.")
    elif _up < 0.05:
        print("  B5 beats the prompting control on the judged scale. The audited Confidence")
        print("  Levels supply something the instruction does not, and 4.1.2 should present")
        print("  B4 as the prompting control that makes B5's margin interpretable.")
    else:
        print("  B5 does NOT beat the prompting control on either measure. Do not bury this:")
        print("  report it, and move the contribution claim from generation time to the")
        print("  knowledge base - the audited levels are computed once, reusable,")
        print("  inspectable and attributable to specific triplets, none of which a prompt")
        print("  provides. Then say plainly that at generation time an instruction achieves")
        print("  comparable disclosure on this corpus.")

    MANIFEST["prompting_control"] = {
        "conditions": _TRI, "rubric_means": _rm, "ur_means": _um,
        "ur_deltas": ({"B5-B3": _um["B5"] - _um["B3"], "B5-B4": _um["B5"] - _um["B4"],
                       "B4-B3": _um["B4"] - _um["B3"]} if _um else {}),
        "wilcoxon": _pv, "disclosure_mcnemar": DISC,
        "b4_system_prompt": BASELINES["B4"]["sys"],
        "b5_system_prompt": BASELINES["B5"]["sys"]}


Primary rubric (FA/CV/AC), 19 queries, judges ['J1', 'J2', 'J3']
          FA      CV      AC     mean   label
B3      3.86    3.49    3.53     3.63   Multi-hop KG-RAG without auditing
B4      3.74    3.19    3.04     3.32   Multi-hop KG-RAG + self-assessment prompt (control)
B5      3.77    3.49    3.32     3.53   Multi-hop KG-RAG with Self-Auditing (proposed)

Uncertainty reporting (UR), 14 disputed queries
          UR      J1      J2      J3   label
B3      2.36    1.86    2.71    2.50   Multi-hop KG-RAG without auditing
B4      3.12    2.86    3.07    3.43   Multi-hop KG-RAG + self-assessment prompt (control)
B5      3.90    3.93    3.86    3.93   Multi-hop KG-RAG with Self-Auditing (proposed)

vocabulary-only disclosure (S1, sensitivity; NOT the reported measure), over the 14 disputed queries (judge-free)
  B5 10/14 (71%)  vs  B3 0/14 (0%)   McNemar exact one-sided p=0.0010  (discordant 10+0)
  B5 10/14 (71%)  vs  B4 1/14 (7%)   McNemar exact one-sided p=0.0020  (discordant 9+0)


part 9e 2nd cell

In [83]:
# ---------- Part 9e (2) - specific disclosure, stricter and still judge-free ----------
# WHY this cell exists. The count in the cell above asks only whether the answer used the
# VOCABULARY of disagreement. A referee's first objection is that a keyword match cannot
# tell whether the answer disclosed the RIGHT disagreement: an answer that says "sources
# conflict" about nothing in particular scores the same as one that states the divergent
# values. This cell raises the bar on the same answers, with no API call and no judge.
#
#   S1  vocabulary            the measure above, repeated as the reference point
#   S2  + divergent values    S1 AND the answer contains at least TWO of the distinct
#                             values the retrieved evidence reports for ONE parameter that
#                             carries a non-Verified item - i.e. it states what the
#                             disagreement IS
#   S3  + names the quantity  S2 AND the answer also names that parameter
#
# S2/S3 are eligibility-limited by construction: a query can earn them only if its
# retrieved evidence actually holds two different values for a flagged parameter. The
# eligible count is printed, and all three levels are reported over the SAME disputed
# queries, so an ineligible query is a miss for every condition alike and the McNemar
# pairing stays valid.
#
# Judge-free is not model-independent: the disputed set AND the value sets both come from
# the audit under test. Say so in 3.6.
from scipy.stats import binomtest
 
# Numbers, with unit exponents removed first: "5.0 mg L-1" must yield {5.0}, not {5.0, 1.0},
# or the exponent of every unit becomes a spurious shared "value".
_SNUM = re.compile(r"(?<![\w.])(\d+(?:\.\d+)?)(?![\w.])")
_SEXP = re.compile(r"(?<=[A-Za-z])[\^\-−–]+\s?\d+")
def _snums(t):
    return {round(float(x), 6) for x in _SNUM.findall(_SEXP.sub(" ", t))}
 
_STRICT_OK = ("REPORT" in globals() and REPORT == "regenerated"
              and "ANSWERS" in globals() and "retrieve" in globals())
if not _STRICT_OK:
    print("[skip] the strict disclosure count needs the regenerated answers.")
else:
    _qd2 = [i for i, q in enumerate(QUERIES)
            if DISPUTED[q] and (i + 1) not in EXCLUDE_QUERIES]
 
    _PLAB = {kg.nodes.get(k, {}).get("label", k): k for k in PARAM_SYNONYMS}
 
    def _param_values(q):
        """{parameter: {distinct values in its evidence}} for parameters that carry a
        non-Verified item in this query's retrieved evidence and report >= 2 values."""
        body, _ = retrieve(q, multihop=True, with_confidence=True)
        _acc = {}
        for line in body.splitlines():
            if not line.startswith("- ") or "-->" not in line:
                continue
            head, tail = line[2:].split("-->", 1)
            param = head.split("--")[0].strip()
            # keep only the value segment: the condition clause and the source tag can
            # both carry digits (years, site codes) that are not reported values
            seg = tail.split("(condition:")[0].split("  <")[0]
            d = _acc.setdefault(param, {"vals": set(), "flag": False})
            d["vals"] |= _snums(seg)
            if "[Conflicted]" in line or "[Disputed]" in line:
                d["flag"] = True
        return {p: d["vals"] for p, d in _acc.items() if d["flag"] and len(d["vals"]) >= 2}
 
    def _names(ans, param):
        """Does the answer name this parameter? Short tokens carrying an upper-case letter
        are abbreviations (DO, EC, pH, PPFD) and are matched CASE-SENSITIVELY - otherwise
        the English verb "do" would count as naming dissolved oxygen. Lower-case tokens of
        two characters or fewer are dropped for the same reason; the long-form synonyms
        ("dissolved oxygen") carry those parameters instead."""
        k = _PLAB.get(param)
        _toks = {param, param.replace("_", " ")}
        if k:
            _toks |= {k} | {s.strip() for s in PARAM_SYNONYMS[k]}
        for c in (t.strip() for t in _toks):
            if not c:
                continue
            if any(ch.isupper() for ch in c) and len(c) <= 4:
                if re.search(rf"\b{re.escape(c)}\b", ans):
                    return True
            elif len(c) > 2 and re.search(rf"\b{re.escape(c)}\b", ans, re.I):
                return True
        return False
 
    PV = {i: _param_values(QUERIES[i]) for i in _qd2}
    _elig = [i for i in _qd2 if PV[i]]
 
    def _hit(b, i, level):
        ans = ANSWERS.get(b, {}).get(str(i), "")
        if not DISCLOSE.search(ans):
            return False
        if level == 1:
            return True
        _anum = _snums(ans)
        return any(len(_anum & vals) >= 2 and (level == 2 or _names(ans, p))
                   for p, vals in PV[i].items())
 
    def _mc(level, a, b):
        f = {x: {i: _hit(x, i, level) for i in _qd2} for x in (a, b)}
        nb_ = sum(1 for i in _qd2 if f[a][i] and not f[b][i])
        nc_ = sum(1 for i in _qd2 if f[b][i] and not f[a][i])
        p = (float(binomtest(nb_, nb_ + nc_, 0.5, alternative="greater").pvalue)
             if nb_ + nc_ else None)
        return sum(f[a].values()), sum(f[b].values()), nb_, nc_, p
 
    print(f"disputed queries: {len(_qd2)}   eligible for S2/S3 (evidence reports >= 2 "
          f"values for a flagged parameter): {len(_elig)}")
    print("An ineligible query is a miss for every condition, so the levels stay paired.\n")
 
    _LV = {1: "S1  vocabulary only (the measure in the cell above)",
           2: "S2  + states two divergent values   <- report this one",
           3: "S3  + also names the quantity"}
    _STRICTLV = {}
    for lv in (1, 2, 3):
        print(f"{_LV[lv]}, over {len(_qd2)} disputed queries (judge-free)")
        _fam = {}
        for a, b in (("B5", "B3"), ("B5", "B4"), ("B4", "B3")):
            na, nbb, x, y, p = _mc(lv, a, b)
            line = (f"  {a} {na}/{len(_qd2)} ({na/len(_qd2):.0%})  vs  "
                    f"{b} {nbb}/{len(_qd2)} ({nbb/len(_qd2):.0%})")
            if p is None:
                print(line + "   no discordant pairs")
            else:
                _fam[f"{a}>{b}"] = p
                print(line + f"   McNemar exact one-sided p={p:.4f}  (discordant {x}+{y})")
        _adj, prev = {}, 0.0
        for n_, (k, p) in enumerate(sorted(_fam.items(), key=lambda kv: kv[1])):
            prev = max(prev, min(1.0, p * (len(_fam) - n_)))
            _adj[k] = prev
        if _adj:
            print("  Holm within this level: "
                  + "  ".join(f"{k} {v:.4f}{'*' if v < 0.05 else ''}"
                              for k, v in sorted(_adj.items())))
        _STRICTLV[f"S{lv}"] = {
            "counts": {b: sum(_hit(b, i, lv) for i in _qd2) for b in ("B3", "B4", "B5")},
            "n": len(_qd2), "raw": _fam, "holm": _adj}
        print()
 
    # what each condition actually wrote, so the count is auditable rather than asserted
    print("per-query detail at S2")
    print(f"  {'q':>3}  {'flagged parameter (values in evidence)':<46}{'B3':>4}{'B4':>4}{'B5':>4}")
    for i in _qd2:
        _p0 = max(PV[i], key=lambda p: len(PV[i][p])) if PV[i] else None
        _desc = (f"{_p0} ({', '.join(f'{v:g}' for v in sorted(PV[i][_p0])[:4])})"
                if _p0 else "- not eligible: no parameter with two values -")
        print(f"  {i+1:>3}  {_desc[:46]:<46}"
              + "".join(f"{'Y' if _hit(b, i, 2) else '.':>4}" for b in ("B3", "B4", "B5")))
 
    MANIFEST["strict_disclosure"] = {
        "levels": _STRICTLV, "n_disputed": len(_qd2), "n_eligible": len(_elig),
        "definition": ("S1 disagreement vocabulary; S2 also >= 2 distinct values reported "
                       "for one flagged parameter in the retrieved evidence; S3 also names "
                       "that parameter. Judge-free, but the disputed set and the value sets "
                       "are supplied by the audit under test, so NOT model-independent.")}
    print("\nReport S2 as the specific-disclosure measure, S1 as the looser companion.")

disputed queries: 14   eligible for S2/S3 (evidence reports >= 2 values for a flagged parameter): 14
An ineligible query is a miss for every condition, so the levels stay paired.

S1  vocabulary only (the measure in the cell above), over 14 disputed queries (judge-free)
  B5 10/14 (71%)  vs  B3 0/14 (0%)   McNemar exact one-sided p=0.0010  (discordant 10+0)
  B5 10/14 (71%)  vs  B4 1/14 (7%)   McNemar exact one-sided p=0.0020  (discordant 9+0)
  B4 1/14 (7%)  vs  B3 0/14 (0%)   McNemar exact one-sided p=0.5000  (discordant 1+0)
  Holm within this level: B4>B3 0.5000  B5>B3 0.0029*  B5>B4 0.0039*

S2  + states two divergent values   <- report this one, over 14 disputed queries (judge-free)
  B5 9/14 (64%)  vs  B3 0/14 (0%)   McNemar exact one-sided p=0.0020  (discordant 9+0)
  B5 9/14 (64%)  vs  B4 0/14 (0%)   McNemar exact one-sided p=0.0020  (discordant 9+0)
  B4 0/14 (0%)  vs  B3 0/14 (0%)   no discordant pairs
  Holm within this level: B5>B3 0.0039*  B5>B4 0.0039*

S3  + also names 

In [ ]:
part 9e 3rd cell

In [85]:
# ---------- Part 9e (3) - what the S2 count actually matched, item by item ----------
# A count is only as good as the reader's ability to check it. This cell prints, for every
# disputed query, WHICH parameter and WHICH two values the answer and the evidence share,
# and the sentence the answer used. It computes nothing new - it is the audit trail for the
# number reported in the cell above, and the material for the worked example in Discussion.
if "PV" not in globals():
    print("[skip] run the strict-disclosure cell above first.")
else:
    def _match(b, i):
        ans = ANSWERS.get(b, {}).get(str(i), "")
        if not DISCLOSE.search(ans):
            return None
        _a = _snums(ans)
        best = None
        for p, vals in PV[i].items():
            ov = sorted(_a & vals)
            if len(ov) >= 2 and (best is None or len(ov) > len(best[1])):
                best = (p, ov)
        return best
 
    def _sentence(ans):
        """The sentence carrying the disagreement statement."""
        for s in re.split(r"(?<=[.!?])\s+", ans):
            if DISCLOSE.search(s):
                return " ".join(s.split())
        return ""
 
    print("S2 audit trail - every disputed query, what B5 matched and what it wrote")
    print("=" * 78)
    _n_named = 0
    for i in _qd2:
        m = _match("B5", i)
        mark = "HIT " if m else "miss"
        print(f"\n[{mark}] q{i+1}   {QUERIES[i][:66]}")
        if m:
            named = _names(ANSWERS["B5"][str(i)], m[0])
            _n_named += bool(named)
            print(f"        matched : {m[0]} = {', '.join(f'{v:g}' for v in m[1])}"
                  f"   ({'parameter named' if named else 'PARAMETER NOT NAMED'})")
            print(f"        evidence: {m[0]} reports "
                  f"{', '.join(f'{v:g}' for v in sorted(PV[i][m[0]]))}")
        else:
            _flag = ", ".join(sorted(PV[i])) or "-"
            print(f"        flagged parameters in evidence: {_flag}")
        _s = _sentence(ANSWERS["B5"].get(str(i), ""))
        print(f"        B5 wrote: {(_s[:150] + '...') if len(_s) > 150 else (_s or '(no disagreement sentence)')}")
 
    print("\n" + "=" * 78)
    print(f"B5 hits at S2: {sum(1 for i in _qd2 if _match('B5', i))}/{len(_qd2)}"
          f"   of which the parameter is named: {_n_named}")
    print("\nB4 and B3, for the record - the sentences behind their S1 counts")
    for b in ("B4", "B3"):
        _hits = [i for i in _qd2 if DISCLOSE.search(ANSWERS.get(b, {}).get(str(i), ""))]
        print(f"\n  {b}: S1 hits on quer{'y' if len(_hits)==1 else 'ies'} "
              f"{[i+1 for i in _hits] or '-'}")
        for i in _hits:
            _s = _sentence(ANSWERS[b][str(i)])
            print(f"     q{i+1}: {(_s[:140] + '...') if len(_s) > 140 else _s}")
    print("\nPaste the strongest HIT into the Discussion as the worked example, and one B4")
    print("sentence beside it: that pair is the qualitative core of the paper.")

S2 audit trail - every disputed query, what B5 matched and what it wrote

[HIT ] q1   What is the optimal water temperature for wasabi hydroponic cultiv
        matched : water_temp = 12, 13, 24   (parameter named)
        evidence: water_temp reports 1.2, 12, 13, 24
        B5 wrote: These values are conflicted, indicating a lack of consensus.

[HIT ] q3   What is the recommended pH range for the wasabi nutrient solution?
        matched : pH = 5.5, 6, 6.5, 7.5   (parameter named)
        evidence: pH reports 2.5, 4, 5.5, 5.8, 6, 6.5, 7, 7.2, 7.5, 8, 11
        B5 wrote: While there are conflicting reports on the tested range for irrigation water (6.0 to 7.5), the optimal nutrient solution pH for wasabi remains within ...

[HIT ] q4   How does low dissolved oxygen (DO) affect wasabi rhizome yield?
        matched : DO = 5, 9.5   (parameter named)
        evidence: DO reports 5, 9.5
        B5 wrote: While the critical threshold for DO is conflicted in the literature, with values rangi

In [ ]:
part 9e 4th cell

In [87]:
# ---------- Part 9e (4) - two corrections the audit trail exposed, and the provenance ----------
# 1. NEGATION. The vocabulary net counts "there is no conflicting information" as a
#    disclosure, because it matches on the word alone. Both B4's single S1 hit and one of
#    B5's ten are negated statements, so the raw S1 count overstates both. A disclosure is
#    counted here only when at least one match is NOT preceded by a negation cue in its own
#    sentence. This is still judge-free and still a text rule.
# 2. TOPICALITY. A query is "disputed" when ANY retrieved item is non-Verified, and
#    multi-hop retrieval pulls in parameters the question never asked about - four of the
#    five misses are queries whose flagged parameter is not the query's own. Not disclosing
#    an irrelevant disagreement is correct behaviour, not a miss. The topical subset is
#    reported ALONGSIDE the full set, never instead of it: the full set stays the primary,
#    conservative denominator.
# 3. PROVENANCE. The raw evidence lines behind every matched value, so a suspicious number
#    can be traced instead of trusted.
if "PV" not in globals():
    print("[skip] run the strict-disclosure cell first.")
else:
    _NEG = re.compile(r"\b(no|not|never|none|neither|nor|without|n't)\b", re.I)
 
    def _pos_disclose(ans):
        """True if the answer asserts a disagreement, rather than denying one."""
        for s in re.split(r"(?<=[.!?])\s+", ans):
            for m in DISCLOSE.finditer(s):
                if not _NEG.search(s[max(0, m.start() - 45):m.start()]):
                    return True
        return False
 
    def _negated_only(ans):
        return bool(DISCLOSE.search(ans)) and not _pos_disclose(ans)
 
    print("1. NEGATION - answers whose only disagreement wording is a DENIAL of disagreement")
    for b in ("B3", "B4", "B5"):
        _raw = [i for i in _qd2 if DISCLOSE.search(ANSWERS.get(b, {}).get(str(i), ""))]
        _neg = [i for i in _raw if _negated_only(ANSWERS[b][str(i)])]
        print(f"  {b}: S1 raw {len(_raw)}/{len(_qd2)}  ->  negation-corrected "
              f"{len(_raw) - len(_neg)}/{len(_qd2)}"
              + (f"   (dropped q{[i+1 for i in _neg]})" if _neg else ""))
        for i in _neg:
            _s = next((x for x in re.split(r"(?<=[.!?])\s+", ANSWERS[b][str(i)])
                       if DISCLOSE.search(x)), "")
            print(f"      q{i+1}: {' '.join(_s.split())[:110]}")
    print("  If the corrected S1 equals S2, two differently-built text measures agree "
          "exactly.\n")
 
    print("2. TOPICALITY - is the flagged parameter one the QUERY itself asks about?")
    _topic = {}
    for i in _qd2:
        _qp = set(match_nodes(QUERIES[i])[0])
        _flagged = {(_PLAB.get(p) or p) for p in PV[i]}
        _topic[i] = bool(_qp & _flagged)
    _on = [i for i in _qd2 if _topic[i]]
    _off = [i for i in _qd2 if not _topic[i]]
    print(f"  on-topic disputed queries : {len(_on)}   q{[i+1 for i in _on]}")
    print(f"  off-topic (the disagreement is about something else): {len(_off)}"
          f"   q{[i+1 for i in _off]}")
    print(f"  {'':<4}{'all disputed':>16}{'on-topic only':>16}")
    for b in ("B3", "B4", "B5"):
        _a = sum(_hit(b, i, 2) for i in _qd2)
        _t = sum(_hit(b, i, 2) for i in _on)
        print(f"  {b:<4}{f'{_a}/{len(_qd2)}':>16}{f'{_t}/{len(_on)}':>16}")
    print("  Report the left column as the headline and the right one as the explanation")
    print("  of where the misses come from. Never report the right column alone.\n")
 
    print("3. PROVENANCE - the raw evidence lines behind each matched value")
    for i in _qd2:
        ans = ANSWERS["B5"].get(str(i), "")
        if not _hit("B5", i, 2):
            continue
        _a = _snums(ans)
        p, ov = max(((p, sorted(_a & v)) for p, v in PV[i].items() if len(_a & v) >= 2),
                    key=lambda t: len(t[1]))
        print(f"\n  q{i+1}  {p}  matched {', '.join(f'{v:g}' for v in ov)}")
        body, _ = retrieve(QUERIES[i], multihop=True, with_confidence=True)
        _shown = 0
        for line in body.splitlines():
            if line.startswith("- ") and "-->" in line \
               and line[2:].split("--")[0].strip() == p:
                print(f"      {line.strip()[:118]}")
                _shown += 1
                if _shown >= 8:
                    print("      ...")
                    break
 
    MANIFEST["disclosure_corrections"] = {
        "s1_negation_corrected": {b: sum(1 for i in _qd2
                                         if _pos_disclose(ANSWERS.get(b, {}).get(str(i), "")))
                                  for b in ("B3", "B4", "B5")},
        "on_topic_queries": [i + 1 for i in _on],
        "off_topic_queries": [i + 1 for i in _off],
        "s2_on_topic": {b: sum(_hit(b, i, 2) for i in _on) for b in ("B3", "B4", "B5")},
        "n_disputed": len(_qd2)}

1. NEGATION - answers whose only disagreement wording is a DENIAL of disagreement
  B3: S1 raw 0/14  ->  negation-corrected 0/14
  B4: S1 raw 1/14  ->  negation-corrected 0/14   (dropped q[17])
      q17: The evidence does not indicate any disagreement on these effects.
  B5: S1 raw 10/14  ->  negation-corrected 9/14   (dropped q[17])
      q17: There is no conflicting or disputed information regarding these specific effects of VPD on wasabi.
  If the corrected S1 equals S2, two differently-built text measures agree exactly.

2. TOPICALITY - is the flagged parameter one the QUERY itself asks about?
  on-topic disputed queries : 14   q[1, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 15, 17, 19]
  off-topic (the disagreement is about something else): 0   q[]
          all disputed   on-topic only
  B3              0/14            0/14
  B4              0/14            0/14
  B5              9/14            9/14
  Report the left column as the headline and the right one as the explanation
  of where 

part 9e 5th for table 5

In [99]:
# ---------- Part 9e (5) - the confirmatory family, adjusted jointly ----------
# Table 5 of the manuscript reports Holm-adjusted p values across FIVE tests: the three UR
# comparisons among B3, B4 and B5, and the two judge-free disclosure comparisons. The cells
# above adjust WITHIN each measure (disclosure in its own family of three; UR reported raw),
# so the joint adjustment the manuscript quotes is not produced anywhere else in this
# notebook. Reproducibility requires that every reported number come from the deposited
# code, so it is computed here. No API calls.
#
# WHICH disclosure measure. The manuscript reports S2 - the answer states a disagreement AND
# reports at least two of the divergent values (Section 3.6.5). That is _STRICTLV["S2"], not
# the vocabulary-only count in DISC, whose numbers differ (S1: 10/14 vs 0/14 and 1/14;
# S2: 9/14 vs 0/14 and 0/14). Reading the wrong one silently changes two p values, so the
# source is asserted rather than assumed.
_S2 = None
if "_STRICTLV" in globals() and "S2" in _STRICTLV:
    _S2 = _STRICTLV["S2"]["raw"]
elif "MANIFEST" in globals():
    _S2 = MANIFEST.get("strict_disclosure", {}).get("levels", {}).get("S2", {}).get("raw")
 
if _S2 is None or "_pv" not in globals():
    print("[skip] needs the strict-disclosure cell (_STRICTLV) and the UR contrast cell (_pv).")
else:
    _fam5 = {}
    for _k in ("B5>B3", "B5>B4", "B4>B3"):
        if _k in _pv:
            _fam5[f"UR {_k}"] = float(_pv[_k])
    for _k in ("B5>B3", "B5>B4"):
        if _k in _S2:
            _fam5[f"disclosure {_k}"] = float(_S2[_k])
 
    if len(_fam5) < 5:
        print(f"[warn] only {len(_fam5)} of the 5 confirmatory tests are available: "
              f"{sorted(_fam5)}")
    _adj, _prev = {}, 0.0
    for _n, (_k, _p) in enumerate(sorted(_fam5.items(), key=lambda kv: kv[1])):
        _prev = max(_prev, min(1.0, _p * (len(_fam5) - _n)))
        _adj[_k] = _prev
 
    print(f"Confirmatory family: {len(_fam5)} tests, Holm adjusted jointly")
    print("  disclosure = S2 (states the disagreement AND reports two divergent values)")
    print(f"\n  {'comparison':<26}{'raw p':>10}{'Holm p':>10}   significant at 0.05")
    for _k, _v in sorted(_adj.items(), key=lambda kv: kv[1]):
        print(f"  {_k:<26}{_fam5[_k]:>10.4f}{_v:>10.4f}   {'yes' if _v < 0.05 else 'no'}")
    print("\nThese are the values Table 5 reports. The disclosure-only Holm printed earlier")
    print("adjusts three tests within that measure and is NOT what Table 5 quotes.")
    MANIFEST["confirmatory_family"] = {
        "n_tests": len(_fam5), "raw": _fam5, "holm": _adj,
        "disclosure_measure": "S2",
        "definition": ("three UR comparisons among B3/B4/B5 and two judge-free specific-"
                       "disclosure comparisons (S2), adjusted jointly by Holm-Bonferroni")}

Confirmatory family: 5 tests, Holm adjusted jointly
  disclosure = S2 (states the disagreement AND reports two divergent values)

  comparison                     raw p    Holm p   significant at 0.05
  disclosure B5>B3              0.0020    0.0098   yes
  disclosure B5>B4              0.0020    0.0098   yes
  UR B5>B3                      0.0033    0.0100   yes
  UR B4>B3                      0.0289    0.0578   no
  UR B5>B4                      0.0429    0.0578   no

These are the values Table 5 reports. The disclosure-only Holm printed earlier
adjusts three tests within that measure and is NOT what Table 5 quotes.


`cell 48`

## Part 9f — The standard RAG rubric  *(§3.5, §4.3.1)*

The FA/CV/AC rubric of Part 9 is three bare labels with no definitions and no scale
anchors, and the judge was never shown the retrieved evidence. So `FA` is **not**
faithfulness — it cannot be, the judge never saw the documents — it is the judge's own
correctness call from parametric knowledge, which is the least reliable instrument
available in a knowledge-sparse domain.

This Part rescores the same answers on the criteria the RAG evaluation literature
standardises — the **RAG triad** (faithfulness, answer relevance, context relevance)
shared by RAGAS and ARES — with anchored 1–5 scales and with each condition's own
retrieved evidence supplied to the judge.

This is what makes the paper's central claim citable. Not *“our rubric could not
separate B3 from B5”* but *“the standard rubric cannot.”*

Part 9's FA/CV/AC results are kept, as the appendix comparison and as the link to the
recorded study. Cost: 3 judges × 5 conditions × 20 queries = 300 calls, cached.


In [50]:
# ====== cell 49 ======
# ===== Part 9f - the standard RAG rubric, judged WITH the evidence in view =====
# WHY this Part exists. The FA/CV/AC rubric of Part 9 has two defects that a referee will
# find, and the paper's central claim depends on it not having them:
#   1. It is three bare labels. "FA factual accuracy, CV causal validity, AC actionability"
#      with no definitions and no scale anchors - so each judge supplied its own, which is
#      part of why absolute agreement was low.
#   2. The judge was never shown the retrieved evidence. FA therefore cannot be
#      FAITHFULNESS (answer vs retrieved documents); it is the judge's own correctness
#      call from parametric knowledge - the least trustworthy instrument available in a
#      knowledge-sparse domain, and not a metric any RAG evaluation framework defines.
#
# This Part scores the same answers on the DIMENSIONS the RAG evaluation literature
# standardises - faithfulness, answer relevance and context relevance, the triad shared by
# RAGAS and ARES and named in the RAG-evaluation survey - with anchored 1-5 scales and with
# the evidence each condition actually received supplied to the judge.
#
# BE PRECISE ABOUT WHAT THIS IS. No RAGAS or ARES implementation is called. Those
# frameworks decompose an answer into atomic claims and score support as a ratio; this is
# an LLM judge on a 1-5 anchored scale over the same three dimensions. The manuscript must
# say "an anchored rubric on the dimensions RAGAS and ARES define", never "we evaluated
# with RAGAS". Running the real libraries would be stronger and is the obvious next step;
# the automatic groundedness measure in Part 9d is already a claim-level faithfulness
# proxy that needs no judge at all.
#
# The claim this enables: it is not "our own rubric could not separate B3 from B5", it is
# "the standard rubric cannot", which is a citable statement.
#
# Part 9's FA/CV/AC results are NOT deleted. They stay as the appendix comparison and as
# the link to the recorded study.
RUN_STD   = True
STD_CACHE = OUT_DIR / "judge_scores_std.json"

# Faithfulness is "is this answer supported by THE EVIDENCE IT WAS GIVEN", and
# std_evidence() rebuilds that evidence from the CURRENT graph. For the regenerated run
# those are the same thing. For the recorded study they are not: those answers were
# written before the 3.4 retrieval fix, so scoring them against today's evidence would
# mark them unfaithful for failing to use text their generator never saw - an artefact,
# not a measurement. The recorded evidence was not deposited, so it cannot be
# reconstructed; the honest move is to decline rather than to report a number that looks
# like faithfulness and is not.
if REPORT != "regenerated":
    RUN_STD = False
    print(f"[skip] REPORT = {REPORT!r}: the standard rubric is defined only for the "
          "regenerated run, because it grades an answer against the evidence that answer "
          "received and only the regenerated evidence can be reconstructed.")
# Every condition now comes from the one answer set Part 9b builds, so there is no
# special case left: a condition is scorable exactly when REPORT_BASES contains it.
STD_BASES = [b for b in REPORT_BASES if b in BASELINES]

STD_RUBRIC = """You grade one retrieval-augmented answer on three criteria. You are given
the QUESTION, the EVIDENCE that was retrieved and shown to the system, and the ANSWER.

FTH - Faithfulness. Is every factual claim in the ANSWER supported by the EVIDENCE?
  5 every claim traceable to the evidence; no unsupported statement, no unsupported number
  4 one minor unsupported detail; nothing the answer rests on
  3 one load-bearing claim is not in the evidence
  2 several claims unsupported, or a quantity given that the evidence does not contain
  1 largely unsupported by the evidence
  Judge support, NOT truth. A claim that is correct in the real world but absent from the
  evidence is UNFAITHFUL. A claim that is wrong but present in the evidence is faithful.

ANR - Answer Relevance. Does the ANSWER address the QUESTION that was asked?
  5 answers it directly and completely; no padding, no drift
  4 answers it, with some material that was not asked for
  3 partially answers it, or answers a neighbouring question
  2 mostly off-target
  1 does not address the question
  An answer that declines to give a value BECAUSE the evidence does not settle it, and
  says so, is fully relevant: score it on whether it addresses the question, not on
  whether it produced a number.

CTX - Context Relevance. Is the EVIDENCE relevant to the QUESTION?
  5 the evidence contains what is needed to answer
  4 relevant, with some unrelated material
  3 partially relevant; key information missing
  2 mostly unrelated
  1 unrelated
  Score CTX from the QUESTION and EVIDENCE ONLY. Ignore the ANSWER entirely. CTX is a
  property of retrieval, not of the answer.

Return strict JSON and nothing else: {"FTH": n, "ANR": n, "CTX": n}"""


def parse_std(raw):
    raw = re.sub(r"<(think|thinking|reasoning)>.*?</\1>", " ", raw, flags=re.S | re.I)
    raw = re.sub(r"```(?:json)?|```", " ", raw)
    for pat in (r'\{[^{}]*"FTH"[^{}]*\}', r"\{.*\}"):
        m = re.search(pat, raw, re.S)
        if m:
            try:
                d = json.loads(m.group())
                return {c: float(d[c]) for c in ("FTH", "ANR", "CTX")}
            except Exception:
                pass
    got = {k.upper(): float(v) for k, v in
           re.findall(r'"?\b(FTH|ANR|CTX)\b"?\s*[:=]\s*([1-5](?:\.\d+)?)', raw, re.I)}
    if len(got) == 3:
        return got
    raise RuntimeError(f"unparsable: {raw[:110]!r}")


def std_call(kind, model, prompt):
    """Same three transports as Part 9b/9d, with the standard rubric and its parser."""
    if kind == "anthropic":
        import anthropic
        m = anthropic.Anthropic().messages.create(
            model=model, max_tokens=500, temperature=0, system=STD_RUBRIC,
            messages=[{"role": "user", "content": prompt}])
        if m.stop_reason not in ("end_turn", "stop_sequence"):
            raise RuntimeError(f"incomplete (stop_reason={m.stop_reason})")
        raw = "".join(b.text for b in m.content if getattr(b, "type", "") == "text")
    else:
        from openai import OpenAI
        cli = (OpenAI(base_url=LOCAL_BASE, api_key="local", timeout=600)
               if kind == "local" else OpenAI())
        r = cli.chat.completions.create(
            model=model, temperature=0, max_tokens=900 if kind == "local" else 500,
            messages=[{"role": "system", "content": STD_RUBRIC},
                      {"role": "user", "content": prompt}])
        raw = (r.choices[0].message.content or "") if r.choices else ""
    if not raw.strip():
        raise RuntimeError("empty response")
    return parse_std(raw)


# The evidence each condition actually received. Cached per condition, because retrieve()
# is deterministic but not free, and B1 rebuilds a TF-IDF index on first use.
_EVC = {}
def std_evidence(b, i):
    if (b, i) not in _EVC:
        _EVC[(b, i)] = evidence_for(b, QUERIES[i])[0]
    return _EVC[(b, i)]

def std_answer(b, i):
    return ANSWERS[b][str(i)]

# Stamped with the rubric itself as well as the answer set: editing an anchor changes what
# the number means, so cached scores from the previous wording must not survive it.
# One stamp is enough now: every condition's answers live in REGEN, so REGEN's
# prompt_version covers all five. (It did not, while the proposed system was generated
# in a separate cell with its own version - that gap let an edited prompt regenerate
# its answers while the scores given to the OLD answers survived. Folding every
# condition into the main loop removes the failure mode rather than patching it.)
_STDV = (f"{EVAL_SOURCE}:{REGEN.get('prompt_version','unversioned')}:"
         f"{hashlib.sha256(STD_RUBRIC.encode('utf-8')).hexdigest()[:12]}")
SD = json.load(open(STD_CACHE, encoding="utf-8")) if STD_CACHE.exists() else {}
_b4 = len(SD)
SD = {k: v for k, v in SD.items()
      if v and v.get("answers_version") == _STDV
      and v.get("model") == JUDGE_MODELS.get(k.split("|")[0], (None, None))[1]}
if _b4 > len(SD):
    print(f"[cache] {_b4 - len(SD)} standard-rubric score(s) discarded "
          "(different judge model, answer set or rubric wording)")
# As in Part 9d: RUN_STD = False stops new calls, but the results block below reads SD.
# A cache from an earlier regenerated run would otherwise be reported under REPORT =
# "recorded", which is exactly the comparison this Part just declined to make.
if REPORT != "regenerated" and SD:
    print(f"[skip] {len(SD)} cached standard-rubric score(s) ignored: they grade "
          f"regenerated answers against regenerated evidence and REPORT is {REPORT!r}. "
          "judge_scores_std.json is left on disk.")
    SD = {}

_needs = [(j, b, i) for j in JUDGE_MODELS for b in STD_BASES for i in range(len(QUERIES))
          if (i + 1) not in EXCLUDE_QUERIES and f"{j}|{b}|{i+1}" not in SD]
print(f"standard-rubric judgments to make: {len(_needs)} "
      f"({len(JUDGE_MODELS)} judges x {len(STD_BASES)} conditions x {len(QUERIES)} queries)")

if RUN_STD and _needs:
    _dn = 0
    _fails = Counter()          # per judge, so one dead key cannot burn the whole run
    _dead = set()
    for j, b, i in _needs:
        if j in _dead:
            continue
        kind, model = JUDGE_MODELS[j]
        prompt = (f"QUESTION:\n{QUERIES[i]}\n\nEVIDENCE:\n{std_evidence(b, i)}"
                  f"\n\nANSWER:\n{std_answer(b, i)}")
        got = None
        for attempt in range(4):
            try:
                got = std_call(kind, model, prompt); break
            except Exception as e:
                if attempt == 3:
                    print(f"  [warn] {j}|{b}|{i+1}: {str(e)[:100]}")
                else:
                    time.sleep(2 * (attempt + 1))
        if got:
            got["model"] = model; got["answers_version"] = _STDV
            SD[f"{j}|{b}|{i+1}"] = got
            _dn += 1
            if _dn % 25 == 0:
                print(f"    {_dn}/{len(_needs)}")
                json.dump(SD, open(STD_CACHE, "w", encoding="utf-8"), ensure_ascii=False)
        else:
            _fails[j] += 1
            # A withdrawn model, a dead key or a stopped local server fails on every call.
            # Without this guard the cell spends 4 attempts and ~12s of backoff on each of
            # 100 items before giving up - twenty wasted minutes and no scores. Part 9b has
            # the same guard; Part 9f was missing it.
            if _fails[j] >= 5 and not any(k.startswith(f"{j}|") for k in SD):
                print(f"  [abort] {j}: {_fails[j]} failures and no score recorded - "
                      f"'{model}' is not answering. Skipping the rest of this judge.")
                _dead.add(j)
    json.dump(SD, open(STD_CACHE, "w", encoding="utf-8"), ensure_ascii=False)
    if _dead:
        print(f"  [!] judges skipped: {sorted(_dead)}. Fix them and re-run this cell; "
              "everything already scored is cached and will not be re-billed.")
elif not _needs:
    print("all standard-rubric judgments already cached")
else:
    print("RUN_STD = False - set it True once to score the standard rubric")

# ------------------------------------------------------------------------- results
STD = {(k.split("|")[0], k.split("|")[1], int(k.split("|")[2])): v for k, v in SD.items()
       if all(c in v for c in ("FTH", "ANR", "CTX"))}
if not STD:
    print("\nno standard-rubric scores yet")
else:
    # Part 9c refuses to report an incomplete panel outright. Here an incomplete
    # condition is dropped rather than fatal - B4 may legitimately not be generated yet -
    # but nothing is dropped SILENTLY, and a short judge panel is stated, because the
    # reliability coefficients below are panel-size dependent.
    JS = sorted({k[0] for k in STD})
    QS = [q for q in range(1, len(QUERIES) + 1) if q not in EXCLUDE_QUERIES]
    BS, _dropped = [], {}
    for b in STD_BASES:
        miss = [(j, q) for j in JS for q in QS if (j, b, q) not in STD]
        if miss:
            _dropped[b] = miss
        else:
            BS.append(b)
    if _dropped:
        for b, miss in _dropped.items():
            print(f"[!] {b} dropped: {len(miss)} of {len(JS)*len(QS)} cells missing "
                  f"(e.g. {miss[:3]}). Re-run with RUN_STD = True to complete it.")
    if len(JS) < 3:
        print(f"[!] {len(JS)} judge(s), not the three 3.5 describes: {JS}. The alphas "
              "below are panel-size dependent and are NOT comparable with Part 9's.")
    print(f"\nscored over judges {JS} | complete conditions {BS} | {len(QS)} queries")

    # The reported analysis is rectangular by construction - every condition in BS is
    # complete over every judge in JS - but a judge can be complete for one condition and
    # partial for another, which is how a panel silently changes shape between rows. The
    # grid is printed so the structure is read, not inferred.
    print(f"\n  cells present, judge x condition (out of {len(QS)} queries each)")
    print("        " + "".join(f"{b:>7}" for b in STD_BASES))
    for j in JS:
        row = [sum(1 for q in QS if (j, b, q) in STD) for b in STD_BASES]
        flag = "" if all(v in (0, len(QS)) for v in row) else "   <- partial"
        print(f"    {j:<4}" + "".join(f"{v:>7}" for v in row) + flag)
    print(f"    reported: conditions {BS} over judges {JS}; any condition not listed was "
          "dropped whole.")

    def sm(b, c): return float(np.mean([STD[(j, b, q)][c] for j in JS for q in QS]))

    print("\nAnchored rubric on the standard RAG evaluation dimensions "
          "(faithfulness / answer\nrelevance / context relevance), evidence shown to "
          "the judge.\nThe DIMENSIONS are the ones RAGAS and ARES define; the 1-5 "
          "anchors and the scoring\nare this notebook's own. No RAGAS or ARES "
          "implementation is called - say so in 3.5.")
    print(f"{'':<4}{'FTH':>8}{'ANR':>8}{'CTX':>8}{'FTH+ANR':>10}   label")
    for b in BS:
        v = [sm(b, c) for c in ("FTH", "ANR", "CTX")]
        print(f"{b:<4}" + "".join(f"{x:>8.2f}" for x in v)
              + f"{np.mean(v[:2]):>10.2f}   {BASELINE_LABELS[b]}")
    print("  FTH+ANR is the generation-side pair. CTX is a property of retrieval and is")
    print("  reported separately: averaging it into a system score would credit the")
    print("  generator for the retriever's work.")

    # ---- instrument check: does CTX actually ignore the ANSWER? ------------------
    # B3 and B4 are the identical-evidence pair now: both are shown the same subgraph
    # with no Confidence Levels, and differ only in their instruction. B5's evidence
    # carries the level annotations, so it is NOT interchangeable with them here.
    # CTX is instructed to score the QUESTION and EVIDENCE only. B3 and B4 receive
    # byte-identical evidence and produce different answers, so under a clean instrument
    # every paired CTX difference is exactly zero. What the difference measures is
    # therefore CONTAMINATION of a retrieval metric by the answer - a validity check on
    # the rubric, not an estimate of noise.
    #
    # It is NOT a significance threshold. A single difference of two means is one
    # realisation, not an estimated noise distribution, and "effects smaller than this
    # are indistinguishable from noise" does not follow from it. The scatter below is
    # reported descriptively and must not be used to screen other effects.
    if "B3" in BS and "B4" in BS:
        _same = all(std_evidence("B3", i) == std_evidence("B4", i)
                    for i in range(len(QUERIES)) if (i + 1) not in EXCLUDE_QUERIES)
        _d = np.array([STD[(j, "B4", q)]["CTX"] - STD[(j, "B3", q)]["CTX"]
                       for j in JS for q in QS], dtype=float)
        _nz = int(np.count_nonzero(_d))
        print("\n  instrument check - CTX is defined to ignore the ANSWER")
        print(f"    B3 / B4 evidence identical : "
              + ("yes" if _same else "NO - check retrieve(); this check is void"))
        print(f"    paired CTX differences     : n={len(_d)}, {_nz} non-zero "
              f"({_nz/len(_d):.0%}), mean {_d.mean():+.3f}")
        _pc = None
        if _nz == 0:
            print("    verdict                    : identical on every cell; the "
                  "instruction held exactly")
        else:
            _pc = float(wilcoxon(_d, alternative="two-sided",
                                 method=WILCOXON_METHOD).pvalue)
            print(f"    Wilcoxon vs 0, two-sided   : p={_pc:.4g}")
            print("    verdict                    : " + (
                "SYSTEMATIC difference between identical-evidence conditions - the judge "
                "did not ignore the answer, so CTX is contaminated and must not be "
                "reported as a property of retrieval" if _pc < 0.05 else
                "no systematic difference; the instruction held"))
            print(f"    scatter on identical input : sd {_d.std(ddof=1):.3f}, "
                  f"95th pct |d| {np.percentile(np.abs(_d), 95):.2f}  "
                  "(descriptive; NOT a threshold)")
        MANIFEST["std_ctx_instrument_check"] = {
            "identical_evidence": bool(_same), "n_pairs": int(len(_d)),
            "n_nonzero": _nz, "mean_delta": float(_d.mean()),
            "sd_delta": float(_d.std(ddof=1)) if len(_d) > 1 else None,
            "wilcoxon_two_sided_p": _pc,
            "interpretation": ("validity check on the CTX instruction, not a noise floor; "
                               "a significant result means the answer leaked into a "
                               "retrieval-side metric")}

    print("\nInter-judge reliability, standard rubric vs the unanchored FA/CV/AC rubric")
    _al = {}
    for c in ("FTH", "ANR", "CTX"):
        u = [[STD[(j, b, q)][c] for j in JS] for b in BS for q in QS]
        _al[c] = krippendorff_alpha(u)
        print(f"  {c:<9} alpha(ordinal) = {_al[c]:.3f}")
    u = [[np.mean([STD[(j, b, q)][c] for c in ("FTH", "ANR")]) for j in JS]
         for b in BS for q in QS]
    _al["FTH+ANR"] = krippendorff_alpha(u)
    print(f"  {'FTH+ANR':<9} alpha(ordinal) = {_al['FTH+ANR']:.3f}")
    _old = MANIFEST.get("evaluation", {}).get("krippendorff_ordinal", {}).get("overall")
    if _old is not None:
        print(f"  {'FA/CV/AC':<9} alpha(ordinal) = {_old:.3f}   (Part 9, no anchors, no "
              "evidence shown)")

    # ---- the claim: the standard rubric cannot separate the auditing layer -------------
    print("\n" + "=" * 76)
    print("Can the STANDARD rubric separate the conditions? (Wilcoxon, one-sided, n=%d)"
          % len(QS))
    _sp, _sd = {}, {}
    for a, b in (("B5", "B3"), ("B5", "B4"), ("B3", "B2"), ("B2", "B1")):
        if a not in BS or b not in BS:
            continue
        for c in ("FTH", "ANR"):
            x = [float(np.mean([STD[(j, a, q)][c] for j in JS])) for q in QS]
            y = [float(np.mean([STD[(j, b, q)][c] for j in JS])) for q in QS]
            d = np.array(x) - np.array(y)
            if np.allclose(d, 0):
                print(f"  {a} vs {b} [{c}]: identical on every query"); continue
            p = float(wilcoxon(x, y, alternative="greater",
                               method=WILCOXON_METHOD).pvalue)
            _sp[f"{a}>{b}[{c}]"] = p
            _sd[f"{a}>{b}[{c}]"] = float(np.mean(d))
    # Eight tests in one family. Raw p-values here would inflate the false-positive rate,
    # and the paper draws conclusions from them, so Holm is applied and the CORRECTED
    # value is the one to quote.
    _holm, _prev = {}, 0.0
    for _n, (_k, _p) in enumerate(sorted(_sp.items(), key=lambda kv: kv[1])):
        _prev = max(_prev, min(1.0, _p * (len(_sp) - _n)))
        _holm[_k] = _prev
    print(f"  {'comparison':<20}{'delta':>9}{'raw p':>11}{'Holm p':>11}")
    for _k in sorted(_sp, key=lambda k: _sp[k]):
        print(f"  {_k:<20}{_sd[_k]:>+9.3f}{_sp[_k]:>11.4g}{_holm[_k]:>11.4g}"
              + ("  *" if _holm[_k] < 0.05 else ""))
    print("=" * 76)
    print("Read this against Part 9d: whatever the standard rubric does or does not show")
    print("for B5 vs B3, the uncertainty-reporting criterion separates them at p<0.01 and")
    print("the judge-free disclosure count at p<0.05. That gap between what the standard")
    print("rubric measures and what the auditing layer does is the paper's finding.")

    MANIFEST["evaluation_standard_rubric"] = {
        "rubric_sha": _STDV.split(":")[-1], "judges": JS, "conditions": BS,
        "n_queries": len(QS), "evidence_shown_to_judge": True,
        "means": {b: {c: sm(b, c) for c in ("FTH", "ANR", "CTX")} for b in BS},
        "krippendorff_ordinal": {k: float(v) for k, v in _al.items()},
        "wilcoxon_raw": _sp, "wilcoxon_holm": _holm, "wilcoxon_delta": _sd,
        "multiple_comparison": "Holm over the whole family printed above",
        "implementation": ("own anchored 1-5 LLM-judge rubric on the faithfulness / "
                           "answer-relevance / context-relevance dimensions; NOT the RAGAS "
                           "or ARES implementations, which score claim-support ratios"),
        "note": ("FA/CV/AC of Part 9 retained as the appendix comparison and the link to "
                 "the recorded study")}


standard-rubric judgments to make: 285 (3 judges x 5 conditions x 20 queries)
    25/285
    50/285
    75/285
    100/285
    125/285
    150/285
    175/285
    200/285
    225/285
    250/285
    275/285

scored over judges ['J1', 'J2', 'J3'] | complete conditions ['B1', 'B2', 'B3', 'B4', 'B5'] | 19 queries

  cells present, judge x condition (out of 19 queries each)
             B1     B2     B3     B4     B5
    J1       19     19     19     19     19
    J2       19     19     19     19     19
    J3       19     19     19     19     19
    reported: conditions ['B1', 'B2', 'B3', 'B4', 'B5'] over judges ['J1', 'J2', 'J3']; any condition not listed was dropped whole.

Anchored rubric on the standard RAG evaluation dimensions (faithfulness / answer
relevance / context relevance), evidence shown to the judge.
The DIMENSIONS are the ones RAGAS and ARES define; the 1-5 anchors and the scoring
are this notebook's own. No RAGAS or ARES implementation is called - say so in 3.5.
         

`cell 50`

## Part 10 — Case study C1: causal chain  *(§4.3.2)*

The query *"How does low dissolved oxygen (DO) affect wasabi rhizome yield?"* is the
worked example in §4.3.2. The cell below reconstructs the chain directly from the graph
built in Part 8 and prints the B1 and B5 answers actually given during the study together
with the scores the three judges awarded them, so the qualitative claim in the text is
traceable to the same artefacts as the quantitative table in Part 9.

In [52]:
# ====== cell 51 ======
# pipeline.traverse_causal_chain does a breadth-first walk with a global visited set, so
# its edge lists are not simple paths and cannot be printed as a chain. The search below
# enumerates simple paths over causal edges only, which is what §4.3.2 describes.
CAUSAL = defaultdict(list)
for e in kg.edges:
    if e.get("edge_type") == "causal":
        CAUSAL[e["source"]].append(e)

def simple_paths(start, max_depth=8, limit=20000):
    out, stack = [], [(start, [], {start})]
    while stack and len(out) < limit:
        node, path, seen = stack.pop()
        for e in CAUSAL.get(node, []):
            if e["target"] in seen:
                continue
            np_ = path + [e]
            out.append(np_)
            if len(np_) < max_depth:
                stack.append((e["target"], np_, seen | {e["target"]}))
    return out

do_nodes = [n for n, d in kg.nodes.items()
            if re.search(r"(^|_)(do|dissolved[_ ]oxygen)($|_)", n.lower())]
print("dissolved-oxygen nodes:", do_nodes)

chains = [p for n in do_nodes for p in simple_paths(n)]
chains.sort(key=len, reverse=True)

if chains:
    best = chains[0]
    print(f"\nlongest simple causal chain from a DO node: {len(best)} hops")
    lab = lambda n: kg.nodes.get(n, {}).get("label", n)
    print(f"    {lab(best[0]['source'])}")
    for e in best:
        print(f"      --{e['predicate']}-->  {lab(e['target'])}   <{e.get('source_id') or '-'}>")
    srcs = sorted({e.get("source_id") for e in best if e.get("source_id")})
    print(f"\n  source documents spanned : {len(srcs)}  {srcs}")
    print(f"  chains of >= 4 hops      : {sum(1 for c in chains if len(c) >= 4)}")
    print(f"  chains of >= 7 hops      : {sum(1 for c in chains if len(c) >= 7)}")
    print("\n  A chain of this length exists only because the graph carries curated causal "
          "edges;\n  relational edges alone cannot connect two different parameters.")
    MANIFEST["case_c1"] = {
        "max_hops": len(best), "sources_spanned": srcs,
        "path": [lab(best[0]["source"])] + [lab(e["target"]) for e in best],
        "n_chains_ge4": sum(1 for c in chains if len(c) >= 4),
        "n_chains_ge7": sum(1 for c in chains if len(c) >= 7)}
else:
    print("no causal chain from a dissolved-oxygen node in this graph")


dissolved-oxygen nodes: ['DO', 'DO_measured_value_S01', 'DO_critical_threshold_S01', 'DO_set_value_S01', 'DO_affects_S01', 'DO_tested_range_S04', 'DO_measured_value_S04', 'DO_set_value_S04', 'DO_optimal_range_S06', 'DO_measured_value_S06', 'DO_set_value_S15', 'DO_critical_threshold_S15', 'DO_measured_value_S15', 'DO_critical_threshold_S16', 'DO_measured_value_S16', 'DO_optimal_range_S19', 'DO_low', 'DO_depletion', 'low_DO', 'DO_high', 'DO_concentration', 'DO_zero', 'DO_increase', 'DO_absence', 'DO_decrease', 'dissolved_oxygen', 'dissolved_oxygen_decrease', 'DO_uneven_distribution']

longest simple causal chain from a DO node: 2 hops
    Low dissolved oxygen concentration
      --causes-->  Decrease in root water and nutrient uptake   <S01>
      --depresses-->  Growth of wasabi plantlets   <S01>

  source documents spanned : 1  ['S01']
  chains of >= 4 hops      : 0
  chains of >= 7 hops      : 0

  A chain of this length exists only because the graph carries curated causal edges;
  re

In [53]:
# ====== cell 52 ======
# The answers that were actually graded, with the scores each judge gave them.
if QUERIES and ANSWERS and JUDGE_ROWS:
    qi = next((i for i, q in enumerate(QUERIES)
               if "dissolved oxygen" in q.lower() and "rhizome" in q.lower()), None)
    if qi is None:
        print("the case-study query is not in the recorded query set")
    else:
        print("=" * 78)
        print(f"Q{qi+1}: {QUERIES[qi]}")
        print("=" * 78)
        for b in ("B1", "B5"):
            sc = {j: CELL[(j, b, qi + 1)] for j in JUDGES if (j, b, qi + 1) in CELL}
            tag = "   ".join(f"{j} FA={s['FA']} CV={s['CV']} AC={s['AC']}" for j, s in sc.items())
            ans = ANSWERS.get(b, {}).get(str(qi), "(not recorded)")
            print(f"\n--- {b} ---   {tag}")
            print(ans[:1400])

        # provenance check: does the recorded B5 answer contain the multi-hop chain that
        # the graph above supports? If not, the narrative example and the scored answer
        # come from different runs and the manuscript must say which it is quoting.
        b4 = ANSWERS.get("B5", {}).get(str(qi), "")
        traced = sum(1 for kw in ("root", "uptake", "photosynth", "rhizome", "respiration")
                     if kw in b4.lower())
        hedged = any(k in b4.lower() for k in
                     ("does not contain", "no direct information", "not available"))
        print("\n" + "-" * 78)
        print(f"recorded B5 answer: {len(b4)} chars | mechanism terms {traced}/5 | "
              f"declares missing evidence: {hedged}")
        if hedged:
            _hops = MANIFEST.get("kg", {}).get("longest_simple_causal_chain_hops", "?")
            print("NOTE: the graded B5 answer for this query reports that the graph held no\n"
                  "direct evidence. The chain shown above is supported by the graph as built\n"
                  "in Part 8, which includes the 374 curated causal edges; the longest simple\n"
                  f"causal chain that graph supports is {_hops} hops, not the seven the\n"
                  "manuscript quotes (seven came from counting revisits in a breadth-first\n"
                  "walk). If the manuscript quotes a chain-style B5 response for this query,\n"
                  "that text does not come from the scored run and must either be regenerated\n"
                  "and labelled as an illustration, or replaced by the graded answer.")


Q4: How does low dissolved oxygen (DO) affect wasabi rhizome yield?

--- B1 ---   J1 FA=4.0 CV=3.0 AC=2.0   J2 FA=3.0 CV=3.0 AC=1.0   J3 FA=2.0 CV=1.0 AC=5.0
The retrieved sources do not provide specific information on how low dissolved oxygen (DO) affects wasabi rhizome yield. The available evidence discusses the general importance of dissolved oxygen for root formation and growth in other plant species, such as Ficus and Chrysanthemum, but does not address wasabi specifically or its rhizome yield in relation to DO levels. Therefore, I cannot provide an answer to the question based on the retrieved evidence.

--- B5 ---   J1 FA=4.0 CV=4.0 AC=3.0   J2 FA=4.0 CV=4.0 AC=3.0   J3 FA=5.0 CV=4.0 AC=5.0
Low dissolved oxygen (DO) negatively impacts wasabi rhizome yield by affecting several critical growth processes. It causes a decrease in root water and nutrient uptake, which depresses the growth of wasabi plantlets and rhizomes. Low DO inhibits root cell division and elongation, reduces roo

In [54]:
# ====== cell 53 ======
# ---- §4.3.2 illustration: the answer B5 produces from the causal graph ----
# DECISION (§4.3.2): the recorded B5 answer for this query predates the causal edges and
# reports that the graph held no direct evidence, so it cannot be the chain quoted in the
# manuscript. The answer below is regenerated from the graph built in Part 8 and is an
# ILLUSTRATION ONLY: it was not seen by the judges and contributes nothing to Table 11.
# The manuscript must label it as such.
ILLUS_CACHE = OUT_DIR / "case_c1_illustration.json"
RUN_C1_ILLUSTRATION = True          # one API call, cached

illus = json.load(open(ILLUS_CACHE, encoding="utf-8")) if ILLUS_CACHE.exists() else {}
C1_QUERY = "How does low dissolved oxygen (DO) affect wasabi rhizome yield?"
# The stamp carries PROMPT_VERSION, so any change to retrieve() or to the B5 system
# prompt discards this cached illustration automatically. Without that, an illustration
# generated under the pre-fix retriever would survive the fix and be quoted as though it
# came from the graph as built here - the exact provenance error decision 7 is about.
RETRIEVAL_VERSION = f"confidence-then-chain-length|{PROMPT_VERSION}"
if illus.get("retrieval_version") != RETRIEVAL_VERSION:
    illus = {}

if RUN_C1_ILLUSTRATION and "answer" not in illus and HAS_OPENAI:
    from openai import OpenAI
    ev, _ = retrieve(C1_QUERY, multihop=True, with_confidence=True)
    r = OpenAI().chat.completions.create(
        model=MODEL_AUDIT, temperature=0.0,
        messages=[{"role": "system", "content": BASELINES["B5"]["sys"]},
                  {"role": "user", "content": f"Question: {C1_QUERY}\n\nEvidence:\n{ev}"}])
    illus = {"query": C1_QUERY, "evidence": ev,
             "answer": r.choices[0].message.content.strip(),
             "model": MODEL_AUDIT, "scored": False,
             "retrieval_version": RETRIEVAL_VERSION,
             "note": "regenerated for illustration; not part of the Table 11 evaluation"}
    json.dump(illus, open(ILLUS_CACHE, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

if illus.get("answer"):
    print("=" * 78)
    print("ILLUSTRATION - regenerated from the Part 8 graph. NOT scored, NOT in Table 11.")
    print("=" * 78)
    print(f"Query: {illus['query']}\n")
    print(illus["answer"][:1800])
    ev_lines = [l for l in illus["evidence"].splitlines() if l.startswith("- ")]
    hops = Counter(l.count("--") for l in ev_lines)
    lvls = Counter(re.findall(r"\[(Verified|Disputed|Conflicted)\]", illus["evidence"]))
    print(f"\n[model {illus['model']}, B5 retrieval, {len(ev_lines)} paths]")
    print(f"  hops per retrieved path : {dict(sorted(hops.items()))}")
    print(f"  path confidence levels  : {dict(lvls)}")
    longest = max((l.count('--') for l in ev_lines), default=0)
    print(f"  longest retrieved chain : {longest} hops")
    if not (set(lvls) - {"Verified"}):
        print("\n  NOTE: every retrieved path is Verified, so B5's uncertainty warning never")
        print("  fires on this query. C1 therefore illustrates multi-hop retrieval, not the")
        print("  confidence layer. A query that exercises the warning is identified below.")
    MANIFEST["case_c1_illustration"] = {
        **{k: illus[k] for k in ("query", "model", "scored", "note")},
        "n_paths": len(ev_lines), "hops": dict(hops), "levels": dict(lvls),
        "longest_retrieved_hops": longest,
        "retrieval_version": illus["retrieval_version"]}

    # find a query where the confidence layer actually has something to warn about
    if QUERIES:
        cand = []
        for q in QUERIES:
            _, lv = retrieve(q, multihop=True, with_confidence=True)
            bad = sum(1 for l in lv if l != "Verified")
            if bad:
                cand.append((bad, q))
        cand.sort(reverse=True)
        if cand:
            print(f"\n  queries whose B5 subgraph contains non-Verified paths: {len(cand)}")
            for n_, q in cand[:3]:
                print(f"    {n_:>3} non-Verified path(s) | {q}")
            print("  Use one of these if §4.3.2 is to illustrate the uncertainty warning.")
            MANIFEST["case_c1_illustration"]["queries_with_uncertainty"] = \
                [{"n_non_verified": n_, "query": q} for n_, q in cand[:5]]
        else:
            print("\n  No query in the evaluation set retrieves a non-Verified path, so the")
            print("  uncertainty warning cannot be illustrated from this query set at all.")
else:
    print("RUN_C1_ILLUSTRATION requires OPENAI_API_KEY; the recorded answers above stand.")


ILLUSTRATION - regenerated from the Part 8 graph. NOT scored, NOT in Table 11.
Query: How does low dissolved oxygen (DO) affect wasabi rhizome yield?

Low dissolved oxygen (DO) affects wasabi rhizome yield by causing a decrease in root water and nutrient uptake, which in turn depresses the growth of wasabi plantlets. This reduction in growth is due to inhibited root cell division and elongation, leading to decreased root formation and growth. Additionally, low DO increases the time required to form adventitious roots, reduces rooting percentages, and decreases the number and average length of roots formed per cutting. These factors collectively impair rhizome growth. Furthermore, low DO can cause physiological disorders and weaken disease resistance in wasabi plants. While the critical threshold for DO is conflicted in the literature, with values ranging from 5 mg L−1 to 9.5 ppm, the negative impact of low DO on wasabi growth is well-verified.

[model gpt-4o, B5 retrieval, 25 paths]
  

`cell 54`

## Part 11 — Generalization: sparse-subset stability on a data-rich crop  *(§4.4)*

The wasabi study rests on 19 documents, which raises the obvious question of whether
knowledge synthesised from so few sources is representative or an artefact of a small
sample. Wasabi has no larger corpus to check against, so the question is answered on
strawberry, where 76 full-text papers were available — deliberately 4 × 19, so the ladder
consists of clean multiples of the wasabi scale.

The identical extraction and aggregation path is used: `metrics.derive_param_values`
collapses a corpus into one value per parameter by confidence-weighted median, after the
unit-compatibility guard (the T2 rule applied at aggregation time) and an outlier trim.
Corpus sizes 19, 38 and 57 are each sampled 100 times at random; size 76 is the full corpus.

**What the experiment does and does not claim.** The full 76-paper corpus is the *internal*
reference. Convergence means a sparse subset lands where the full corpus lands — not that
either is the true agronomic optimum. Nothing here is scored against an external ground
truth, which is what keeps the claim to sampling representativeness.

Four robustness layers accompany the headline result, each answering a predictable
reviewer question:

1. **Verification against the published Table 12**, cell by cell with an explicit tolerance.
2. **Normalizer robustness** — the same convergence measured against the plausible range,
   the interquartile range, the standard deviation and the observed range, so the decline
   is not an artefact of the denominator.
3. **Draw-count robustness** — the first 25 of the same 100 draws against all 100, nested
   rather than independently resampled.
4. **Estimator sensitivity** — weighted median, plain median, mean and 10% trimmed mean,
   with DLI reported separately because §4.4.3 identifies it as a derived, right-skewed
   quantity whose residual is largely a property of the estimator.

The sampling plan is generated once from the fixed seed and stored, so every table below
reads the same draws. A closed-book control — the same questions with no documents —
separates what the pipeline extracted from what the model already knew.

In [56]:
# ====== cell 55 ======
BERRY_OK = False
if BERRY_DIR:
    sys.path.insert(0, str(BERRY_DIR))
    try:
        from strawberry_config import STRAWBERRY_NORMAL_RANGE as NR, STRAWBERRY_REFERENCE as REF
        from pipeline_hf import audit_corpus
        from metrics import (derive_param_values, make_param_key, unit_compatible,
                             weighted_median, CONF_WEIGHT)
        BERRY_OK = True
    except Exception as e:
        print("strawberry modules not importable:", str(e)[:160])

N_DRAWS = 100                       # the paper reports 100 random draws per size
SUBSET_SIZES = [19, 38, 57]
PARAMS = ["water_temp", "air_temp", "DO", "EC", "pH", "PPFD", "photoperiod", "DLI"]

if BERRY_OK:
    berry_cache = json.load(open(BERRY_DIR / "extractions.json", encoding="utf-8"))
    SIDS = sorted(berry_cache)
    n_trip = sum(len(v) for v in berry_cache.values())
    print(f"strawberry corpus : {len(SIDS)} papers, {n_trip} triplets")
    print(f"cache sha256      : {sha256(BERRY_DIR / 'extractions.json')[:16]}")
    print(f"confidence weights: {dict(CONF_WEIGHT)}")

    # Aggregation reuses metrics.py exactly: same parameter mapping, same unit-compatibility
    # guard (the T2 rule applied at aggregation time), same outlier trim. Only the estimator
    # is swapped, and only in the sensitivity analysis below.
    _pk = make_param_key(NR)

    def buckets(source_ids):
        """Values and weights per parameter for one subset, filtered as metrics.py does."""
        trip = [t for sid in source_ids for t in berry_cache[sid]]
        b = {p: ([], []) for p in NR}
        for t in audit_corpus(trip, do_audit=False):
            p = _pk(t.get("subject", ""))
            if p is None or not t.get("values"):
                continue
            if not unit_compatible(p, t.get("unit", "")):
                continue
            lo, hi, _ = NR[p]
            w = CONF_WEIGHT.get(t.get("confidence", "Verified"), 1.0)
            for v in t["values"]:
                if lo - 0.5 * (hi - lo) <= v <= hi + 0.5 * (hi - lo):
                    b[p][0].append(v); b[p][1].append(w)
        return b

    # §4.4.2 names a "trimmed mean" among the four aggregators but does not state the
    # trimming proportion, and metrics.py implements only the weighted median. The value
    # is therefore fixed here and recorded in the manifest so the figure is reproducible.
    TRIM_PROP = 0.10

    def trimmed_mean(x, prop=TRIM_PROP):
        x = np.sort(np.asarray(x, float))
        k = int(np.floor(len(x) * prop))
        return float(np.mean(x if 2 * k >= len(x) or len(x) < 3 else x[k:len(x) - k]))

    def estimate(b, method="weighted_median"):
        out = {}
        for p, (v, w) in b.items():
            if not v:
                out[p] = None; continue
            out[p] = {"weighted_median": lambda: weighted_median(v, w),
                      "median":          lambda: float(np.median(v)),
                      "mean":            lambda: float(np.mean(v)),
                      "trimmed_mean":    lambda: trimmed_mean(v)}[method]()
        return out

    # the swapped-estimator path must agree with metrics.derive_param_values on the default
    full_b = buckets(SIDS)
    chk = derive_param_values(audit_corpus([t for s in SIDS for t in berry_cache[s]],
                                           do_audit=False), NR)
    FULL = estimate(full_b, "weighted_median")
    same = all(abs((FULL.get(p) or 0) - (chk.get(p) or 0)) < 1e-9 for p in chk)
    print(f"\nagrees with metrics.derive_param_values: {same}")

    print(f"\n{'parameter':<14}{'full corpus':>13}{'literature reference':>24}")
    for p in PARAMS:
        v = FULL.get(p)
        print(f"  {p:<12}{(f'{v:.2f}' if v is not None else '-'):>13}{str(REF.get(p)):>24}")
    print("\nThe full 76-paper corpus is the INTERNAL reference for convergence, not an")
    print("external ground truth. The literature column is context only; nothing is scored")
    print("against it, which is what keeps the claim to sparse-sampling representativeness.")
    MANIFEST["strawberry"] = {"n_papers": len(SIDS), "n_triplets": n_trip,
                              "cache_sha256": sha256(BERRY_DIR / "extractions.json"),
                              "parameters": PARAMS, "confidence_weights": dict(CONF_WEIGHT),
                              "full_corpus": {p: FULL.get(p) for p in PARAMS},
                              "agrees_with_metrics_py": bool(same)}
else:
    print("Part 11 skipped - set STRAWBERRY_DIR to the folder with extractions.json "
          "and the strawberry modules")


strawberry corpus : 76 papers, 2700 triplets
cache sha256      : 05126b5f080d10fd
confidence weights: {'Verified': 1.0, 'Disputed': 0.5, 'Conflicted': 0.25}

agrees with metrics.derive_param_values: True

parameter       full corpus    literature reference
  water_temp          20.00                    20.0
  air_temp            20.70                    20.0
  DO                   5.99                     6.0
  EC                   1.50                     1.2
  pH                   5.90                     6.0
  PPFD               300.00                   300.0
  photoperiod         14.00                    16.0
  DLI                 13.00                    20.0

The full 76-paper corpus is the INTERNAL reference for convergence, not an
external ground truth. The literature column is context only; nothing is scored
against it, which is what keeps the claim to sparse-sampling representativeness.


In [57]:
# ====== cell 56 ======
# ---------- one fixed draw plan, generated once and reused by every analysis ----------
if BERRY_OK:
    # random.Random(SEED).sample is the sampler used for the published run; keeping it
    # (rather than a numpy Generator) is what makes the tables below match the paper.
    plan_rng = random.Random(SEED)
    DRAWS = {k: [plan_rng.sample(SIDS, k) for _ in range(N_DRAWS)] for k in SUBSET_SIZES}
    DRAWS[len(SIDS)] = [SIDS]
    print("draw plan:", {k: len(v) for k, v in DRAWS.items()}, f"| seed {SEED}")

    # Buckets are computed once per draw and every estimator and normalizer below reads
    # them, so no later analysis silently resamples a different set of papers.
    BUCKETS = {k: [buckets(ids) for ids in v] for k, v in DRAWS.items()}
    EST = {m: {k: [estimate(b, m) for b in BUCKETS[k]] for k in BUCKETS}
           for m in ("weighted_median", "median", "mean", "trimmed_mean")}
    print("estimators computed:", list(EST))

    def col(method, k, p):
        return np.array([d[p] for d in EST[method][k] if d.get(p) is not None], float)

    print(f"\n{'parameter':<14}" + "".join(f"{('n='+str(k)):>16}" for k in SUBSET_SIZES)
          + f"{('full n='+str(len(SIDS))):>14}")
    TABLE12 = {}
    for p in PARAMS:
        row, TABLE12[p] = f"  {p:<12}", {}
        for k in SUBSET_SIZES:
            x = col("weighted_median", k, p)
            m, s = (float(x.mean()), float(x.std(ddof=1))) if len(x) > 1 else (np.nan, 0.0)
            TABLE12[p][k] = (m, s)
            row += f"{f'{m:.1f} +- {s:.1f}':>16}"
        fv = FULL.get(p)
        print(row + f"{(f'{fv:.1f}' if fv is not None else '-'):>14}")
    MANIFEST["strawberry_table"] = {p: {str(k): list(v) for k, v in d.items()}
                                    for p, d in TABLE12.items()}


draw plan: {19: 100, 38: 100, 57: 100, 76: 1} | seed 42
estimators computed: ['weighted_median', 'median', 'mean', 'trimmed_mean']

parameter                 n=19            n=38            n=57     full n=76
  water_temp       19.8 +- 0.8     20.0 +- 0.2     20.0 +- 0.0          20.0
  air_temp         20.1 +- 1.9     20.3 +- 0.8     20.6 +- 0.5          20.7
  DO                5.6 +- 0.5      5.7 +- 0.4      5.8 +- 0.4           6.0
  EC                1.5 +- 0.2      1.5 +- 0.1      1.5 +- 0.0           1.5
  pH                5.9 +- 0.3      5.9 +- 0.1      5.9 +- 0.1           5.9
  PPFD           271.6 +- 35.3   286.6 +- 20.7   289.1 +- 18.8         300.0
  photoperiod      14.2 +- 1.8     14.2 +- 1.5     14.2 +- 1.2          14.0
  DLI              15.8 +- 3.8     14.9 +- 3.5     14.3 +- 3.0          13.0


In [58]:
# ====== cell 57 ======
# ---------- convergence, representativeness, draw and estimator robustness ----------
if BERRY_OK:
    # §4.4.2 names THREE normalizers: the parameter's plausible range, its interquartile
    # range and its standard deviation. Those three are the paper-reported robustness
    # analysis. Observed range is an extra diagnostic and is reported separately so it is
    # not mistaken for one of the three.
    pooled = {p: np.array(full_b[p][0], float) for p in PARAMS if full_b[p][0]}
    PAPER_SCALES = {
        "plausible range (primary)": {p: NR[p][1] - NR[p][0] for p in PARAMS if p in NR},
        "interquartile range":       {p: float(np.percentile(v, 75) - np.percentile(v, 25))
                                      for p, v in pooled.items()},
        "standard deviation":        {p: float(v.std(ddof=1)) for p, v in pooled.items() if len(v) > 1},
    }
    EXTRA_SCALES = {
        "observed range (exploratory)": {p: float(v.max() - v.min()) for p, v in pooled.items()},
    }

    def mean_norm_error(k, scale, method="weighted_median"):
        errs = []
        for p in PARAMS:
            s = scale.get(p)
            if not s or not np.isfinite(s) or FULL.get(p) is None:
                continue
            x = col(method, k, p)
            if len(x):
                errs.append(abs(x.mean() - FULL[p]) / s)
        return float(np.mean(errs)) if errs else np.nan

    print("§4.4.2 robustness: mean normalized error under the three reported normalizers")
    print(f"{'normalizer':<28}" + "".join(f"{('n='+str(k)):>10}" for k in SUBSET_SIZES)
          + "   monotone")
    norm_tab = {}
    for name, sc in PAPER_SCALES.items():
        vals = [mean_norm_error(k, sc) for k in SUBSET_SIZES]
        norm_tab[name] = vals
        print(f"  {name:<26}" + "".join(f"{v:>10.3f}" for v in vals)
              + f"   {'yes' if all(a >= b for a, b in zip(vals, vals[1:])) else 'NO'}")
    print("\n  additional diagnostic, not one of the three reported normalizers:")
    for name, sc in EXTRA_SCALES.items():
        vals = [mean_norm_error(k, sc) for k in SUBSET_SIZES]
        norm_tab[name] = vals
        print(f"  {name:<26}" + "".join(f"{v:>10.3f}" for v in vals)
              + f"   {'yes' if all(a >= b for a, b in zip(vals, vals[1:])) else 'NO'}")
    # The paper's "approximately 2-3x larger under data-driven spreads" refers to the two
    # data-driven normalizers it names -- interquartile range and standard deviation.
    # Observed range is an extra diagnostic and is excluded from this comparison.
    base = norm_tab["plausible range (primary)"][0]
    ratios = {n: norm_tab[n][0] / base
              for n in ("interquartile range", "standard deviation")}
    print(f"\n  at k=19 the data-driven normalizers give "
          + ", ".join(f"{n} {r:.1f}x" for n, r in ratios.items())
          + f"\n  the plausible-range value, matching the paper's 'approximately 2-3x "
          "larger under data-driven spreads'.")
    print(f"  (observed range {norm_tab['observed range (exploratory)'][0]/base:.1f}x is "
          "excluded: it is not one of the three reported normalizers.)")

    # §4.4.2 sufficiency criterion: "deviations remaining below one data-standard-deviation
    # (<= 0.1 SD at k = 57)". Computed per parameter, not only as a mean.
    print("\nsufficiency criterion: |subset mean - full corpus| as a fraction of the data s.d.")
    print(f"{'parameter':<14}" + "".join(f"{('n='+str(k)):>10}" for k in SUBSET_SIZES)
          + f"{'data s.d.':>12}")
    sd_ratio = {}
    for p in PARAMS:
        sd = PAPER_SCALES["standard deviation"].get(p)
        if not sd or FULL.get(p) is None:
            continue
        row = []
        for k in SUBSET_SIZES:
            x = col("weighted_median", k, p)
            row.append(abs(x.mean() - FULL[p]) / sd if len(x) else np.nan)
        sd_ratio[p] = row
        print(f"  {p:<12}" + "".join(f"{v:>10.3f}" for v in row) + f"{sd:>12.2f}")
    at57 = {p: v[-1] for p, v in sd_ratio.items()}
    worst_p = max(at57, key=at57.get)
    n_01 = sum(1 for v in at57.values() if v <= 0.10)
    n_1  = sum(1 for v in at57.values() if v <= 1.0)
    print(f"\n  max at k=57 : {at57[worst_p]:.3f} SD ({worst_p})")
    print(f"  <= 0.10 SD  : {n_01}/{len(at57)} parameters   |  "
          f"<= 1.00 SD : {n_1}/{len(at57)}")
    claim_ok = n_01 == len(at57)
    print(f"\n  VERDICT on §4.4.2's '<= 0.1 SD at k = 57': "
          f"{'holds' if claim_ok else 'DOES NOT HOLD as an upper bound for all parameters'}")
    if not claim_ok:
        over = {p: v for p, v in at57.items() if v > 0.10}
        print(f"    above 0.1 SD: " + ", ".join(f"{p} {v:.2f}" for p, v in
                                                sorted(over.items(), key=lambda kv: -kv[1])))
        print(f"    accurate wording: \"all eight parameters remain below one data standard\n"
              f"    deviation at k = 57 (maximum {at57[worst_p]:.2f} SD, {n_01} of "
              f"{len(at57)} at or below 0.1 SD)\"")
    SD_CLAIM = {"max_at_57": at57[worst_p], "argmax": worst_p,
                "n_le_0_1": n_01, "n_le_1_0": n_1, "n_params": len(at57),
                "paper_claim_holds": bool(claim_ok)}

    # §4.4.2 representativeness. The paper says the full-corpus value lay "within the
    # central region ... (mean |z| = 0.40)". "Central region" is not defined numerically in
    # the manuscript, so three concrete readings are reported and one must be adopted.
    print("\nrepresentativeness of a 19-paper subset")
    print(f"{'parameter':<13}{'full':>8}{'mean':>8}{'s.d.':>7}{'|z|':>7}"
          f"{'|z|<1':>7}{'central 80%':>13}{'central 95%':>13}")
    rep = {}
    for p in PARAMS:
        x, f = col("weighted_median", 19, p), FULL.get(p)
        if f is None or len(x) < 2:
            continue
        m, s = float(x.mean()), float(x.std(ddof=1))
        z = (f - m) / s if s > 0 else np.nan
        lo80, hi80 = np.percentile(x, [10, 90])
        lo95, hi95 = np.percentile(x, [2.5, 97.5])
        rep[p] = {"full": f, "mean": m, "sd": s, "z": float(z), "abs_z": float(abs(z)),
                  "inside_z1": bool(abs(z) < 1), "c80": [float(lo80), float(hi80)],
                  "inside_80": bool(lo80 <= f <= hi80),
                  "c95": [float(lo95), float(hi95)], "inside_95": bool(lo95 <= f <= hi95)}
        r = rep[p]
        print(f"  {p:<11}{f:>8.2f}{m:>8.2f}{s:>7.2f}{abs(z):>7.2f}"
              f"{('yes' if r['inside_z1'] else 'NO'):>7}"
              f"{('yes' if r['inside_80'] else 'NO'):>13}"
              f"{('yes' if r['inside_95'] else 'NO'):>13}")
    mz = float(np.mean([v["abs_z"] for v in rep.values()]))
    mx = max(rep, key=lambda p_: rep[p_]["abs_z"])
    print(f"\n  REPORTED STATISTICS (§4.4.2 reports the first of these)")
    print(f"    mean |z|               : {mz:.2f}")
    print(f"    max  |z|               : {rep[mx]['abs_z']:.2f}  ({mx})")
    print(f"\n  DIAGNOSTICS - interval coverage, none of which the manuscript specifies")
    print(f"    within |z| < 1         : {sum(1 for v in rep.values() if v['inside_z1'])}/{len(rep)}")
    print(f"    within the central 80% : {sum(1 for v in rep.values() if v['inside_80'])}/{len(rep)}")
    print(f"    within the empirical 95%: {sum(1 for v in rep.values() if v['inside_95'])}/{len(rep)}")
    print("\n  §4.4.2 says the full-corpus value lay 'within the central region' but never")
    print("  defines that region, so no interval criterion was pre-specified and this notebook")
    print("  does not invent one. The manuscript should report the |z| statistics above -")
    print(f"  mean |z| = {mz:.2f}, max |z| = {rep[mx]['abs_z']:.2f}, i.e. every parameter lies "
          "within one subset standard\n  deviation of the subset mean - rather than assert "
          "coverage of an undefined interval.")
    print("  The three coverage rows are empirical intervals over subset draws, not confidence")
    print("  intervals for a population mean, and are diagnostics only.")

    print("\ndraw-count robustness: the first 25 draws against all 100 (nested, same plan)")
    print(f"{'parameter':<14}{'mean 25':>10}{'mean 100':>10}{'d mean':>9}"
          f"{'sd 25':>9}{'sd 100':>9}{'d sd':>8}")
    rob = {}
    for p in PARAMS:
        x = col("weighted_median", 19, p)
        if len(x) < 26:
            continue
        a = x[:25]
        rob[p] = {"mean25": float(a.mean()), "mean100": float(x.mean()),
                  "sd25": float(a.std(ddof=1)), "sd100": float(x.std(ddof=1))}
        print(f"  {p:<12}{a.mean():>10.2f}{x.mean():>10.2f}{abs(a.mean()-x.mean()):>9.3f}"
              f"{a.std(ddof=1):>9.2f}{x.std(ddof=1):>9.2f}"
              f"{abs(a.std(ddof=1)-x.std(ddof=1)):>8.3f}")

    print(f"\n§4.4.2 robustness: mean normalized error under the four reported aggregators")
    print(f"  (trimming proportion for the trimmed mean = {TRIM_PROP:.0%}; the manuscript "
          f"does not state one,\n   so it is fixed here and recorded in the manifest)")
    print(f"{'aggregator':<20}" + "".join(f"{('n='+str(k)):>10}" for k in SUBSET_SIZES))
    est_tab = {}
    for m in ("weighted_median", "median", "mean", "trimmed_mean"):
        vals = [mean_norm_error(k, PAPER_SCALES["plausible range (primary)"], m)
                for k in SUBSET_SIZES]
        est_tab[m] = vals
        print(f"  {m:<18}" + "".join(f"{v:>10.3f}" for v in vals))

    print("\nDLI alone, by aggregator (§4.4.3: a derived, right-skewed quantity)")
    print(f"{'aggregator':<20}{'full':>9}" + "".join(f"{('n='+str(k)):>10}" for k in SUBSET_SIZES)
          + f"{'|gap| at 19':>13}")
    dli, span = {}, NR["DLI"][1] - NR["DLI"][0]
    for m in ("weighted_median", "median", "mean", "trimmed_mean"):
        fv = estimate(full_b, m)["DLI"]
        row = [float(col(m, k, "DLI").mean()) for k in SUBSET_SIZES]
        gap = abs(row[0] - fv) / span
        dli[m] = {"full": fv, "by_size": row, "norm_gap_at_19": gap}
        print(f"  {m:<18}{fv:>9.1f}" + "".join(f"{v:>10.1f}" for v in row) + f"{gap:>13.3f}")
    g_med, g_mean = dli["weighted_median"]["norm_gap_at_19"], dli["mean"]["norm_gap_at_19"]
    print(f"  median-family gap {g_med:.3f} vs mean {g_mean:.3f} "
          f"({'more than halves' if g_mean < g_med / 2 else 'does not halve'}), "
          f"which is the\n  §4.4.3 claim stated quantitatively.")

    MANIFEST["strawberry_convergence"] = {
        "n_draws": N_DRAWS, "sizes": SUBSET_SIZES, "trimmed_mean_proportion": TRIM_PROP,
        "normalizers_reported_in_paper": list(PAPER_SCALES),
        "mean_normalized_error_by_normalizer": norm_tab,
        "headline_error": dict(zip(map(str, SUBSET_SIZES),
                                   norm_tab["plausible range (primary)"])),
        "deviation_over_data_sd": sd_ratio,
        "sd_sufficiency_claim": SD_CLAIM,
        "representativeness": rep, "mean_abs_z": mz,
        "max_abs_z": {"parameter": mx, "value": rep[mx]["abs_z"]},
        "central_region_definition": None,
        "coverage": {"z_lt_1": f"{sum(1 for v in rep.values() if v['inside_z1'])}/{len(rep)}",
                     "central_80": f"{sum(1 for v in rep.values() if v['inside_80'])}/{len(rep)}",
                     "empirical_95": f"{sum(1 for v in rep.values() if v['inside_95'])}/{len(rep)}"},
        "draw_robustness_25_vs_100": rob,
        "aggregator_sensitivity": est_tab, "dli_by_aggregator": dli}


§4.4.2 robustness: mean normalized error under the three reported normalizers
normalizer                        n=19      n=38      n=57   monotone
  plausible range (primary)      0.077     0.046     0.032   yes
  interquartile range            0.167     0.094     0.062   yes
  standard deviation             0.220     0.130     0.093   yes

  additional diagnostic, not one of the three reported normalizers:
  observed range (exploratory)     0.080     0.049     0.036   yes

  at k=19 the data-driven normalizers give interquartile range 2.2x, standard deviation 2.9x
  the plausible-range value, matching the paper's 'approximately 2-3x larger under data-driven spreads'.
  (observed range 1.0x is excluded: it is not one of the three reported normalizers.)

sufficiency criterion: |subset mean - full corpus| as a fraction of the data s.d.
parameter           n=19      n=38      n=57   data s.d.
  water_temp       0.085     0.005     0.000        2.07
  air_temp         0.137     0.086     

In [59]:
# ====== cell 58 ======
# ---------------------- closed-book prior-knowledge control ----------------------
# The convergence analysis is scored against the INTERNAL reference (the full corpus).
# This control is the one place an external literature value is used, and only as a
# common yardstick for comparing two estimators - the pipeline and the model's prior
# knowledge - not to validate either of them.
if BERRY_OK:
    RUN_CLOSED_BOOK = True          # 8 calls, cached after the first run
    CB_CACHE = OUT_DIR / "closed_book.json"
    QMAP = {"water_temp": "optimal root-zone/water temperature (C)",
            "air_temp": "optimal air temperature (C)", "DO": "optimal dissolved oxygen (mg/L)",
            "EC": "optimal nutrient solution EC (dS/m)", "pH": "optimal nutrient pH",
            "PPFD": "optimal PPFD (umol/m2/s)", "photoperiod": "optimal photoperiod (h)",
            "DLI": "optimal DLI (mol/m2/d)"}
    cb = json.load(open(CB_CACHE, encoding="utf-8")) if CB_CACHE.exists() else {}
    if RUN_CLOSED_BOOK and HAS_OPENAI:
        from openai import OpenAI
        cli = OpenAI()
        for p, q in QMAP.items():
            if p in cb:
                continue
            r = cli.chat.completions.create(
                model=MODEL_AUDIT, temperature=0.0, max_tokens=60,
                messages=[{"role": "user", "content":
                           f"What is the {q} for hydroponic strawberry? Give one number."}])
            m = re.findall(r"\d+\.?\d*", r.choices[0].message.content or "")
            cb[p] = float(m[0]) if m else None
        json.dump(cb, open(CB_CACHE, "w", encoding="utf-8"), ensure_ascii=False)

    if cb:
        span = lambda p: NR[p][1] - NR[p][0]
        print(f"\n{'parameter':<14}{'closed-book':>13}{'pipeline':>11}{'reference':>12}"
              f"{'|cb-ref|/span':>15}{'|pipe-ref|/span':>17}")
        for p in QMAP:
            fv, rv, cv = FULL.get(p), REF.get(p), cb.get(p)
            d_cb = abs(cv - rv) / span(p) if cv is not None and rv is not None else None
            d_pi = abs(fv - rv) / span(p) if fv is not None and rv is not None else None
            print(f"  {p:<12}{str(cv):>13}{(f'{fv:.2f}' if fv else '-'):>11}{str(rv):>12}"
                  f"{(f'{d_cb:.3f}' if d_cb is not None else '-'):>15}"
                  f"{(f'{d_pi:.3f}' if d_pi is not None else '-'):>17}")
        print("\n  Where the pipeline estimate tracks the literature reference and the "
              "closed-book answer\n  does not, the value came from the documents rather than "
              "from the model's prior knowledge.")
        MANIFEST["strawberry_closed_book"] = cb
    else:
        print("\nRUN_CLOSED_BOOK = False (skipped)")



parameter       closed-book   pipeline   reference  |cb-ref|/span  |pipe-ref|/span
  water_temp           20.0      20.00        20.0          0.000            0.000
  air_temp             24.0      20.70        20.0          0.400            0.070
  DO                    8.0       5.99         6.0          0.667            0.003
  EC                    1.4       1.50         1.2          0.222            0.333
  pH                    5.8       5.90         6.0          0.200            0.100
  PPFD                250.0     300.00       300.0          0.125            0.000
  photoperiod          16.0      14.00        16.0          0.000            0.500
  DLI                  17.0      13.00        20.0          0.214            0.500

  Where the pipeline estimate tracks the literature reference and the closed-book answer
  does not, the value came from the documents rather than from the model's prior knowledge.


`cell 59`

## Part 12 — Figures and tables

Every figure and table is generated from this run and written to `repro_out/figures/`
(PNG at 300 dpi and PDF) and `repro_out/tables/` (CSV). Nothing here reads, patches or
compares against previously published artwork — the manuscript's figures and tables are
to be replaced by these.

Colour follows a validated scheme: three categorical hues that stay separable under
colour-vision deficiency, and a reserved status palette for the Confidence Levels. Every
Confidence Level is also written out as text, so the level is never carried by colour
alone. Figures use no dual axes, label values directly, and keep grid and axes recessive.

In [67]:
# ====== cell 60 ======
# =============== Figures ===============
# All figures are generated from this run and written to repro_out/figures/ as PNG (300 dpi)
# and PDF. Nothing here reads or reproduces a previously published figure.
#
# Colour: categorical slots are a validated three-hue set (all-pairs CVD safe); Confidence
# Levels use a reserved status palette and ALWAYS carry a text label, so the level is never
# encoded by colour alone.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

FIG_DIR = OUT_DIR / f"figures{OUT_TAG}"; FIG_DIR.mkdir(exist_ok=True)
C1_, C2_, C3_ = "#2a78d6", "#eb6834", "#1baf7a"          # categorical, fixed order
LVL_COLOR = {"Verified": "#0ca30c", "Disputed": "#fab219", "Conflicted": "#d03b3b"}
INK, INK2, GRID = "#0b0b0b", "#52514e", "#dedcd7"

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "axes.edgecolor": GRID, "axes.linewidth": 0.8, "axes.labelcolor": INK2,
    "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
    "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": GRID, "grid.linewidth": 0.6, "legend.frameon": False,
})

def finish(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{name}.{ext}")
    plt.close(fig)
    print(f"  {name}.png / .pdf")

def barlabels(ax, bars, fmt="{:.0f}", dy=0):
    for b in bars:
        h = b.get_height()
        ax.annotate(fmt.format(h), (b.get_x() + b.get_width() / 2, h),
                    textcoords="offset points", xytext=(0, 2 + dy),
                    ha="center", fontsize=8, color=INK)

def _draw_f1_f5():
    print("writing figures to", FIG_DIR)

    # --- F1  two-stage funnel: what each stage removes -------------------------------
    fig, ax = plt.subplots(figsize=(5.6, 3.0))
    stages = ["corpus\ntriplets", "flagged by\nthe rule stage", "confirmed by\nthe auditor"]
    vals = [len(TRIPLETS), sum(1 for f in STEP2.values() if f),
            D.get("Conflicted", 0) + D.get("Disputed", 0)]
    bars = ax.bar(stages, vals, color=[C1_, C2_, C3_], width=0.55)
    barlabels(ax, bars)
    for b, v in zip(bars, vals):
        share, xc = f"{v/len(TRIPLETS):.1%}", b.get_x() + b.get_width() / 2
        if v > 0.22 * len(TRIPLETS):
            ax.annotate(share, (xc, v / 2), ha="center", va="center",
                        fontsize=8, color="white")
        else:
            ax.annotate(share, (xc, v), textcoords="offset points", xytext=(0, 14),
                        ha="center", fontsize=8, color=INK2)
    ax.set_ylabel("triplets"); ax.set_ylim(0, len(TRIPLETS) * 1.12)
    ax.set_title("Two-stage self-auditing narrows the corpus to confirmed conflicts", loc="left")
    ax.grid(axis="y"); ax.set_axisbelow(True)
    finish(fig, "F1_pipeline_funnel")

    # --- F2  detection and confirmation by conflict type ------------------------------
    types = ["T1", "T2", "T3_candidate"]
    raised = [by_type.get(t, 0) for t in types]
    confd  = [sum(1 for r in ROWS if t in r["confirmed_types"]) for t in types]
    x = np.arange(len(types)); w = 0.38
    fig, ax = plt.subplots(figsize=(5.6, 3.0))
    b1 = ax.bar(x - w/2, raised, w, label="raised by the rule stage", color=C1_)
    b2 = ax.bar(x + w/2, confd,  w, label="confirmed by the auditor", color=C2_)
    barlabels(ax, b1); barlabels(ax, b2)
    for i, (r_, c_) in enumerate(zip(raised, confd)):
        ax.annotate(f"{c_/r_:.0%}" if r_ else "-", (i + w/2, c_), textcoords="offset points",
                    xytext=(0, 14), ha="center", fontsize=8, color=INK2)
    ax.set_xticks(x); ax.set_xticklabels(["T1\nquantitative", "T2\nunit", "T3\nconditional"])
    ax.set_ylabel("flags"); ax.legend(loc="upper right")
    ax.set_title("Confirmation rate falls from T1 to T3", loc="left")
    ax.grid(axis="y"); ax.set_axisbelow(True)
    finish(fig, "F2_detection_by_type")

    # --- F3  Confidence Level distribution (status palette + labels) -------------------
    fig, ax = plt.subplots(figsize=(6.0, 2.3))
    total = len(ROWS)
    floor = 0.015 * total          # a segment thinner than this would not be visible
    left, tiny, small = 0, [], []
    for lvl in ("Verified", "Conflicted", "Disputed"):
        v = D.get(lvl, 0)
        drawn = max(v, floor) if v else 0
        if v and drawn > v:
            tiny.append(lvl)
        if drawn:
            ax.barh([0], [drawn], left=left, color=LVL_COLOR[lvl], height=0.42,
                    edgecolor="white", linewidth=1.2)
            if drawn > 0.10 * total:
                ax.annotate(f"{lvl}  {v}", (left + drawn / 2, 0), ha="center", va="center",
                            fontsize=8.5, color="white")
            else:
                small.append((lvl, v, left + drawn / 2))
        left += drawn
    # The narrow segments cannot hold a label and their callouts would collide with each
    # other, so they are placed alternately above and below the bar, each with a leader.
    for i_, (lvl, v, xc) in enumerate(small):
        up = (i_ % 2 == 0)
        ax.annotate(f"{lvl}  {v}", (xc, 0.21 if up else -0.21),
                    xytext=(xc + (0.03 if up else 0.06) * total, 0.72 if up else -0.62),
                    ha="left", va="center", fontsize=8, color=INK2,
                    arrowprops=dict(arrowstyle="-", color=INK2, lw=0.7,
                                    shrinkA=0, shrinkB=2))
    ax.set_xlim(0, left * 1.16); ax.set_ylim(-0.95, 1.05)
    ax.set_yticks([]); ax.set_xlabel("triplets")
    ax.spines["left"].set_visible(False)
    handles = [Patch(facecolor=LVL_COLOR[l],
                     label=f"{l}  {D.get(l,0)}  ({D.get(l,0)/total:.1%})")
               for l in ("Verified", "Conflicted", "Disputed")]
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.45), ncol=3)
    note = "" if not tiny else f"  ({', '.join(tiny)} widened to stay visible)"
    ax.set_title("Confidence Level assigned by Definition 1" + note, loc="left")
    finish(fig, "F3_confidence_levels")

    # --- F4  injection test: detection and both audit protocols -----------------------
    if INJ:
        tys = ["T1", "T2", "T3"]
        n_   = [per[(t, "n")] for t in tys]
        det_ = [per[(t, "det")] / per[(t, "n")] for t in tys]
        cor_ = [sum(1 for it in INJ if it["type"] == t and any(
                    normalize_verdict(icorpus.get(f"{it['id']}|{f}", {})) == "TRUE_CONFLICT"
                    for f, _ in inj_flags[it["id"]])) / per[(t, "n")] for t in tys]
        ora_ = [sum(1 for it in INJ if it["type"] == t and any(
                    normalize_verdict(ioracle.get(f"{it['id']}|{f}", {})) == "TRUE_CONFLICT"
                    for f, _ in inj_flags[it["id"]])) / per[(t, "n")] for t in tys]
        x = np.arange(3); w = 0.26
        fig, ax = plt.subplots(figsize=(5.8, 3.1))
        for off, vals, lab, col in ((-w, det_, "detected (rule stage)", C1_),
                                    (0.0, cor_, "confirmed, corpus-referenced", C2_),
                                    (w,  ora_, "confirmed, oracle-referenced", C3_)):
            bb = ax.bar(x + off, vals, w, label=lab, color=col)
            barlabels(ax, bb, "{:.0%}")
        ax.set_xticks(x); ax.set_xticklabels([f"{t}\nn={m}" for t, m in zip(tys, n_)])
        ax.set_ylim(0, 1.18); ax.set_ylabel("share of injected corruptions")
        ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=3, fontsize=8)
        ax.set_title("Only T3 depends on having the uncorrupted reference", loc="left")
        ax.grid(axis="y"); ax.set_axisbelow(True)
        finish(fig, "F4_injection")

    # --- F5  Gold-Standard recall against the similarity threshold --------------------
    fig, ax = plt.subplots(figsize=(5.2, 3.0))
    taus = sorted(rec); vals = [rec[t] for t in taus]
    ax.plot(taus, vals, marker="o", ms=6, lw=2, color=C1_)
    ax.axvline(FUZZY_TAU, color=GRID, lw=1.2, zorder=0)
    ax.annotate(f"reported threshold {FUZZY_TAU:.2f}\n{rec[FUZZY_TAU]:.1%} recall",
                (FUZZY_TAU, rec[FUZZY_TAU]), xytext=(FUZZY_TAU + 0.09, 0.34),
                fontsize=8, color=INK2,
                arrowprops=dict(arrowstyle="-", color=GRID, lw=0.9))
    ax.set_xlabel("cosine-similarity threshold"); ax.set_ylabel("Gold-Standard recall")
    ax.set_ylim(0, 1.05); ax.yaxis.set_major_formatter(lambda v, p: f"{v:.0%}")
    ax.set_title("Recall is sensitive to the matching threshold", loc="left")
    ax.grid(axis="y"); ax.set_axisbelow(True)
    finish(fig, "F5_recall_threshold")

_draw_f1_f5()


writing figures to repro_out\figures_v2
  F1_pipeline_funnel.png / .pdf
  F2_detection_by_type.png / .pdf
  F3_confidence_levels.png / .pdf
  F4_injection.png / .pdf
  F5_recall_threshold.png / .pdf


In [91]:
# ====== cell 61 ======
def _draw_f6_f9():
    # --- F6  knowledge-graph structure -------------------------------------------------
    try:
        import networkx as nx
        HAS_NX = True
    except ImportError:
        HAS_NX = False
        print("  [skip] networkx not installed - F6/F7 not drawn (pip install networkx)")

    if HAS_NX:
        # F6: whole-graph structure. 1,017 nodes is too many to label, so this figure shows
        # shape only - the causal backbone against the relational star clusters.
        G = nx.DiGraph()
        for e in kg.edges:
            G.add_edge(e["source"], e["target"], kind=e.get("edge_type", "relational"),
                       lvl=e.get("confidence_level", "Verified"))
        pos = nx.spring_layout(G, k=0.28, iterations=60, seed=SEED)
        fig, ax = plt.subplots(figsize=(7.2, 5.4))
        rel = [(u, v) for u, v, d in G.edges(data=True) if d["kind"] == "relational"]
        cau = [(u, v) for u, v, d in G.edges(data=True) if d["kind"] == "causal"]
        nx.draw_networkx_edges(G, pos, edgelist=rel, ax=ax, edge_color=GRID, width=0.4,
                               arrows=False, alpha=0.8)
        nx.draw_networkx_edges(G, pos, edgelist=cau, ax=ax, edge_color=C1_, width=0.7,
                               arrows=False, alpha=0.85)
        graded = [(u, v, d["lvl"]) for u, v, d in G.edges(data=True) if d["lvl"] != "Verified"]
        for lvl in ("Conflicted", "Disputed"):
            el = [(u, v) for u, v, l in graded if l == lvl]
            if el:
                nx.draw_networkx_edges(G, pos, edgelist=el, ax=ax, edge_color=LVL_COLOR[lvl],
                                       width=2.4, arrows=False, alpha=1.0)
        vals = [n for n, d in kg.nodes.items() if d.get("type") == "value"]
        ents = [n for n in G.nodes if n not in vals]
        nx.draw_networkx_nodes(G, pos, nodelist=vals, node_size=4, node_color=GRID, ax=ax)
        nx.draw_networkx_nodes(G, pos, nodelist=ents, node_size=10, node_color=INK2,
                               ax=ax, alpha=0.75)
        for lvl in ("Conflicted", "Disputed"):          # redrawn last so they stay visible
            el = [(u, v) for u, v, l in graded if l == lvl]
            if el:
                nx.draw_networkx_edges(G, pos, edgelist=el, ax=ax, edge_color=LVL_COLOR[lvl],
                                       width=2.4, arrows=False, alpha=1.0)
        ax.legend(handles=[
            Line2D([], [], color=C1_, lw=1.4, label=f"causal edge ({len(cau)})"),
            Line2D([], [], color=GRID, lw=1.0, label=f"relational edge ({len(rel)})"),
            Line2D([], [], color=LVL_COLOR["Conflicted"], lw=2, label=f"Conflicted ({D.get('Conflicted',0)})"),
            Line2D([], [], color=LVL_COLOR["Disputed"], lw=2, label=f"Disputed ({D.get('Disputed',0)})"),
        ], loc="lower left", fontsize=8)
        ax.set_title(f"Knowledge graph: {len(kg.nodes)} nodes, {len(kg.edges)} edges", loc="left")
        ax.axis("off")
        finish(fig, "F6_knowledge_graph")

        # F7: the subgraph one query retrieves, drawn as hop layers running left to
        # right from the matched parameter - the order the retriever walks - so that
        # every node carries one readable label.
        import textwrap
        C1_QUERY = "How does low dissolved oxygen (DO) affect wasabi rhizome yield?"
        params, entities = match_nodes(C1_QUERY)
        keep_edges, touched = [], set(params)
        for s0 in params + entities:
            for path in causal_paths(s0):
                for e in path:
                    keep_edges.append(e); touched.add(e["source"]); touched.add(e["target"])
        for n in params + entities:            # relational edges of the matched parameters
            keep_edges += RELATIONAL.get(n, [])
        seen_k, uniq = set(), []
        for e in keep_edges:
            k = (e["source"], e["target"])
            if k not in seen_k:
                seen_k.add(k); uniq.append(e)

        H = nx.DiGraph()
        for e in uniq:
            H.add_edge(e["source"], e["target"], kind=e.get("edge_type", "relational"),
                       lvl=e.get("confidence_level", "Verified"), pred=e.get("predicate", ""))

        # hop layer of every node, measured from the matched parameters
        roots = [p for p in params if p in H] or [n for n, d in H.in_degree() if d == 0][:1]
        layer = {r: 0 for r in roots}
        frontier = list(roots)
        while frontier:
            nxt = []
            for u in frontier:
                for v in H.successors(u):
                    if v not in layer:
                        layer[v] = layer[u] + 1; nxt.append(v)
            frontier = nxt
        for n in H.nodes:                       # anything unreached hangs off its source
            layer.setdefault(n, 1)

        # Cap each layer so the figure stays readable. Every endpoint of a non-Verified
        # edge is kept whatever its degree - those edges are what the figure is about -
        # and the remaining slots go to the best-connected nodes.
        PER_LAYER = 10
        must = set(roots)
        for u, v, d in H.edges(data=True):
            if d["lvl"] != "Verified":
                must.add(u); must.add(v)
        by_layer = defaultdict(list)
        for n, L in layer.items():
            by_layer[L].append(n)
        keep = set(must)
        for L in sorted(by_layer):
            ns = sorted(by_layer[L], key=lambda n: -(H.in_degree(n) + H.out_degree(n)))
            room = PER_LAYER - sum(1 for n in ns if n in must)
            keep.update(n for n in ns[:max(0, room)])
        dropped = len(H) - len(keep)
        H = H.subgraph(keep).copy()

        # Layered coordinates: one column per hop, generous column spacing so each label
        # sits beside its own node instead of on a neighbour.
        DX, DY = 3.0, 1.0
        posH, nlayers = {}, max(layer[n] for n in H) + 1
        widest = 1
        for L in range(nlayers):
            ns = sorted([n for n in H if layer[n] == L],
                        key=lambda n: -(H.in_degree(n) + H.out_degree(n)))
            widest = max(widest, len(ns))
            for r_, n in enumerate(ns):
                posH[n] = (L * DX, -(r_ - (len(ns) - 1) / 2) * DY)

        fig, ax = plt.subplots(figsize=(2.4 + 2.6 * nlayers, max(3.4, 0.62 * widest + 1.2)))
        for kind, style in (("causal", "solid"), ("relational", "dashed")):
            for lvl in ("Verified", "Disputed", "Conflicted"):
                el = [(u, v) for u, v, d in H.edges(data=True)
                      if d["kind"] == kind and d["lvl"] == lvl]
                if el:
                    nx.draw_networkx_edges(
                        H, posH, edgelist=el, ax=ax, style=style,
                        width=1.2 if lvl == "Verified" else 2.2,
                        edge_color=LVL_COLOR[lvl], alpha=0.95, arrows=True, arrowsize=9,
                        node_size=170, connectionstyle="arc3,rad=0.05")
        vnodes = [n for n in H if kg.nodes.get(n, {}).get("type") == "value"]
        enodes = [n for n in H if n not in vnodes]
        nx.draw_networkx_nodes(H, posH, nodelist=enodes, node_size=170, node_color="white",
                               edgecolors=INK2, linewidths=0.9, ax=ax)
        nx.draw_networkx_nodes(H, posH, nodelist=vnodes, node_size=80, node_color=GRID,
                               edgecolors="white", linewidths=0.8, node_shape="s", ax=ax)
        for n, (px, py) in posH.items():
            t = str(kg.nodes.get(n, {}).get("label", n))
            t = textwrap.fill(t if len(t) <= 40 else t[:38] + "...", 20)
            ax.text(px + 0.30, py, t, ha="left", va="center", fontsize=6.6, color=INK,
                    linespacing=1.15)
        top = max(y for _, y in posH.values())
        for L in range(nlayers):
            ax.text(L * DX, top + 0.85, "matched" if L == 0 else f"hop {L}",
                    ha="left", fontsize=8, color=INK2)
        ax.set_xlim(-0.5, (nlayers - 1) * DX + 2.5)
        ax.set_ylim(min(y for _, y in posH.values()) - 0.9, top + 1.5)
        ax.legend(handles=[
            Line2D([], [], color=INK2, lw=1.6, label="causal edge (solid)"),
            Line2D([], [], color=INK2, lw=1.0, ls="--", label="relational edge (dashed)"),
            Line2D([], [], color=LVL_COLOR["Verified"], lw=2, label="Verified"),
            Line2D([], [], color=LVL_COLOR["Disputed"], lw=2.6, label="Disputed"),
            Line2D([], [], color=LVL_COLOR["Conflicted"], lw=2.6, label="Conflicted"),
        ], loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=5, fontsize=8)
        ax.set_title("Subgraph retrieved for the case-study query, with Confidence Levels"
                     + (f"  ({dropped} lower-degree nodes not drawn)" if dropped else ""),
                     loc="left")
        ax.axis("off")
        finish(fig, "F7_retrieved_subgraph")

    # --- F8  multi-LLM judge ------------------------------------------------------------
    if JUDGE_ROWS:
        LBL = globals().get("BASELINE_LABELS", {b: b for b in REPORT_BASES})
        x = np.arange(len(REPORT_BASES)); w = 0.24
        fig, ax = plt.subplots(figsize=(7.6, 3.6))
        for i, (j, col) in enumerate(zip(JUDGES, (C1_, C2_, C3_))):
            v = [np.mean([mean3(CELL[(j, b, q)]) for q in QIDS]) for b in REPORT_BASES]
            bb = ax.bar(x + (i - 1) * w, v, w, label=j, color=col)
            barlabels(ax, bb, "{:.2f}")
        m = [np.mean([np.mean([mean3(CELL[(j, b, q)]) for q in QIDS]) for j in JUDGES])
             for b in REPORT_BASES]
        hi_ = [max(np.mean([mean3(CELL[(j, b, q)]) for q in QIDS]) for j in JUDGES)
               for b in REPORT_BASES]
        for x_, m_, h_ in zip(x, m, hi_):
            ax.annotate(f"mean {m_:.2f}", (x_, h_), textcoords="offset points",
                        xytext=(0, 20), ha="center", fontsize=8, color=INK,
                        fontweight="bold")
        ax.set_xticks(x)
        import textwrap as _tw
        ax.set_xticklabels([b + "\n" + _tw.fill(str(LBL.get(b, b)), 14) for b in REPORT_BASES],
                           fontsize=7.5)
        ax.set_ylim(0, 6.0); ax.set_ylabel("rubric score (1-5)")
        ax.legend(ncol=4, loc="upper left", bbox_to_anchor=(0, 1.02), fontsize=8)
        ax.set_title("Multi-LLM judge evaluation, per judge", loc="left")
        ax.grid(axis="y"); ax.set_axisbelow(True)
        finish(fig, "F8_judge_by_baseline")

        fig, ax = plt.subplots(figsize=(6.0, 3.2))
        crits = ("FA", "CV", "AC")
        x = np.arange(len(REPORT_BASES)); w = 0.26
        for i, (c, col) in enumerate(zip(crits, (C1_, C2_, C3_))):
            v = [np.mean([CELL[(j, b, q)][c] for j in JUDGES for q in QIDS]) for b in REPORT_BASES]
            bb = ax.bar(x + (i - 1) * w, v, w,
                        label={"FA": "factual accuracy", "CV": "causal validity",
                               "AC": "actionability"}[c], color=col)
            barlabels(ax, bb, "{:.2f}")
        ax.set_xticks(x); ax.set_xticklabels(REPORT_BASES)
        ax.set_ylim(0, 6.0); ax.set_ylabel("rubric score (1-5)")
        ax.legend(ncol=3, loc="upper left", bbox_to_anchor=(0, 1.02), fontsize=8)
        ax.set_title("Score by rubric criterion", loc="left")
        ax.grid(axis="y"); ax.set_axisbelow(True)
        finish(fig, "F9_judge_by_criterion")

_draw_f6_f9()


  F6_knowledge_graph.png / .pdf
  F7_retrieved_subgraph.png / .pdf
  F8_judge_by_baseline.png / .pdf
  F9_judge_by_criterion.png / .pdf


In [101]:
# ---------- Part 12 (extra) - F13: the paper's headline contrast ----------
# Table 5 reports 9/14 vs 0/14 vs 0/14 as three numbers. The paired structure behind
# those numbers - which queries each condition disclosed on - is what makes McNemar the
# right test, and it is invisible in a table. Panel A shows it directly. Panel B puts the
# judged UR scale beside it, with each judge's mean, so the reader can see that the two
# measures agree in direction while differing in strength.
#
# No API calls. Requires the Part 9e cells (PV, _hit, _qd2) and Part 9d (UCELL, JU, QU).
if not all(k in globals() for k in ("_hit", "_qd2", "UCELL", "JU", "QU")):
    print("[skip] F13 needs Part 9d and the strict-disclosure cell in Part 9e.")
else:
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
 
    # Categorical slots 1-3 of the validated palette, assigned to conditions (not to rank)
    # and used identically wherever these conditions appear.
    COL = {"B5": "#2a78d6", "B3": "#eb6834", "B4": "#1baf7a"}
    ORDER = ["B3", "B4", "B5"]                      # reading order, weakest to proposed
    INK, MUTED, GRID = "#0b0b0b", "#52514e", "#d8d8d4"
 
    fig, (axA, axB) = plt.subplots(
        1, 2, figsize=(7.2, 3.3), dpi=300, gridspec_kw={"width_ratios": [2.15, 1]})
 
    # ---------------- Panel A: which disputed queries were disclosed ----------------
    qs = [i + 1 for i in _qd2]
    for r, b in enumerate(ORDER):
        y = len(ORDER) - 1 - r
        for c, i in enumerate(_qd2):
            hit = _hit(b, i, 2)
            axA.scatter(c, y, s=118, marker="s",
                        facecolor=COL[b] if hit else "none",
                        edgecolor=COL[b] if hit else GRID,
                        linewidths=0 if hit else 1.1, zorder=3)
        n = sum(_hit(b, i, 2) for i in _qd2)
        axA.text(len(_qd2) - 0.35, y, f"{n}/{len(_qd2)}", va="center", ha="left",
                 fontsize=9, color=INK, fontweight="bold" if n else "normal")
    axA.set_yticks(range(len(ORDER)))
    axA.set_yticklabels(list(reversed(ORDER)), fontsize=9.5, color=INK)
    axA.set_xticks(range(len(_qd2)))
    axA.set_xticklabels(qs, fontsize=7.5, color=MUTED)
    axA.set_xlabel("disputed query", fontsize=9, color=MUTED)
    axA.set_xlim(-0.7, len(_qd2) + 0.9)
    axA.set_ylim(-0.6, len(ORDER) - 0.4)
    axA.set_title("A   Specific disclosure, judge-free", loc="left",
                  fontsize=10, color=INK, pad=8)
    for s in ("top", "right", "left", "bottom"): axA.spines[s].set_visible(False)
    axA.tick_params(length=0)
    axA.legend(handles=[
        Line2D([], [], marker="s", linestyle="", markersize=8,
               markerfacecolor="#8c8c8c", markeredgecolor="#8c8c8c",
               label="states the disagreement and reports two divergent values"),
        Line2D([], [], marker="s", linestyle="", markersize=8, markerfacecolor="none",
               markeredgecolor=GRID, label="does not")],
        loc="upper left", bbox_to_anchor=(0, -0.22), frameon=False,
        fontsize=7.6, labelcolor=MUTED, handletextpad=0.5)
 
    # ---------------- Panel B: judged uncertainty reporting ----------------
    for r, b in enumerate(ORDER):
        y = len(ORDER) - 1 - r
        per = [float(np.mean([UCELL[(j, b, q)] for q in QU if (j, b, q) in UCELL]))
               for j in JU]
        m = float(np.mean(per))
        axB.plot([min(per), max(per)], [y, y], color=COL[b], lw=1.6, alpha=0.35,
                 solid_capstyle="round", zorder=2)
        axB.scatter(per, [y] * len(per), s=22, facecolor="white",
                    edgecolor=COL[b], linewidths=1.3, zorder=5)
        axB.scatter([m], [y], s=95, color=COL[b], zorder=4)
        axB.text(m, y + 0.28, f"{m:.2f}", ha="center", va="bottom",
                 fontsize=9, color=INK)
    axB.set_yticks(range(len(ORDER)))
    axB.set_yticklabels([""] * len(ORDER))
    axB.set_ylim(-0.6, len(ORDER) - 0.4)
    axB.set_xlim(1, 5)
    axB.set_xticks([1, 2, 3, 4, 5])
    axB.tick_params(labelsize=8, colors=MUTED, length=0)
    axB.set_xlabel("uncertainty reporting (1–5)", fontsize=9, color=MUTED)
    axB.set_title("B   Judged scale", loc="left", fontsize=10, color=INK, pad=8)
    axB.grid(axis="x", color=GRID, lw=0.6); axB.set_axisbelow(True)
    for s in ("top", "right", "left"): axB.spines[s].set_visible(False)
    axB.spines["bottom"].set_color(GRID)
    axB.legend(handles=[
        Line2D([], [], marker="o", linestyle="", markersize=6, color="#8c8c8c",
               label="mean over judges"),
        Line2D([], [], marker="o", linestyle="", markersize=5, markerfacecolor="white",
               markeredgecolor="#8c8c8c", label="individual judge")],
        loc="upper left", bbox_to_anchor=(0, -0.22), frameon=False,
        fontsize=7.6, labelcolor=MUTED, handletextpad=0.5)
 
    fig.subplots_adjust(bottom=0.32, wspace=0.12)
    finish(fig, "F13_disclosure_and_UR")
    print("F13_disclosure_and_UR written")

  F13_disclosure_and_UR.png / .pdf
F13_disclosure_and_UR written


In [69]:
# ====== cell 62 ======
def _draw_f10_f12():
    # --- F10/F11  strawberry generalization --------------------------------------------
    if BERRY_OK:
        ps = [p for p in PARAMS if FULL.get(p) is not None]
        ncol = 4; nrow = (len(ps) + ncol - 1) // ncol
        fig, axes = plt.subplots(nrow, ncol, figsize=(3.1 * ncol, 2.5 * nrow), sharey=True)
        for ax, p in zip(np.ravel(axes), ps):
            span = NR[p][1] - NR[p][0]
            xs, ys, es = [], [], []
            for k in SUBSET_SIZES + [len(SIDS)]:
                v = col("weighted_median", k, p)
                if not len(v):
                    continue
                xs.append(k); ys.append((v.mean() - FULL[p]) / span)
                es.append((v.std(ddof=1) if len(v) > 1 else 0.0) / span)
            ax.axhline(0, color=GRID, lw=1.0, zorder=0)
            ax.errorbar(xs, ys, yerr=es, marker="o", ms=5, capsize=3, lw=1.6, color=C1_,
                        ecolor=GRID, elinewidth=1.4)
            ax.set_title(p, fontsize=9, loc="left"); ax.set_xlabel("papers", fontsize=8)
            ax.tick_params(labelsize=7); ax.grid(axis="y"); ax.set_axisbelow(True)
        for ax in np.ravel(axes)[len(ps):]:
            ax.axis("off")
        np.ravel(axes)[0].set_ylabel("deviation / parameter range", fontsize=8)
        fig.suptitle(f"Convergence to the full-corpus value ({N_DRAWS} draws per size); "
                     "bars = s.d. across draws", fontsize=10, x=0.01, ha="left")
        fig.tight_layout()
        finish(fig, "F10_strawberry_convergence")

        fig, axes = plt.subplots(nrow, ncol, figsize=(3.1 * ncol, 2.5 * nrow))
        for ax, p in zip(np.ravel(axes), ps):
            v = col("weighted_median", 19, p)
            ax.hist(v, bins=14, color=GRID, edgecolor="white", linewidth=0.8)
            lo, hi = np.percentile(v, [2.5, 97.5])
            ax.axvspan(lo, hi, color=C1_, alpha=0.12, zorder=0)
            ax.axvline(FULL[p], color=C2_, lw=1.8)
            ax.set_title(f"{p}   |z| = {rep[p]['abs_z']:.2f}", fontsize=8.5, loc="left")
            ax.tick_params(labelsize=7); ax.set_yticks([])
        for ax in np.ravel(axes)[len(ps):]:
            ax.axis("off")
        fig.legend(handles=[
            Line2D([], [], color=C2_, lw=1.8, label="full-corpus value"),
            Patch(facecolor=C1_, alpha=0.12, label="empirical 95% interval (diagnostic)"),
        ], loc="lower center", ncol=2, fontsize=8, bbox_to_anchor=(0.5, -0.02))
        fig.suptitle("Distribution of 19-paper subset estimates", fontsize=10, x=0.01, ha="left")
        fig.tight_layout(rect=(0, 0.05, 1, 1))
        finish(fig, "F11_strawberry_representativeness")

        fig, ax = plt.subplots(figsize=(5.4, 3.0))
        for (name, vals), col_ in zip(norm_tab.items(), (C1_, C2_, C3_, GRID)):
            ax.plot(SUBSET_SIZES, vals, marker="o", ms=5, lw=1.8, color=col_, label=name)
        ax.set_xlabel("papers in the subset"); ax.set_ylabel("mean normalized error")
        ax.set_xticks(SUBSET_SIZES)
        ax.legend(fontsize=8); ax.grid(axis="y"); ax.set_axisbelow(True)
        ax.set_title("Convergence holds under every normalizer", loc="left")
        finish(fig, "F12_strawberry_normalizers")

    print(f"\n{len(list(FIG_DIR.glob('*.png')))} figures written to {FIG_DIR}")
    MANIFEST["figures"] = sorted(p.stem for p in FIG_DIR.glob("*.png"))

_draw_f10_f12()


  F10_strawberry_convergence.png / .pdf
  F11_strawberry_representativeness.png / .pdf
  F12_strawberry_normalizers.png / .pdf

12 figures written to repro_out\figures_v2


In [70]:
# ====== cell 63 ======
# =============== Tables ===============
# Every table the manuscript needs, written to repro_out/tables/ as CSV. These are
# generated from this run; they are not compared against any previously published table.
import csv
TAB_DIR = OUT_DIR / f"tables{OUT_TAG}"; TAB_DIR.mkdir(exist_ok=True)

def write_table(name, header, rows, note=""):
    p = TAB_DIR / f"{name}.csv"
    with open(p, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)
        if note:
            w.writerow([]); w.writerow([f"# {note}"])
    print(f"  {name}.csv  ({len(rows)} rows)")

def _write_tables():
    print("writing tables to", TAB_DIR)

    write_table("T1_corpus", ["source_id", "triplets", "share"],
                [[s, n, f"{n/len(TRIPLETS):.1%}"]
                 for s, n in sorted(Counter(t.get("source_id") for t in TRIPLETS).items())],
                f"{len(TRIPLETS)} triplets from {len(by_src)} documents; "
                f"{sum(1 for t in TRIPLETS if t.get('unit'))} carry a unit, "
                f"{sum(1 for t in TRIPLETS if t.get('condition'))} a condition")

    write_table("T2_reference_tables", ["parameter", "min", "max", "canonical_unit", "aliases"],
                [[k, v["min"], v["max"], v["unit"] or "(dimensionless)",
                  "; ".join(KNOWN_UNIT_ALIASES.get(k, []))] for k, v in THRESHOLDS.items()],
                f"{len(UNIT_CONVERSIONS)} unit conversions, "
                f"{len(UNIT_NOTATION_MAP)} notation variants normalised before comparison")

    write_table("T3_step2_detection", ["conflict_type", "flags_raised", "confirmed",
                                       "confirmation_rate"],
                [[t, by_type.get(t, 0), sum(1 for r in ROWS if t in r["confirmed_types"]),
                  f"{sum(1 for r in ROWS if t in r['confirmed_types'])/by_type[t]:.1%}"
                  if by_type.get(t) else "-"] for t in ("T1", "T2", "T3_candidate")],
                f"{sum(1 for f in STEP2.values() if f)} triplets flagged, {n_flags} flags; "
                f"T1 by criterion {dict(t1_crit)}")

    write_table("T4_confidence_levels", ["level", "triplets", "share"],
                [[l, D.get(l, 0), f"{D.get(l,0)/len(ROWS):.1%}"]
                 for l in ("Verified", "Conflicted", "Disputed")],
                "Definition 1 over the set of confirmed types; "
                f"{D.get('Conflicted',0)+D.get('Disputed',0)} triplets carry a confirmed conflict")

    write_table("T5_confirmed_combinations", ["confirmed_types", "triplets"],
                [[" + ".join(k) if k else "-", v] for k, v in
                 Counter(tuple(sorted(r["confirmed_types"])) for r in ROWS
                         if r["confirmed_types"]).most_common()],
                "combinations Table 4 of the manuscript must cover")

    if INJ:
        write_table("T6_injection",
                    ["injected", "n", "detected", "type_match", "confirmed_corpus",
                     "confirmed_oracle"],
                    [[t, per[(t, "n")], per[(t, "det")], per[(t, "type_ok")],
                      sum(1 for it in INJ if it["type"] == t and any(
                          normalize_verdict(icorpus.get(f"{it['id']}|{f}", {})) == "TRUE_CONFLICT"
                          for f, _ in inj_flags[it["id"]])),
                      sum(1 for it in INJ if it["type"] == t and any(
                          normalize_verdict(ioracle.get(f"{it['id']}|{f}", {})) == "TRUE_CONFLICT"
                          for f, _ in inj_flags[it["id"]]))]
                     for t in ("T1", "T2", "T3")],
                    "corpus-referenced is the system's rate; oracle-referenced is a ceiling "
                    "obtained by showing the auditor the uncorrupted entry")

    write_table("T7_gold_false_alarm", ["gold_id", "subject", "object", "condition",
                                        "flag", "auditor"],
                [[gid, gs_by_id[gid]["subject"], gs_by_id[gid]["object"],
                  gs_by_id[gid].get("condition") or "none",
                  ",".join(ft for ft, _ in fl),
                  ";".join(str(normalize_verdict(gaudits.get(f"{gid}|{ft}", {})))
                           for ft, _ in fl)] for gid, fl in sorted(flagged.items())],
                f"rule stage {len(flagged)}/{len(GOLD)}; "
                f"after the audit {gs_conf}/{len(GOLD)}")

    write_table("T8_recall", ["tau", "recall", "matched"],
                [[f"{t:.2f}", f"{rec[t]:.1%}", int(round(rec[t] * len(GOLD)))] for t in sorted(rec)],
                f"exact-string recall {exact}/{len(GOLD)}; not recovered at "
                f"tau={FUZZY_TAU}: {', '.join(miss)}")

    write_table("T9_knowledge_graph", ["metric", "value"],
                [[k, v] for k, v in kg.get_stats().items()] +
                [["simple_causal_paths_by_hops", str(dict(sorted(depths.items())))],
                 ["longest_simple_causal_chain", max(depths) if depths else 0],
                 ["causal_entities", len(ents)]])

    write_table("T10_baselines",
                ["id", "label", "retrieval", "multi_hop", "community_summaries",
                 "confidence_shown"],
                [[b, BASELINE_LABELS[b], s["retrieval"], s["multi_hop"],
                  s["community_summaries"], s["confidence_shown"]]
                 for b, s in sorted(BASELINE_SPEC.items())],
                "no condition implements community-summary indexing")

    if JUDGE_ROWS:
        write_table("T11_judge_by_baseline", ["system", "label"] + JUDGES + ["mean"],
                    [[b, BASELINE_LABELS.get(b, b)] +
                     [f"{np.mean([mean3(CELL[(j,b,q)]) for q in QIDS]):.2f}" for j in JUDGES] +
                     [f"{means[b]:.2f}"] for b in REPORT_BASES],
                    f"{len(JUDGE_ROWS)} recorded judgments; "
                    f"Krippendorff alpha (ordinal) overall {alphas['overall']:.3f}")
        write_table("T12_judge_by_criterion", ["system", "FA", "CV", "AC"],
                    [[b] + [f"{per_crit[b][c]:.2f}" for c in ("FA", "CV", "AC")] for b in REPORT_BASES])
        write_table("T13_significance", ["comparison", "raw_p", "holm_p", "significant"],
                    [[k, f"{fam9[k]:.3g}", f"{h9[k]:.3g}", "yes" if h9[k] < 0.05 else "no"]
                     for k in sorted(fam9, key=lambda k: fam9[k])] +
                    [[k, f"{fam3[k]:.3g}", f"{h3[k]:.3g}", "yes" if h3[k] < 0.05 else "no"]
                     for k in sorted(fam3, key=lambda k: fam3[k])],
                    f"Wilcoxon signed-rank, one-sided, method={WILCOXON_METHOD}; Holm within family")

    if BERRY_OK:
        write_table("T14_strawberry_ladder",
                    ["parameter"] + [f"n={k}" for k in SUBSET_SIZES] + ["full", "literature_ref"],
                    [[p] + [f"{TABLE12[p][k][0]:.1f} +- {TABLE12[p][k][1]:.1f}" for k in SUBSET_SIZES]
                     + [f"{FULL[p]:.1f}" if FULL.get(p) is not None else "-", REF.get(p)]
                     for p in PARAMS],
                    f"{N_DRAWS} random draws per size, seed {SEED}; the full corpus is the "
                    "internal reference, the literature column is context only")
        write_table("T15_strawberry_robustness", ["analysis", "n=19", "n=38", "n=57"],
                    [[f"normalizer: {k}"] + [f"{v:.3f}" for v in vals]
                     for k, vals in norm_tab.items()] +
                    [[f"aggregator: {k}"] + [f"{v:.3f}" for v in vals]
                     for k, vals in est_tab.items()],
                    f"trimmed-mean proportion {TRIM_PROP:.0%}; "
                    f"max deviation at k=57 {SD_CLAIM['max_at_57']:.2f} SD ({SD_CLAIM['argmax']})")

    print(f"\n{len(list(TAB_DIR.glob('*.csv')))} tables written to {TAB_DIR}")
    MANIFEST["tables"] = sorted(p.stem for p in TAB_DIR.glob("*.csv"))

_write_tables()


writing tables to repro_out\tables_v2
  T1_corpus.csv  (19 rows)
  T2_reference_tables.csv  (10 rows)
  T3_step2_detection.csv  (3 rows)
  T4_confidence_levels.csv  (3 rows)
  T5_confirmed_combinations.csv  (4 rows)
  T6_injection.csv  (3 rows)
  T7_gold_false_alarm.csv  (3 rows)
  T8_recall.csv  (6 rows)
  T9_knowledge_graph.csv  (10 rows)
  T10_baselines.csv  (5 rows)
  T11_judge_by_baseline.csv  (5 rows)
  T12_judge_by_criterion.csv  (5 rows)
  T13_significance.csv  (12 rows)
  T14_strawberry_ladder.csv  (8 rows)
  T15_strawberry_robustness.csv  (8 rows)

15 tables written to repro_out\tables_v2


`cell 64`

## Part 13 — Conformance audit and revision checklist

Two tables, and the split between them is the point.

**A. Paper-to-code implementation audit.** Audited against the manuscript **as submitted**.
The *claim* column quotes what the current text says, the *status* is whether this code does
that, and the *note* carries the wording the revision should use. Nothing in A is stated as
the revised paper will state it — that is B's job, and keeping the two apart is what makes
the status column mean anything.

| status | meaning | what is owed |
|---|---|---|
| `implemented` | the submitted text says X and the code does X | nothing |
| `EDIT PENDING` | the code does what B decided; the submitted text still says something else | a rewrite, named in B |
| `PARTIAL` | the code does something narrower or different from the submitted claim | a rewrite **and** a limitation sentence |
| `CLAIM NOT SUPPORTED` | this run contradicts a figure the text asserts | the corrected figure, computed here |
| `NOT IMPLEMENTED` | the text claims a component this code does not contain | a rewrite or the component |

A row does **not** flip to `implemented` because the code is right. It flips when the
manuscript sentence has actually been changed. Flipping it early is how an edit that is
still owed disappears from the checklist, which is the one thing this Part exists to prevent.

**B. Decisions taken.** Every conflict found between the manuscript and the code, which side
moves, and the concrete edit. Every figure in B is computed from this run rather than typed
in, so a decision cannot go stale against the numbers it is about.
`MANIFEST["decisions"]` holds the same list in machine-readable form.

**Every identified inconsistency has an explicit resolution and a named manuscript edit —
and the edits themselves are still outstanding.** Nothing here is open in the sense of
undecided; a great deal is open in the sense of not yet written into the paper. A is the
list of what remains to be typed into the manuscript, and it is complete only for the
inconsistencies this notebook can see.


In [72]:
# ====== cell 65 ======
# ============ A. Paper-to-code implementation audit ============
# Audited against the manuscript AS SUBMITTED, not against the revision B proposes.
#   claim  = what the current text says
#   status = whether this code does that
#   note   = the wording the revision should use, with every figure taken from this run
# A row flips to "implemented" when the manuscript sentence has been changed, never
# because the code is right. See the markdown above for the full status legend.

def _mf(*keys, default=None):
    d = MANIFEST
    for k in keys:
        if not isinstance(d, dict) or k not in d:
            return default
        d = d[k]
    return d

def _judge_panel():
    """The judges that actually produced scores in this run, read back from the manifest.

    Only judges in judges_present are named. A judge that was configured but never
    answered (J3 when the local server is down) must not appear in a sentence the
    manuscript will copy - that is how a two-judge panel gets written up as three.
    """
    rj = MANIFEST.get("regenerated_judging", {})
    m, present = rj.get("models", {}), rj.get("judges_present", [])
    named = [(j, m[j]) for j in sorted(present) if j in m]
    local = (rj.get("local_judge") or {}).get("model")
    return named, local

def _judge_panel_str():
    named, _ = _judge_panel()
    if not named:
        return "NOT RESOLVED in this run - run Part 9b before quoting section 3.5"
    return ", ".join(f"{j} = {mid}" for j, mid in named)

def _judge_panel_sentence():
    """Verbatim replacement text for 3.5, or a refusal to write one."""
    named, local = _judge_panel()
    if len(named) < 3:
        return (f"CANNOT BE WRITTEN YET: only {len(named)} judge(s) scored this run "
                f"({_judge_panel_str()}); 3.5 claims three. Either start the local "
                f"open-weight server and re-run Part 9b, or revise 3.5 and 4.3.1 to a "
                f"{len(named)}-judge panel and re-report the reliability coefficients, "
                f"which are panel-size dependent.")
    tail = (f", the last ({local}) served locally from open weights through an "
            "OpenAI-compatible endpoint" if local else
            ", the last served locally from open weights")
    return f'"the panel comprised {_judge_panel_str()}{tail}"'

def _sd_claim_str():
    """The measured sufficiency figures, so no SD number is typed by hand."""
    c = MANIFEST.get("strawberry_convergence", {}).get("sd_sufficiency_claim")
    if not c:
        return None
    return (f"measured maximum is {c['max_at_57']:.2f} SD ({c['argmax']}); "
            f"{c['n_le_0_1']} of {c['n_params']} parameters are at or below 0.1 SD and "
            f"{c['n_le_1_0']} of {c['n_params']} below 1.0 SD at k = 57")
IMPLEMENTATION = [
 ("§3.2.1 Table 3", "T1 = differing quantitative values for the same parameter",
  "implemented", "criteria (i) and (ii) in detect_all"),
 ("§3.2.1 Table 3", "T2 = unit-notation mismatch for the same parameter",
  "implemented", "pipeline._check_unit_mismatch"),
 ("§3.2.1 Table 3", "T3 = apparent contradiction from differing measurement conditions",
  "implemented", "Algorithm 1 line 7 trigger"),
 ("§3.2.2", "criterion (ii) = value outside the Normal Range Table",
  "implemented", "pipeline._check_numeric_range"),
 ("§3.2.2", "only the two reference tables are domain-specific",
  "EDIT PENDING", "Part 2 prints FIVE domain tables: THRESHOLDS, KNOWN_UNIT_ALIASES, "
                  "EQUIVALENT_UNITS, UNIT_CONVERSIONS and UNIT_NOTATION_MAP. Say five and "
                  "list them; the portability claim is weaker but defensible, and five "
                  "small tables is still a small port - see decision 10"),
 ("§3.2.2 criterion (i)", "T1 criterion (i) fires on NON-IDENTICAL values from different "
  "sources",
  "EDIT PENDING", "the code implements NON-OVERLAPPING ranges, which is what Table 3 and "
                  "§3.4 already imply. Strict non-identity would flag 8.0 against 8.1, and "
                  "the counts in §4.2.1 follow the non-overlapping rule - see decision 2"),
 ("Algorithm 1 L5-11", "the T3 branch is an `else if`, so T3 is tested only when T1 and T2 "
  "did not fire",
  "EDIT PENDING", "the code evaluates T2, T1 and T3 in one pass, which is what the note "
                  "below the pseudocode already says. Under the else-if reading T3 "
                  "candidates fall from 34 to 10. Redraw the pseudocode and delete the "
                  "reconciling note - see decision 3"),
 ("§3.2.2", "unit-notation normalisation before comparison",
  "implemented", "UNIT_NOTATION_MAP, 20 variants"),
 ("Algorithm 1 L7", "T3 trigger = condition NULL AND subject in THRESHOLDS",
  "implemented", "no peer requirement by default"),
 ("Algorithm 1 L6/L8", "LLM_Audit(t, flag, d_i) - the auditor receives the source document",
  "EDIT PENDING", "the auditor is called with the domain thresholds and the corpus peers "
                  "for the parameter. A single document cannot support a CROSS-SOURCE "
                  "criterion, so this is not a detail: change the signature to "
                  "LLM_Audit(t, flag, THRESHOLDS, peers(t, T)) - see decision 4"),
 ("§3.2.4 Table 4", "FALSE_ALARM -> Verified; T1/T2 -> Conflicted; T3 -> Disputed",
  "implemented", "assign_confidence, with a self-test"),
 ("§3.2.4 Definition 1", "D(t) takes a single value from {none, T1, T2, T3}",
  "EDIT PENDING", "detection is multi-label - confirmed_types is a SET - and Table 4 has "
                  "no row for a triplet confirmed T1 and T3 at once. Restate Definition 1 "
                  "over the set (any confirmed T1 or T2 -> Conflicted; else any confirmed "
                  "T3 -> Disputed; else Verified) and add the combined row - decision 5"),
 ("§3.3", "two nodes per triplet joined by a directed predicate edge",
  "implemented", "pipeline.KnowledgeGraph.add_triplet"),
 ("§3.3", "parallel edges preserved rather than overwritten",
  "implemented", "one edge per triplet, no de-duplication"),
 ("§3.3", "edges annotated with the Confidence Level",
  "PARTIAL", "relational edges carry the audited level; causal edges come from "
             "causal_relations.json, which the detector never sees, so they are "
             "Verified by construction - see decision 12"),
 ("§3.3", "stored in Neo4j 5.x",
  "PARTIAL", "in-memory graph; pipeline.export_for_neo4j emits the Cypher"),
 ("§3.4", "query terms matched to entity and parameter nodes",
  "implemented", "match_nodes, with the parameter-synonym map added in Part 8; before "
                 "that map no query reached a parameter node - see decision 11"),
 ("§3.4", "connected CAUSAL edges traversed outward",
  "implemented", "causal_paths restricts traversal to causal edges"),
 ("§3.4", "Confidence Level propagated along the path",
  "implemented", "path_confidence = weakest edge on the path"),
 ("§3.4", "Disputed/Conflicted paths trigger an explicit uncertainty warning",
  "implemented", "the note appended by retrieve() for B5"),
 # The judge identities are read from the run rather than asserted, so this row cannot
 # drift from what actually scored the answers. It stays PARTIAL until §3.5 is rewritten:
 # the status describes the manuscript-to-code gap, not the code.
 ("§3.5 judge panel", "three heterogeneous judges, two vendors plus one open-weight",
  "PARTIAL", "the panel is heterogeneous, but not the models §3.5 names. The judges that "
             "actually scored this run are " + _judge_panel_str() + ". The open-weight "
             "judge (J3) is served "
             "LOCALLY through an OpenAI-compatible endpoint because the 70B model the "
             "recorded study used was withdrawn from its provider's free and developer "
             "tiers on 2026-08-16; open weights served locally cannot be deprecated out "
             "of a deposit. §3.5 must name exactly these model ids and say that "
             "J3 is run locally - REPLACEMENT TEXT: " + _judge_panel_sentence()),
 ("§3.5 primary rubric", "three criteria scored 1-5 (FA, CV, AC), overall mean reported",
  "implemented", "unchanged from the manuscript: the overall mean is the arithmetic mean "
                 "of FA, CV and AC over every judge and query. No fourth criterion enters "
                 "it, so Table 11 remains comparable with the recorded study"),
 ("§3.5 UR rubric", "uncertainty reporting is not among the scored criteria",
  "EDIT PENDING", "a fourth criterion, UR, is added as a SEPARATE pass with its own rubric, "
             "restricted to the queries whose retrieved evidence carries a confirmed "
             "disagreement. It is reported beside the primary rubric (3 + 1), never "
             "averaged into it. FA/CV/AC cannot register what the auditing layer does, "
             "which is why B5 - B3 is null on the primary rubric and positive on UR; "
             "§3.5 must describe UR as a second, restricted rubric and say so"),
 ("§3.4", "the uncertainty warning names the items it warns about",
  "implemented", "graded items are held back from the retrieval limit rather than sorted "
                 "to the end of it; before that fix the note told the generator a "
                 "disagreement existed on 5 of 14 queries while showing none of it"),
 ("§4.1.2 generation control", "the four conditions differ only in retrieval and the "
  "confidence layer",
  "EDIT PENDING", "the submitted text describes FOUR conditions; this run has FIVE. B4, a "
                  "prompting control that receives B3's retrieval and an instruction to "
                  "report disagreement instead of the Confidence Levels, did not exist "
                  "when the manuscript was written and must be added to §4.1.2 and "
                  "Table 6. The conditions no longer differ by retrieval and the "
                  "confidence layer alone: B3, B4 and B5 share one retrieval and differ "
                  "only in what the generator is TOLD, which is what makes B5 - B4 an "
                  "comparison that is not an ablation, rather than another one. Separately, "
                  "COMMON_RULE is appended verbatim to all FIVE system prompts - answer "
                  "from the supplied evidence only, no prior-knowledge advice, at most "
                  "150 words, no padding - and that control is not in the submitted text "
                  "either: without it answer length correlated with the judge score at "
                  "r = 0.66 in the recorded evaluation and the longest-answer condition "
                  "won on presentation alone"),
 ("§4.1.2 Table 6", "B4 = the prompting control",
  "EDIT PENDING", "absent from the submitted Table 6, which predates it. B4 is B3's "
                  "retrieval with no Confidence Levels shown and an instruction to look "
                  "for disagreement and report it, written to be as strong as B5's. Call "
                  "it a PROMPTING CONTROL, not an external baseline: it uses this "
                  "system's own retrieval and is not an independent published method. "
                  "What it adds is the only comparison that is not an ablation - B1-B3 "
                  "are all this system with a component removed - so without it the "
                  "referee question 'would an instruction have done the same?' is "
                  "unanswered"),
 ("§4.1.2 Table 6", "B2 = single-hop KG-RAG",
  "implemented", "one-hop neighbours of the matched nodes"),
 ("§4.1.2 Table 6", "B3 = GraphRAG with full graph indexing (community summaries)",
  "EDIT PENDING", "no condition builds community summaries. The code implements B3 as "
                  "multi-hop KG-RAG without the auditing layer. Rewrite Table 6 from "
                  "BASELINE_SPEC and rename B3 away from 'GraphRAG', or B3 < B2 reads as "
                  "evidence against GraphRAG's indexing - a claim this study is not "
                  "designed to support - see decision 6"),
 ("§4.1.2 Table 6", "B5 = B3 plus the full Self-Auditing pipeline",
  "implemented", "B3 and B5 share one retrieval call and differ only in whether the "
                 "Confidence Levels are shown, so B5 - B3 is a clean ablation of the "
                 "auditing layer - see decision 13"),
 ("§5.4 limitations", "community-summary indexing is not listed among the limitations",
  "EDIT PENDING", "GraphRAG's community-summary indexing was not implemented in any "
                  "condition. §5.4 must say so; without it B3 is readable as a full "
                  "GraphRAG reimplementation - see decision 6"),
 ("§4.1.1", "the 770 triplets come from a single extraction run",
  "EDIT PENDING", "two passes, 405 then 365, merged. A two-stage extraction is "
                  "unremarkable; not describing it is what a reviewer objects to - "
                  "see decision 17"),
 ("§4.2.2", "criterion (i) found five cross-source disagreements",
  "CLAIM NOT SUPPORTED",
  f"the rule stage raises {_mf('step2','t1_by_criterion','cross_source_disagreement', default='?')} "
  f"flags and the context audit confirms "
  f"{_mf('confidence','confirmed_t1_by_criterion','cross_source_disagreement', default='?')}. "
  "Neither is five, and neither is fixed: the auditor is not deterministic. State both "
  "figures with the stage each belongs to - see decision 18"),
 ("§4.2.5", "the Gold-Standard false-alarm rate is 1.7%",
  "CLAIM NOT SUPPORTED",
  (lambda n, r, p: f"the rule stage flags {r} of {n} ({r/n:.1%}) and {p} of {n} ({p/n:.1%}) "
                   f"survive the context audit" if n else "Part 7 did not run")(
      _mf('gold_false_alarm','n_gold', default=0), _mf('gold_false_alarm','rule_stage', default=0),
      _mf('gold_false_alarm','post_audit', default=0))
  + ". Report both and name the stage each belongs to; the drop is the second stage doing "
    "exactly what it exists for - see decision 15"),
 ("§4.2.6", "Gold-Standard recall is 100%",
  "CLAIM NOT SUPPORTED",
  f"recall at tau = {FUZZY_TAU} is "
  f"{(lambda v: f'{v:.1%}' if isinstance(v, float) else '?')(_mf('recall','by_tau', str(FUZZY_TAU)))}"
  ". State the threshold, and say the metric measures surface overlap with the nearest "
  "corpus string, not semantic recall; Part 7 prints the whole curve - see decision 16"),
 ("§4.3.2", "the C1 causal chain has seven steps",
  "CLAIM NOT SUPPORTED",
  f"the longest SIMPLE causal chain the graph supports is "
  f"{_mf('kg','longest_simple_causal_chain_hops', default='?')} hops. The seven came from "
  "traverse_causal_chain, a breadth-first walk with a global visited set that revisits "
  "nodes and counts each revisit as a step - see decision 19"),
 ("§4.2.4", "the auditor judges the corrupted triplet in context",
  "implemented", "corpus-referenced protocol primary; oracle protocol separate"),
 ("§4.4.1", "76 papers, 8 parameters, k = 19/38/57/76, 100 draws",
  "implemented", "one fixed draw plan shared by every analysis"),
 ("§4.4", "the Self-Auditing layer is exercised on the second crop",
  "PARTIAL", "audit_corpus runs with do_audit=False and the cached Confidence Levels "
             "enter only as aggregation weights; the generalisation claim is about the "
             "extraction-and-aggregation pipeline - see decision 14"),
 ("§4.4.2", "four aggregators and three normalizers",
  "implemented", "trimming proportion is not stated in the paper; fixed and recorded"),
 ("§4.4.2", "deviations <= 0.1 SD at k = 57",
  "CLAIM NOT SUPPORTED", (_sd_claim_str() or "Part 11 did not run, so the claim is "
                          "untested here") + ". Replacement wording in audit B, "
                          "decision 9 - the manuscript sentence must be edited"),
 ("§4.4.2", "the full-corpus value lay 'within the central region'",
  "EDIT PENDING", "'central region' is never defined numerically, so as written it is "
                  "not an auditable claim. The code reports mean |z| and max |z| and "
                  "invents no interval criterion; three concrete readings are printed as "
                  "diagnostics and one must be adopted - see decision 8"),
]

order = {"CLAIM NOT SUPPORTED": 0, "NOT IMPLEMENTED": 1, "PARTIAL": 2,
         "EDIT PENDING": 3, "implemented": 4}
OWED = ("CLAIM NOT SUPPORTED", "NOT IMPLEMENTED", "PARTIAL", "EDIT PENDING")
print("A. PAPER-TO-CODE IMPLEMENTATION AUDIT")
print("Audited against the manuscript AS SUBMITTED. The claim column quotes the current")
print("text; the note carries the replacement wording. What each edit is, and why, is in B.")
print("=" * 100)
print(f"{'section':<26}{'claim (as submitted)':<54}{'status':<20}")
print("-" * 100)
for sec, claim, st, note in sorted(IMPLEMENTATION, key=lambda r: order[r[2]]):
    print(f"{sec:<26}{claim[:52]:<54}{st:<20}")
    if st != "implemented":
        print(f"{'':<26}  -> {note}")
tally_a = Counter(r[2] for r in IMPLEMENTATION)
_owed = [r for r in IMPLEMENTATION if r[2] in OWED]
print(f"\n{dict(tally_a)}")
print(f"\n{len(_owed)} of {len(IMPLEMENTATION)} rows still owe the manuscript an edit.")
print("None of them is undecided - B gives every one a resolution and the wording to use.")
print("They are outstanding in the sense that the text has not been changed yet. When a")
print("sentence is actually rewritten, flip its row here; the count above is the progress")
print("meter for the revision.")
MANIFEST["audit_edits_owed"] = {"n_owed": len(_owed), "n_rows": len(IMPLEMENTATION),
                                "by_status": dict(tally_a),
                                "sections": [f"{r[0]} - {r[1][:70]}" for r in _owed]}
MANIFEST["audit_implementation"] = [{"section": s, "claim": c, "status": st, "note": n}
                                    for s, c, st, n in IMPLEMENTATION]


A. PAPER-TO-CODE IMPLEMENTATION AUDIT
Audited against the manuscript AS SUBMITTED. The claim column quotes the current
text; the note carries the replacement wording. What each edit is, and why, is in B.
section                   claim (as submitted)                                  status              
----------------------------------------------------------------------------------------------------
§4.2.2                    criterion (i) found five cross-source disagreements   CLAIM NOT SUPPORTED 
                            -> the rule stage raises 18 flags and the context audit confirms 7. Neither is five, and neither is fixed: the auditor is not deterministic. State both figures with the stage each belongs to - see decision 18
§4.2.5                    the Gold-Standard false-alarm rate is 1.7%            CLAIM NOT SUPPORTED 
                            -> the rule stage flags 3 of 60 (5.0%) and 2 of 60 (3.3%) survive the context audit. Report both and name the stage each belong

In [73]:
# ====== cell 66 ======
_SDC   = MANIFEST.get("strawberry_convergence", {}).get("sd_sufficiency_claim")
_HOPS  = MANIFEST.get("kg", {}).get("longest_simple_causal_chain_hops", "?")
_HOPDIST = MANIFEST.get("kg", {}).get("simple_causal_paths_by_depth", {})

# ============ B. Decisions taken — manuscript revision checklist ============
# Each row is a conflict that has been resolved, the side that moves, and the concrete edit.
DECISIONS = [
 ("§3.2.2 worked example",
  "criterion (i) is named cross-source, but the example compares DO 8 mg/L (S01) with "
  "6 mg/L (S01) and both values carry a condition",
  "MANUSCRIPT",
  "Replace the example with a cross-source, unconditioned pair. Table 3's "
  "'water temp. 13-15 C vs 17-20 C (cross-source)' already qualifies."),

 ("§3.2.2 criterion (i) rule",
  "the text says NON-IDENTICAL values; Table 3 and §3.4 imply non-overlapping ranges",
  "MANUSCRIPT",
  "State the rule as non-overlapping ranges. Strict non-identity would flag 8.0 against "
  "8.1; the code implements non-overlapping and the counts in §4.2.1 follow it."),

 ("Algorithm 1, lines 5-11",
  "the pseudocode makes the T3 branch an `else if`; the note below says all three types "
  "are detected in a single call",
  "MANUSCRIPT",
  "Redraw the pseudocode so T2, T1 and T3 are all evaluated, and delete the reconciling "
  "note. Under the else-if reading T3 candidates fall from 34 to 10."),

 ("Algorithm 1, lines 6 and 8",
  "LLM_Audit is written as receiving the source document d_i; the auditor actually "
  "receives the domain thresholds and the corpus peers for the parameter",
  "MANUSCRIPT",
  "Change the call to LLM_Audit(t, flag, THRESHOLDS, peers(t, T)). Corpus-wide context is "
  "also the correct input for a cross-source criterion; a single document cannot support it."),

 ("§3.2.4 Definition 1",
  "D(t) is written as a single value from {none, T1, T2, T3}, but multi-label detection "
  "makes it a set and Table 4 has no row for a triplet flagged T1 and T3 at once",
  "MANUSCRIPT",
  "Restate over the set of confirmed types: any confirmed T1 or T2 -> Conflicted; else any "
  "confirmed T3 -> Disputed; else Verified. Add the combined row to Table 4."),

 ("§4.1.2 Table 6, baseline naming",
  "B3 is defined as GraphRAG with full graph indexing (community summaries); no condition "
  "builds community summaries",
  "MANUSCRIPT",
  "Rewrite Table 6 from BASELINE_SPEC (Part 8): B2 single-hop KG-RAG, B3 multi-hop KG-RAG "
  "without auditing, B5 multi-hop with Self-Auditing. Rename B3 away from 'GraphRAG' so it "
  "cannot be read as a full reimplementation of that method. Note in §5.4 that GraphRAG's "
  "community-summary indexing was not implemented. Rationale: under the old definition B5 "
  "inherits community summaries too, so B3 < B2 would read as evidence against GraphRAG's "
  "indexing, which this study is not designed to support; under the new one it is the "
  "paper's own thesis. Implementing it would also void the recorded Table 11."),

 ("§4.3.2 case study C1",
  "the quoted B5 response is a seven-step chain; the recorded B5 answer for that query "
  "reports that the graph held no direct evidence",
  "MANUSCRIPT",
  "Label the quoted response an illustration regenerated from the audited graph and state "
  "that it was not scored. Then check it against Part 10's output before quoting: the "
  "regenerated answer describes parallel mechanisms rather than one linear seven-step "
  "chain, so claim what the run shows - multi-hop retrieval spanning several source "
  "documents with per-path Confidence Levels - and quote the longest chain the graph "
  "supports as retrieval evidence, separately from the generated text."),

 ("§4.4.2 'central region'",
  "the phrase is never defined, so an interval-coverage claim is made against a criterion "
  "that was never specified",
  "MANUSCRIPT",
  "Do not invent a criterion. Drop the interval-coverage wording and report the statistics "
  "actually computed: mean |z| = 0.40 and max |z| = 0.81, i.e. every parameter lies within "
  "one subset standard deviation of the subset mean. The |z| < 1, central 80% and "
  "empirical 95% coverages (8/8 in all three) stay in the Supplementary Material as "
  "diagnostics."),

 ("§4.4.2 sufficiency criterion",
  "'deviations remaining below one data-standard-deviation (<= 0.1 SD at k = 57)' treats "
  "0.1 SD as an upper bound for every parameter, which this run contradicts: "
  + (_sd_claim_str() or "Part 11 did not run"),
  "MANUSCRIPT",
  "Replace the sentence with the measured figures printed in Part 11. Verbatim "
  "replacement: " + (
      f"\"all {_SDC['n_le_1_0']} parameters remain below one data standard deviation at "
      f"k = 57 (maximum {_SDC['max_at_57']:.2f} SD, for {_SDC['argmax']}), and "
      f"{_SDC['n_le_0_1']} of {_SDC['n_params']} are at or below 0.1 SD\""
      if _SDC else "run Part 11 to obtain it") +
  ". Do not keep 0.1 SD as a bound; it holds for most parameters, not all."),
]

# Findings from the reruns below. They change reported NUMBERS rather than method, so
# they are listed here and applied to the manuscript in one pass.
DECISIONS += [
 ("§3.2.2 'the two reference tables'",
  "five tables are domain-specific, not two: the Normal Range Table, the unit-alias "
  "table, the equivalent-unit pairs, the unit-conversion table and the notation map",
  "MANUSCRIPT",
  "Say five, and list them. Part 2 prints all five. The claim that porting the method "
  "costs only two tables is the weaker one to defend; the honest count is still small."),

 ("§3.4 retrieval reaching the auditing layer",
  "before the parameter-synonym map no query matched a parameter node, so no answer "
  "carried a Confidence Level and B5 was identical to B3 in substance; and until graded "
  "items were held back from the retrieval limit, 5 of the 14 queries with a disagreement "
  "showed the generator a warning without the items it warned about",
  "CODE (already fixed)",
  "Part 8 adds PARAM_SYNONYMS, pulls the relational edges of the parameters a causal chain "
  "touches, and spends the retrieval budget on graded items first. Table 11 and every "
  "judge figure have been regenerated on the corrected retrieval; the recorded evaluation "
  "is retained as an input file and is not reported."),

 ("§3.3 Confidence Level on causal edges",
  "causal edges are loaded from causal_relations.json and never pass the detector, so "
  "their Verified label is a default, not an audit result",
  "MANUSCRIPT",
  "State that the Confidence Level is assigned to the relational (measurement) edges and "
  "propagated along causal paths, and that causal edges themselves are not audited in "
  "this study. Do not claim every edge carries an audited level."),

 ("§4.1.2 B3 vs B5",
  "B3 and B5 issue the same retrieval and differ only in whether the audited Confidence "
  "Levels are shown to the generator",
  "MANUSCRIPT",
  "Describe B5 - B3 as an ablation of the auditing layer alone. That is a stronger claim "
  "than a general system comparison, and it is what the code implements."),

 ("§4.4 the second crop",
  "the strawberry analysis aggregates with do_audit=False; the auditing layer does not "
  "run on that corpus, and the cached Confidence Levels act only as aggregation weights",
  "MANUSCRIPT",
  "Scope the generalisation claim to the extraction-and-aggregation pipeline under "
  "sparsity. Do not claim the conflict detector was validated on a second crop."),

 ("§4.2.5 Gold-Standard false-alarm rate",
  f"the rule stage flags {len(flagged)} of {len(GOLD)} Gold triplets "
  f"({len(flagged)/len(GOLD):.1%}); after the context audit {gs_conf} of {len(GOLD)} "
  f"survive ({gs_conf/len(GOLD):.1%}). The mechanism is that criterion (ii) compares "
  "against the Normal Range Table without reading the triplet's condition field",
  "MANUSCRIPT",
  "Report both numbers and name the stage each belongs to. Explaining the mechanism is "
  "the point: it is exactly the case the second stage exists to catch."),

 ("§4.2.6 Gold-Standard recall",
  "recall at tau = 0.50 is 90.0% (54/60), not 100%. Five of the six misses differ from "
  "the corpus only in predicate wording; one (GS-048) is a genuine extraction miss",
  "MANUSCRIPT",
  "Report 90.0% with the threshold stated, and say what the metric measures - surface "
  "overlap between the Gold string and the nearest corpus string, not semantic recall. "
  "Part 7 prints the full recall-versus-threshold curve (Figure F5, Table T8)."),

 ("§4.1.1 corpus construction",
  "the 770 triplets were extracted in two passes - 405 then 365 - and merged, not in a "
  "single run",
  "MANUSCRIPT",
  "Describe both passes and the merge. A two-stage extraction is unremarkable; not "
  "describing it is what a reviewer would object to."),

 ("§4.2.2 'five cross-source disagreements'",
  f"criterion (i) raises {t1_crit.get('cross_source_disagreement', 0)} flags at the rule "
  f"stage, of which the context audit confirms "
  f"{t1_basis.get('cross_source_disagreement', 0)}. Neither figure is fixed: the auditor "
  "is not deterministic, and across the five repeat runs of Part 4 the confirmed count "
  "moves with the Confidence Level distribution",
  "MANUSCRIPT",
  "Do not quote a bare count as though it were a property of the method. State the "
  "rule-stage figure and the confirmed figure with the stage each belongs to, both taken "
  "from the deposited run, and say that the confirmed count is auditor-dependent."),

 ("§4.3.2 'seven-step causal chain'",
  f"the longest SIMPLE causal chain the graph supports is {_HOPS} hops. The 7 in the "
  "manuscript came from pipeline.traverse_causal_chain, a breadth-first walk with a "
  "global visited set: it revisits nodes and counts each revisit as a step, so its "
  "'chains' are not paths and its hop counts are inflated",
  "MANUSCRIPT",
  f"Delete 'seven-step' everywhere it appears. Quote {_HOPS} hops and the hop histogram "
  f"Part 8 prints ({_HOPDIST}); Table T9 carries both. Part 10 also prints the longest "
  "chain actually RETRIEVED for C1, which is the number to quote if the sentence is "
  "about the case study rather than about the graph. Note that entity fragmentation "
  "(DO_low, low_DO, DO_decrease as separate nodes) is what caps the depth; that is a "
  "limitation to state in §5.4, not a number to inflate."),

 ("§4.4.2 aggregation weights",
  "CONF_WEIGHT is uniform in the deposited configuration, so the weighted median equals "
  "the plain median on this corpus",
  "MANUSCRIPT",
  "Either state that the weights were uniform in this run, or report the weighted and "
  "unweighted estimates side by side. Part 11 prints both."),

 ("§4.4.2 closed-book control",
  "the control asks the model for eight parameter values without documents; it is a "
  "yardstick for comparing two estimators, not a validation of either",
  "MANUSCRIPT",
  "Keep the framing the notebook uses: the internal reference is the full corpus; the "
  "literature column is context only."),
]

print("B. DECISIONS TAKEN - manuscript revision checklist")
print("=" * 100)
for i, (loc, conflict, side, edit) in enumerate(DECISIONS, 1):
    print(f"\n[{i}] {loc}    -> change the {side}")
    print(f"    conflict : {conflict}")
    print(f"    edit     : {edit}")

_code_side = sum(1 for d in DECISIONS if d[2].startswith("CODE"))
print(f"\n{len(DECISIONS)} decisions; {len(DECISIONS) - _code_side} move the manuscript, "
      f"{_code_side} moved the code and are already applied here.")
print("The code implements the chosen side throughout, so the numbers this notebook")
print("prints are the numbers the revised manuscript should carry.")
MANIFEST["decisions"] = [{"location": l, "conflict": c, "side": s, "manuscript_edit": e}
                         for l, c, s, e in DECISIONS]


B. DECISIONS TAKEN - manuscript revision checklist

[1] §3.2.2 worked example    -> change the MANUSCRIPT
    conflict : criterion (i) is named cross-source, but the example compares DO 8 mg/L (S01) with 6 mg/L (S01) and both values carry a condition
    edit     : Replace the example with a cross-source, unconditioned pair. Table 3's 'water temp. 13-15 C vs 17-20 C (cross-source)' already qualifies.

[2] §3.2.2 criterion (i) rule    -> change the MANUSCRIPT
    conflict : the text says NON-IDENTICAL values; Table 3 and §3.4 imply non-overlapping ranges
    edit     : State the rule as non-overlapping ranges. Strict non-identity would flag 8.0 against 8.1; the code implements non-overlapping and the counts in §4.2.1 follow it.

[3] Algorithm 1, lines 5-11    -> change the MANUSCRIPT
    conflict : the pseudocode makes the T3 branch an `else if`; the note below says all three types are detected in a single call
    edit     : Redraw the pseudocode so T2, T1 and T3 are all evaluated, and

`cell 67`

## Part 14 — Manifest

In [109]:
# ====== cell 68 ======
MANIFEST["finished_utc"] = datetime.now(timezone.utc).isoformat()
mpath = OUT_DIR / f"MANIFEST{OUT_TAG}.json"
json.dump(MANIFEST, open(mpath, "w", encoding="utf-8"), ensure_ascii=False, indent=2, default=str)
print(f"manifest written -> {mpath}\n")
print("artefacts:")
for p in sorted(OUT_DIR.glob("*")):
    print(f"  {p.name:<34}{p.stat().st_size:>12,} bytes")

print("\nheadline figures recorded in MANIFEST.json")
print("-" * 62)
def g(*keys, default="-"):
    d = MANIFEST
    for k in keys:
        if not isinstance(d, dict) or k not in d:
            return default
        d = d[k]
    return d
print(f"  corpus triplets                : {g('corpus','n_triplets')}")
print(f"  Step 2 flagged / flags         : {g('step2','flagged')} / {g('step2','total_flags')}")
print(f"  Confidence distribution        : {g('confidence','distribution')}")
print(f"  Gold false-alarm rule stage    : {g('gold_false_alarm','rule_stage')}/{g('gold_false_alarm','n_gold')}")
print(f"  Gold false-alarm after audit   : {g('gold_false_alarm','post_audit')}")
print(f"  recall at tau={FUZZY_TAU}             : {g('recall','by_tau',str(FUZZY_TAU))}")
print(f"  injection detected             : {g('injection','detected')}/{g('injection','n')}")
print(f"  injection confirmed (corpus)   : {g('injection','confirmed_corpus_referenced')}/{g('injection','n')}")
print(f"  knowledge graph                : {g('kg','nodes')} nodes, {g('kg','edges')} edges, "
      f"longest simple causal chain {g('kg','longest_simple_causal_chain_hops')} hops")
print(f"  judge means                    : {g('evaluation','means')}")
print(f"  Krippendorff (ordinal)         : {g('evaluation','krippendorff_ordinal')}")
print(f"  strawberry normalized error    : {g('strawberry_convergence','headline_error')}")
print(f"  UR (disputed queries only)     : {g('ur_scores','means')}")
print(f"  reported / judged run          : {g('reporting_source')} / {g('judging_source')}")
print("\nMANIFEST.json records the input hashes, package versions and every reported "
      "figure. Cite it alongside this notebook.")

manifest written -> repro_out\MANIFEST_v2.json

artefacts:
  answers.json                            95,749 bytes
  answers_b5.json                         12,754 bytes
  answers_regenerated.json                57,952 bytes
  archive_20260901T064624Z                 4,096 bytes
  archive_20260901T064806Z                     0 bytes
  audited_cross_source.json              298,293 bytes
  audited_def1.json                      455,334 bytes
  audited_multilabel.json                298,743 bytes
  audited_triplets.json                  319,069 bytes
  case_c1_illustration.json                3,357 bytes
  closed_book.json                           120 bytes
  fig_convergence.png                     72,281 bytes
  fig_strawberry_convergence.png          78,132 bytes
  fig_strawberry_representativeness.png      78,969 bytes
  figures                                  8,192 bytes
  figures_v2                               8,192 bytes
  gold_audits.json                         2,061 bytes
  i